<a href="https://colab.research.google.com/github/hawa1983/Capstone-Final-Modeling-and-Data/blob/main/dow_ridership_notebook_v7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Beyond the Weekday/Weekend Binary
## Day-of-Week Demand Regimes in the Post-Telework Era: Evidence from NYC Subway Turnstile Data

**Author:** Fomba Kassoh | **Target Journal:** Transportation Research Record (TRR) | **Date:** 2026

---

### Research Overview
This notebook implements the full analytical pipeline for a TRR publication examining how telework-driven behavioral change has disrupted the traditional weekday/weekend demand binary in the NYC subway system.

**Research Question:** Have hybrid work schedules structurally altered day-of-week ridership patterns at the station-complex level, and if so, do distinct demand regime typologies emerge that challenge MTA's current binary weekday/weekend scheduling paradigm?

### Analytical Pipeline
| Section | Task | Key Output |
|---------|------|-----------|
| 0 | Environment Setup | Libraries, Drive mount, global config |
| 1 | Data Acquisition | Pre-COVID turnstile (2017–2019), post-COVID hourly (2022–2024), ACS, LODES |
| 2 | Data Processing | Station-complex daily ridership, 7-day DOW vectors |
| 3 | Exploratory Analysis | Summary stats, recovery rates, Figure 2 |
| 4 | Cluster Analysis | K-means typology, Figures 1 & 3, Tables 2 & 3 |
| 5 | Regression Analysis | Multinomial logit, Table 4 |
| 6 | Visualization | Figure 4 (NYC cluster map) |
| 7 | Robustness | K-sensitivity, weekend definition tests |
| 8 | Deliverables | Final checklist, key findings |

### Data Sources
- **MTA Turnstile 2017–2019:** data.ny.gov archived annual datasets (v5y5-mwpb, t5e5-jamc, xfn5-qji9)
- **MTA Subway Hourly Ridership 2022–2024:** data.ny.gov dataset wujg-7c2s
- **ACS 5-Year Estimates:** US Census Bureau API (block-group level)
- **LODES/LEHD WAC:** US Census LEHD LODES8 (workplace job counts)
- **TIGER/Line Block Groups:** US Census TIGER 2022
- **Station Complexes:** Fomba Kassoh GitHub (hawa1983/Capstone)


---
## SECTION 0: Environment Setup

Install packages, import libraries, mount Google Drive, and define all global configuration variables in one place.

**Why configure globally:** All study-period labels, K-means parameters, colour palettes, and file paths are set here once. Downstream cells read from these globals rather than hard-coding values, so you can change K_FINAL or add a study year by editing a single cell.

> **Important:** Run cells 0.1 → 0.3 in order at the start of every new Colab session before running any other section.


In [1]:
# ==============================================================
# Section 0.1 — Install Dependencies
#
# Purpose:
#   Install all required Python packages for this notebook.
#   Run once per Colab session (or when packages are missing).
#   Safe to re-run; pip skips already-installed packages.
# ==============================================================

!pip install pandas numpy scikit-learn matplotlib seaborn geopandas scipy statsmodels requests tqdm plotly --quiet

print("Package installation complete.")


Package installation complete.


In [2]:
# ==============================================================
# Section 0.2 — Import Libraries
#
# Purpose:
#   Import all third-party libraries used throughout the notebook.
#   Centralising imports here makes dependency issues easy to spot.
#
# Key packages:
#   pandas / numpy     — data manipulation
#   scikit-learn       — K-means, StandardScaler, cross-validation
#   matplotlib/seaborn — all figures
#   statsmodels        — multinomial logistic regression (MNLogit)
#   geopandas          — spatial join for built-environment features
#   scipy.stats        — paired t-tests (Section 7.2)
#   requests / tqdm    — data downloads with progress bars
# ==============================================================

from __future__ import annotations

import os
import re
import warnings
from io import StringIO, BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from tqdm import tqdm

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline

import statsmodels.api as sm
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print("All libraries loaded successfully.")


All libraries loaded successfully.


In [3]:
# ==============================================================
# Section 0.2b — Google Colab: Mount Google Drive
#
# Purpose:
#   Mounts Google Drive so all downloaded data, cached CSVs,
#   and output figures persist between Colab sessions.
#
#   Without this, every runtime disconnect wipes all files.
#   The MTA turnstile annual files are large; you do not want
#   to re-download them each session.
#
# Instructions (first run):
#   1. Run this cell — a Google auth popup will appear.
#   2. Sign in and allow Drive access.
#   3. Your Drive mounts at /content/drive/MyDrive/
#
# To run locally: set USE_DRIVE = False.
# ==============================================================

from pathlib import Path

USE_DRIVE = True   # Set False if running locally

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    OUT_DIR = Path("/content/drive/MyDrive/dow_ridership_paper/outputs")
else:
    OUT_DIR = Path("./outputs")   # local fallback

OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "raw_turnstile").mkdir(exist_ok=True)

print(f"Output directory: {OUT_DIR}")
print(f"Drive mounted:    {USE_DRIVE}")


Mounted at /content/drive
Output directory: /content/drive/MyDrive/dow_ridership_paper/outputs
Drive mounted:    True


In [4]:
# ==============================================================
# Section 0.3 — Global Configuration
#
# Purpose:
#   Define ALL study-wide constants in one place.
#   Downstream cells read from these globals; never hard-code
#   values like K_FINAL or CATCHMENT_MILES in later cells.
#
# Key decisions documented here:
#   K_FINAL = 4   — selected after inspecting elbow + silhouette
#                   diagnostics in Section 4.1. K=2 has the highest
#                   silhouette but is too coarse; K=4 yields four
#                   interpretable demand regimes without over-
#                   fragmenting the station universe.
#   PRE_COVID_YEARS = [2017, 2018, 2019]
#                 — three-year average smooths year-specific
#                   disruptions (2017 Summer of Hell, 2018 L train
#                   prep weekends, 2019 L cancellation shock).
# ==============================================================

import matplotlib.pyplot as plt

# ---- Study periods -----------------------------------------------
PERIODS = {
    "precovid": {
        "label": "Pre-COVID Baseline",
        "years": [2017, 2018, 2019],
        "baseline_file": "station_daily_precovid_avg.csv",
    },
    "2022": {"label": "Early Recovery",          "year": 2022},
    "2024": {"label": "Stabilized New Normal",   "year": 2024},
}

PRE_COVID_YEARS  = [2017, 2018, 2019]
POST_COVID_YEARS = [2022, 2023, 2024]   # 2023 used for robustness only

# ---- Day-of-week mapping -----------------------------------------
# pandas dt.dayofweek: Mon=0 ... Sun=6
DOW_LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
DOW_MAP    = {0:"Mon", 1:"Tue", 2:"Wed", 3:"Thu", 4:"Fri", 5:"Sat", 6:"Sun"}

# ---- K-means parameters ------------------------------------------
K_RANGE     = range(2, 9)   # sweep K=2..8 in Section 4.1
K_FINAL     = 4             # SINGLE source of truth — do not override in later cells
KMEANS_SEED = 42            # for reproducibility
KMEANS_INIT = 50            # number of random initialisations

# ---- Built-environment catchment ---------------------------------
CATCHMENT_MILES = 0.5       # 0.5-mile buffer around each station complex

# ---- Correct dataset IDs (verified against data.ny.gov) ----------
# 2017: v5y5-mwpb  ✓ confirmed MTA turnstile
# 2018: t5e5-jamc  ✓ confirmed MTA turnstile (bjcb-yee3 was COVID health data — wrong)
# 2019: xfn5-qji9  ✓ confirmed MTA turnstile
DATA_NY_DATASET_IDS = {
    2017: "v5y5-mwpb",
    2018: "t5e5-jamc",   # CORRECTED from bjcb-yee3
    2019: "xfn5-qji9",
}

# ---- MTA station complex reference file --------------------------
# Sourced from Fomba Kassoh's verified GitHub repository.
# Columns confirmed: Complex ID, Stop Name, Latitude, Longitude,
# Borough, Daytime Routes, Structure Type, ADA, etc.
STATION_COMPLEX_GITHUB_URL = (
    "https://raw.githubusercontent.com/hawa1983/Capstone/"
    "refs/heads/main/MTA_Subway_Stations_and_Complexes.csv"
)

# ---- Plot style ---------------------------------------------------
PALETTE = ["#1F5C99", "#E8702A", "#2E9E4F", "#C03B2B", "#7B4EA0",
           "#8C564B", "#E377C2", "#7F7F7F"]

plt.rcParams.update({
    "figure.dpi":        150,
    "font.family":       "sans-serif",
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

print("Configuration complete.")
print(f"  K_FINAL            = {K_FINAL}")
print(f"  PRE_COVID_YEARS    = {PRE_COVID_YEARS}")
print(f"  POST_COVID_YEARS   = {POST_COVID_YEARS}")
print(f"  CATCHMENT_MILES    = {CATCHMENT_MILES}")
print(f"  Output directory   = {OUT_DIR}")


Configuration complete.
  K_FINAL            = 4
  PRE_COVID_YEARS    = [2017, 2018, 2019]
  POST_COVID_YEARS   = [2022, 2023, 2024]
  CATCHMENT_MILES    = 0.5
  Output directory   = /content/drive/MyDrive/dow_ridership_paper/outputs


---
## SECTION 1: Data Acquisition

Collect all raw data needed for the analysis. Three data streams are acquired in parallel:

1. **MTA Turnstile Data (2017–2019)** — Pre-COVID baseline from official annual archived datasets on `data.ny.gov`. Each year is a single ~200–400 MB CSV containing cumulative entry/exit counts for every physical turnstile unit at every audit interval (~4 hours). The older `web.mta.info/developers` weekly file URL is retired and no longer serves data.

2. **MTA Subway Hourly Ridership (2022–2024)** — Post-COVID ridership already aggregated to station-complex level by the MTA, available via the Socrata API at `data.ny.gov/resource/wujg-7c2s.csv`. Covers 2020–2024.

3. **Built Environment Data** — ACS 5-year block-group estimates (US Census API) and LODES Workplace Area Characteristics job counts (US Census LEHD). Used in Section 5 to explain cluster membership via multinomial logistic regression.

All files are cached to Google Drive on first download; subsequent runs load from cache in seconds.


### 1.1 Pre-COVID Baseline: Annual Turnstile Data (2017, 2018, 2019)

**Purpose:** Download three full calendar years of MTA turnstile data from `data.ny.gov` archived annual datasets, compute net daily entries per station, apply extreme-event exclusion flags, and save one `station_daily_{year}.csv` per year. Then compute a 3-year averaged baseline (`station_daily_precovid_avg.csv`) that smooths year-specific disruptions.

**Why three years:** Using a single year as the baseline risks conflating structural commute patterns with year-specific anomalies. Key disruptions include: the 2017 "Summer of Hell" (Penn Station emergency repairs Jul 10–Sep 1 inflated subway ridership at Manhattan trunk stations), 2018 L train prep weekend shutdowns (15 weekends), and the January 2019 L train cancellation behavioral shock. Averaging across 2017–2019 produces a more defensible baseline — standard practice in transit recovery literature.

**Exclusion flags:** Holidays and acute system-wide events are removed before DOW averaging. Single-line incidents are NOT excluded because the analysis operates at the station-complex level and only full-system events confound DOW profiles.


In [5]:
# ==============================================================
# Section 1.1 — Pre-COVID Baseline: Annual Turnstile Download
#                Years: 2017, 2018, 2019
#
# Data source:
#   data.ny.gov annual archived turnstile CSVs (one file per year).
#   Dataset IDs defined in Section 0.3 global config.
#
# Extreme event exclusions (system-wide acute events only):
#   2017: Apr 21 power failure (half-system delays), May 7–9 Con Ed
#         outage, Jun 27–29 A-train derailment + State of Emergency,
#         Jul 10–Sep 1 Penn Station repairs (commuter overflow).
#   2018: L train prep weekend shutdowns (15 Sat/Sun pairs).
#   2019: All of January (L train cancellation behavioral shock),
#         Feb–Apr L prep weekend closures, Apr 21 original shutdown date.
#
# Processing pipeline per year:
#   1. Download annual CSV from data.ny.gov (cached after first run).
#   2. Read in chunks to handle large files in Colab memory.
#   3. Parse DATETIME from Date + Time fields.
#   4. Sort within each (C/A, Unit, SCP) physical turnstile unit.
#   5. Compute net entries: diff() within unit, clip [0, 10000].
#      Negative diff = counter reset → treated as 0.
#      >10000 per 4-hr audit = implausible → capped at 10000.
#   6. Add date, day-of-week, and DOW label columns.
#   7. Apply holiday and extreme-event exclusion masks.
#   8. Aggregate to station × date (sum net entries across all
#      turnstile units within the station).
#
# Outputs:
#   raw_turnstile/turnstile_{year}.csv   (raw annual download, cached)
#   station_daily_{year}.csv             (cleaned station-date aggregation)
#   station_daily_precovid_avg.csv       (3-year DOW average baseline)
# ==============================================================

import time
import datetime
from datetime import date, timedelta

# ---- Session recovery: reload globals if runtime was reset -------
if 'OUT_DIR' not in globals():
    OUT_DIR = Path("/content/drive/MyDrive/dow_ridership_paper/outputs")
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"OUT_DIR recovered: {OUT_DIR}")

if 'DOW_MAP' not in globals():
    DOW_MAP    = {0:"Mon",1:"Tue",2:"Wed",3:"Thu",4:"Fri",5:"Sat",6:"Sun"}
    DOW_LABELS = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
    print("DOW_MAP recovered.")

if 'DATA_NY_DATASET_IDS' not in globals():
    DATA_NY_DATASET_IDS = {2017:"v5y5-mwpb", 2018:"t5e5-jamc", 2019:"xfn5-qji9"}
    print("DATA_NY_DATASET_IDS recovered.")

if 'PRE_COVID_YEARS' not in globals():
    PRE_COVID_YEARS = [2017, 2018, 2019]

# Protect the cleaned dict — DO NOT wipe if it already has years loaded
# from a prior cell run in this session.
if 'cleaned' not in globals():
    cleaned = {}
    print("Initialized empty cleaned dict.")

(OUT_DIR / "raw_turnstile").mkdir(exist_ok=True)

# ---- Turnstile column validation set ----
TURNSTILE_REQUIRED_COLS = {"C/A","Unit","SCP","Station","Date","Time","Entries"}

# ---- Holidays to exclude (month, day) — all pre-COVID years ----
HOLIDAY_MD = {
    (1, 1),   # New Year's Day
    (1,15),   # MLK Day (approximate)
    (5,27),(5,28),(5,29),   # Memorial Day 2019, 2018, 2017
    (7, 4),   # Independence Day
    (9, 2),(9, 3),(9, 4),   # Labor Day 2019, 2018, 2017
    (11,28),(11,22),(11,23), # Thanksgiving 2019, 2018, 2017
    (12,25),  # Christmas
}

# ---- Specific extreme-event dates to exclude ----
# Rationale documented in cell markdown above.
EXCLUDE_DATES = {
    # 2017 — power failures, derailment, Penn Station overflow
    "2017-04-21",
    "2017-05-07","2017-05-08","2017-05-09",
    "2017-06-05","2017-06-27","2017-06-28","2017-06-29",
    "2017-07-17",
    # Penn Station repairs Jul 10 – Sep 1 (ridership inflated by overflow)
    *[f"2017-07-{d:02d}" for d in range(10,32)],
    *[f"2017-08-{d:02d}" for d in range(1,32)],
    "2017-09-01",
    # 2018 — L train prep weekend shutdowns
    "2018-08-11","2018-08-12","2018-08-18","2018-08-19",
    "2018-10-06","2018-10-07","2018-10-13","2018-10-14",
    "2018-10-20","2018-10-21","2018-10-27","2018-10-28",
    "2018-11-10","2018-11-11","2018-11-17","2018-11-18",
    # 2019 — L train cancellation shock (all January) + prep weekends
    *[f"2019-01-{d:02d}" for d in range(1,32)],
    "2019-02-02","2019-02-03","2019-02-09","2019-02-10",
    "2019-02-16","2019-02-17","2019-02-23","2019-02-24",
    "2019-03-02","2019-03-03","2019-03-09","2019-03-10",
    "2019-03-16","2019-03-17","2019-04-21",
}
# Validate all date strings — remove any malformed entries silently
# Validate all date strings and remove any malformed entries
valid_excl = set()
for s in EXCLUDE_DATES:
    try:
        datetime.date.fromisoformat(s)
        valid_excl.add(s)
    except ValueError:
        pass
EXCLUDE_DATES = valid_excl

print(f"Exclusion flags loaded: {len(HOLIDAY_MD)} holiday patterns | "
      f"{len(EXCLUDE_DATES)} specific event dates")
print(f"  2017 event dates: {sum(1 for d in EXCLUDE_DATES if d.startswith('2017'))}")
print(f"  2018 event dates: {sum(1 for d in EXCLUDE_DATES if d.startswith('2018'))}")
print(f"  2019 event dates: {sum(1 for d in EXCLUDE_DATES if d.startswith('2019'))}")


# ── Download annual CSV ──────────────────────────────────────────
def download_year_csv(year: int) -> Path:
    """
    Download the annual MTA turnstile CSV from data.ny.gov.
    Caches to Drive. Validates column headers after download.
    Returns the path to the cached file.
    """
    dataset_id = DATA_NY_DATASET_IDS[year]
    url        = (f"https://data.ny.gov/api/views/{dataset_id}"
                  f"/rows.csv?accessType=DOWNLOAD")
    cache_f    = OUT_DIR / "raw_turnstile" / f"turnstile_{year}.csv"

    if cache_f.exists() and cache_f.stat().st_size > 1_000_000:
        print(f"  {year}: using cached file "
              f"({cache_f.stat().st_size/1e6:.0f} MB)")
        return cache_f

    print(f"  {year}: downloading from data.ny.gov...")
    print(f"  URL: {url}")

    with requests.get(url, stream=True, timeout=300) as r:
        if r.status_code != 200:
            raise RuntimeError(f"{year}: HTTP {r.status_code} — "
                               f"check dataset ID '{dataset_id}'")
        total = int(r.headers.get("content-length", 0))
        pbar  = tqdm(total=total, unit="B", unit_scale=True,
                     desc=f"  {year}") if total else None
        with open(cache_f, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
                    if pbar: pbar.update(len(chunk))
        if pbar: pbar.close()

    # Validate — must have required turnstile columns
    test = pd.read_csv(cache_f, nrows=3, low_memory=False)
    test.columns = test.columns.str.strip()
    print(f"  {year}: columns found: {list(test.columns)}")
    missing = TURNSTILE_REQUIRED_COLS - set(test.columns)
    if missing:
        cache_f.unlink()
        raise RuntimeError(f"{year}: missing columns {missing} — "
                           f"wrong dataset ID?")
    print(f"  {year}: download validated ✓")
    return cache_f


# ── Build station-daily per year ────────────────────────────────
def build_station_daily(year: int, chunk_size: int = 750_000) -> pd.DataFrame:
    """
    Reads the annual turnstile CSV in chunks, computes net entries,
    applies exclusion flags, and aggregates to station × date level.

    Parameters
    ----------
    year       : Calendar year (2017, 2018, or 2019).
    chunk_size : Rows per chunk. 750 000 works well in Colab.

    Returns
    -------
    DataFrame with columns:
      station_name, date, dow (0–6), dow_label, net_entries
    """
    cache_out = OUT_DIR / f"station_daily_{year}.csv"
    if cache_out.exists():
        df = pd.read_csv(cache_out, parse_dates=["date"])
        print(f"  {year}: loaded from cache — "
              f"{df['station_name'].nunique()} stations | "
              f"{df['date'].nunique()} days | {len(df):,} rows")
        return df

    raw_csv = download_year_csv(year)

    print(f"\n  {year}: reading in chunks (chunk_size={chunk_size:,})...")
    frames  = []
    reader  = pd.read_csv(
        raw_csv,
        usecols=["C/A","Unit","SCP","Station","Date","Time","Entries"],
        chunksize=chunk_size,
        low_memory=False,
    )

    for chunk in tqdm(reader, desc=f"  {year} chunks"):
        chunk.columns = chunk.columns.str.strip()

        # Parse DATETIME; drop unparseable rows
        chunk["DATETIME"] = pd.to_datetime(
            chunk["Date"].astype(str).str.strip() + " " +
            chunk["Time"].astype(str).str.strip(),
            format="%m/%d/%Y %H:%M:%S",
            errors="coerce",
        )
        chunk = chunk.dropna(subset=["DATETIME"])
        # Restrict to target year (files sometimes bleed into adjacent years)
        chunk = chunk[chunk["DATETIME"].dt.year == year].copy()
        if chunk.empty:
            continue

        chunk["Entries"] = pd.to_numeric(chunk["Entries"], errors="coerce")
        chunk = chunk.dropna(subset=["Entries"])
        frames.append(chunk[["C/A","Unit","SCP","Station","DATETIME","Entries"]])

    if not frames:
        raise RuntimeError(f"{year}: no valid records found after reading CSV.")

    raw = pd.concat(frames, ignore_index=True)
    print(f"    Raw rows: {len(raw):,}")

    # Sort within each physical turnstile unit before differencing
    unit_key = ["C/A","Unit","SCP"]
    raw = raw.sort_values(unit_key + ["DATETIME"]).reset_index(drop=True)

    # Net entries: diff within unit; negative = counter reset → 0;
    # >10 000 per audit = implausible → cap at 10 000
    raw["net_entries"] = (
        raw.groupby(unit_key)["Entries"]
           .diff()
           .clip(lower=0, upper=10_000)
           .fillna(0)
    )

    # Date and DOW columns
    raw["date"]      = raw["DATETIME"].dt.normalize()
    raw["dow"]       = raw["DATETIME"].dt.dayofweek
    raw["dow_label"] = raw["dow"].map(DOW_MAP)

    # Apply exclusion flags
    raw["date_str"]  = raw["date"].dt.strftime("%Y-%m-%d")
    raw["month_day"] = list(zip(raw["date"].dt.month, raw["date"].dt.day))
    holiday_mask     = raw["month_day"].isin(HOLIDAY_MD)
    event_mask       = raw["date_str"].isin(EXCLUDE_DATES)
    excluded         = int((holiday_mask | event_mask).sum())
    raw = raw[~(holiday_mask | event_mask)].drop(
        columns=["date_str","month_day"]
    ).copy()
    print(f"    Excluded rows: {excluded:,} | Remaining: {len(raw):,}")

    # Normalise Station name to uppercase for crosswalk consistency
    raw["Station"] = raw["Station"].astype(str).str.strip().str.upper()

    # Aggregate to station × date
    daily = (
        raw.groupby(["Station","date","dow","dow_label"])["net_entries"]
           .sum()
           .reset_index()
           .rename(columns={"Station": "station_name"})
    )
    daily.to_csv(cache_out, index=False)
    print(f"    Final: {daily['station_name'].nunique()} stations | "
          f"{daily['date'].nunique()} days | {len(daily):,} rows")
    return daily


# ── Build 3-year pre-COVID average ──────────────────────────────
def build_precovid_average(cleaned: dict) -> pd.DataFrame:
    """
    Compute the 3-year averaged pre-COVID baseline.

    For each station × day-of-week, computes the mean of
    mean_entries across 2017, 2018, and 2019. Outer-joins years
    so stations present in only 2 of 3 years are still retained
    (skipna=True). Stations with fewer than 3 years are flagged.

    This averaged baseline is more robust than any single year
    because it smooths year-specific service disruptions.

    Returns
    -------
    DataFrame with columns:
      station_name, dow, dow_label,
      mean_entries_2017, mean_entries_2018, mean_entries_2019,
      mean_entries_precovid, n_years_available
    """
    cache_out = OUT_DIR / "station_daily_precovid_avg.csv"
    if cache_out.exists():
        df = pd.read_csv(cache_out)
        print(f"Pre-COVID average: loaded from cache — "
              f"{df['station_name'].nunique()} stations | "
              f"{len(df):,} rows")
        return df

    frames = []
    for yr in PRE_COVID_YEARS:
        if yr not in cleaned:
            print(f"  Warning: {yr} missing — skipping")
            continue
        dow_agg = (
            cleaned[yr]
            .groupby(["station_name","dow","dow_label"])["net_entries"]
            .mean()
            .reset_index()
            .rename(columns={"net_entries": f"mean_entries_{yr}"})
        )
        frames.append(dow_agg.set_index(["station_name","dow","dow_label"]))

    combined = frames[0]
    for f in frames[1:]:
        combined = combined.join(f, how="outer")
    combined = combined.reset_index()

    yr_cols = [f"mean_entries_{yr}" for yr in PRE_COVID_YEARS
               if f"mean_entries_{yr}" in combined.columns]
    combined["mean_entries_precovid"] = combined[yr_cols].mean(axis=1, skipna=True)
    combined["n_years_available"]     = combined[yr_cols].notna().sum(axis=1)

    n_partial = int((combined["n_years_available"] < 3).sum())
    combined.to_csv(cache_out, index=False)

    print(f"\nPre-COVID 3-year average built:")
    print(f"  Stations:           {combined['station_name'].nunique()}")
    print(f"  DOW rows:           {len(combined):,}")
    print(f"  Full 3-yr coverage: {(combined['n_years_available']==3).sum():,}")
    print(f"  Partial coverage:   {n_partial:,}")
    return combined


# ── RUN ─────────────────────────────────────────────────────────
print("=" * 60)
print("Building pre-COVID baseline: 2017, 2018, 2019")
print("=" * 60)

for yr in PRE_COVID_YEARS:
    if yr in cleaned:
        print(f"  {yr}: already in memory ({len(cleaned[yr]):,} rows)")
    else:
        print(f"\nYear {yr}:")
        cleaned[yr] = build_station_daily(yr)

print("\n" + "=" * 60)
print("Computing 3-year pre-COVID DOW average...")
precovid_avg = build_precovid_average(cleaned)

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
for yr in PRE_COVID_YEARS:
    if yr in cleaned:
        df = cleaned[yr]
        dow_range = df.groupby("dow")["date"].nunique()
        print(f"  cleaned[{yr}]: {df['station_name'].nunique()} stations | "
              f"{df['date'].nunique()} days | "
              f"DOW coverage: {dow_range.min()}–{dow_range.max()} days/DOW")
print(f"  precovid_avg:  {precovid_avg['station_name'].nunique()} stations")
print(f"\nFiles saved to: {OUT_DIR}")
print("\nNext: Run Section 1.2 — Station Complex Crosswalk")


Initialized empty cleaned dict.
Exclusion flags loaded: 13 holiday patterns | 124 specific event dates
  2017 event dates: 62
  2018 event dates: 16
  2019 event dates: 46
Building pre-COVID baseline: 2017, 2018, 2019

Year 2017:
  2017: loaded from cache — 380 stations | 247 days | 92,356 rows

Year 2018:
  2018: loaded from cache — 379 stations | 315 days | 117,701 rows

Year 2019:
  2019: loaded from cache — 379 stations | 285 days | 107,522 rows

Computing 3-year pre-COVID DOW average...
Pre-COVID average: loaded from cache — 381 stations | 2,662 rows

SUMMARY
  cleaned[2017]: 380 stations | 247 days | DOW coverage: 34–36 days/DOW
  cleaned[2018]: 379 stations | 315 days | DOW coverage: 39–48 days/DOW
  cleaned[2019]: 379 stations | 285 days | DOW coverage: 37–44 days/DOW
  precovid_avg:  381 stations

Files saved to: /content/drive/MyDrive/dow_ridership_paper/outputs

Next: Run Section 1.2 — Station Complex Crosswalk


### 1.2 MTA Station Complex Crosswalk

**Purpose:** Load the official MTA Subway Stations and Complexes reference file, which maps every station name to a `complex_id` with confirmed latitude/longitude coordinates.

**Why this file matters:** Pre-COVID turnstile data uses MTA-internal `STATION` text strings that don't map 1:1 to station complexes. Post-COVID hourly data already has `station_complex_id` natively. Section 2.2 uses this file to fuzzy-match pre-COVID station names to complex IDs so that all three study periods share a common station universe.

**Data source:** Fomba Kassoh's verified GitHub repository (`hawa1983/Capstone`). This is the authoritative source — the equivalent `data.ny.gov` endpoint returns an entrance-level file with different columns and no lat/lon.


In [6]:
# ==============================================================
# Section 1.2 — MTA Station Complex Crosswalk
#
# Purpose:
#   Load the MTA Subway Stations and Complexes file from GitHub
#   and build a clean reference table with:
#     complex_id, stop_name, latitude, longitude,
#     borough, daytime_routes, structure_type, ada
#
# Source:
#   Fomba Kassoh GitHub (verified columns, 445 rows):
#   https://raw.githubusercontent.com/hawa1983/Capstone/
#   refs/heads/main/MTA_Subway_Stations_and_Complexes.csv
#
# Column names (confirmed from file inspection):
#   Complex ID | Is Complex | Number Of Stations In Complex
#   Stop Name | Display Name | Constituent Station Names
#   Station IDs | GTFS Stop IDs | Borough | CBD
#   Daytime Routes | Structure Type | Latitude | Longitude
#   ADA | ADA Notes
#
# Stale cache detection:
#   If the cache was downloaded from the old data.ny.gov endpoint
#   (which returns entrance-level data with 'entrance_latitude'),
#   it is deleted and re-downloaded from GitHub automatically.
# ==============================================================

if 'OUT_DIR' not in globals():
    OUT_DIR = Path("/content/drive/MyDrive/dow_ridership_paper/outputs")

STATION_COMPLEX_URL = (
    "https://raw.githubusercontent.com/hawa1983/Capstone/"
    "refs/heads/main/MTA_Subway_Stations_and_Complexes.csv"
)


def load_station_complexes(url: str) -> pd.DataFrame:
    """
    Download the MTA station complexes file from GitHub.
    Detects and removes stale cache from the old data.ny.gov source.
    Normalises column names to snake_case.
    """
    cache = OUT_DIR / "mta_station_complexes.csv"

    # Detect stale cache (old data.ny.gov file has 'entrance_latitude')
    if cache.exists():
        df_test = pd.read_csv(cache, nrows=1)
        df_test.columns = (df_test.columns.str.strip().str.lower()
                           .str.replace(r"[\s/]+","_",regex=True))
        if ("entrance_latitude" in df_test.columns
                or "latitude" not in df_test.columns):
            print("  Stale cache detected — deleting and re-downloading...")
            cache.unlink()

    if not cache.exists():
        print(f"  Downloading from GitHub: {url}")
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        df = pd.read_csv(StringIO(resp.text))
        df.to_csv(cache, index=False)
        print(f"  Saved to cache: {cache}")
    else:
        df = pd.read_csv(cache)

    # Normalise column names to snake_case
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(r"[\s/]+","_",regex=True))
    print(f"Station complexes loaded: {len(df):,} rows")
    print(f"Columns: {list(df.columns)}")
    return df


station_complex_df = load_station_complexes(STATION_COMPLEX_URL)

# Confirm key columns are present
for col in ["complex_id","stop_name","latitude","longitude"]:
    present = col in station_complex_df.columns
    sample  = station_complex_df[col].iloc[0] if present else "MISSING"
    print(f"  {col}: {'OK' if present else 'MISSING'} — sample: {sample}")

print("\nPreview:")
station_complex_df.head(3)


Station complexes loaded: 445 rows
Columns: ['complex_id', 'is_complex', 'number_of_stations_in_complex', 'stop_name', 'display_name', 'constituent_station_names', 'station_ids', 'gtfs_stop_ids', 'borough', 'cbd', 'daytime_routes', 'structure_type', 'latitude', 'longitude', 'ada', 'ada_notes']
  complex_id: OK — sample: 398
  stop_name: OK — sample: 77 St
  latitude: OK — sample: 40.77362
  longitude: OK — sample: -73.959874

Preview:


,complex_id,is_complex,number_of_stations_in_complex,stop_name,display_name,constituent_station_names,station_ids,gtfs_stop_ids,borough,cbd,daytime_routes,structure_type,latitude,longitude,ada,ada_notes
0,398,False,1,77 St,77 St (6),77 St,398,627,M,False,6,Subway,40.77,-73.96,0,NaN
1,399,False,1,68 St-Hunter College,68 St-Hunter College (6),68 St-Hunter College,399,628,M,False,6,Subway,40.77,-73.96,1,NaN
2,403,False,1,33 St,33 St (6),33 St,403,632,M,True,6,Subway,40.75,-73.98,0,NaN


### 1.3 Built Environment Data

Three data streams characterise the land-use and access environment around each station complex:

**1.3a — ACS Block Group Data:** American Community Survey 5-year estimates at the Census block-group level. Variables: total population, households, no-vehicle households, median household income, total commuters, transit commuters.
- 2019 ACS 5-year → pre-COVID period
- 2022 ACS 5-year → 2022 and 2024 periods (2024 BG 5-year not yet available)

**1.3b — LODES Workplace and Residence Area Characteristics:** Census LEHD LODES 8 job counts at the Census block level, aggregated to block groups. This analysis extends the standard office/retail/essential sectors to include:
- **Education jobs** (CNS13) — proxy for university and school commuter trip generation
- **Healthcare jobs** (CNS15) — proxy for telehealth-substitutable medical appointment trips
- **Public administration jobs** (CNS20) — proxy for government service virtualization
- **LODES RAC** (Residence Area Characteristics) — workers who *live* near each station, enabling a WAC/RAC destination-vs-origin ratio

**1.3c — Commuter Rail Terminal Proximity:** Distance from each station complex to the five major multimodal transfer terminals (Grand Central, Penn Station, Atlantic Terminal, Jamaica, Port Authority Bus Terminal). Stations within 0.25 miles of a terminal capture the suburban commuter drain that drives the Friday ridership depression documented in MTA commuter rail data.

These datasets are spatially joined to 0.5-mile station-complex buffers in Section 5.1.


In [7]:
# ==============================================================
# Section 1.3a — ACS Block Group Built-Environment Data
#
# Purpose:
#   Fetch ACS 5-year estimates at the Census block-group level
#   for all five NYC counties (Manhattan, Bronx, Brooklyn,
#   Queens, Staten Island).
#
# Variables fetched:
#   B01003_001E  total_population
#   B25044_001E  total_households           (vehicle availability)
#   B25044_003E  owner_no_vehicle_hh        (owner-occ, no vehicle)
#   B25044_010E  renter_no_vehicle_hh       (renter-occ, no vehicle)
#   B19013_001E  median_hh_income
#   B08301_001E  total_commuters
#   B08301_010E  transit_commuters
#
# Note on B25044:
#   The standard no-vehicle table (B08201) is used in many studies,
#   but B25044 gives vehicle availability by tenure (owner/renter),
#   which allows clean summation of no-vehicle households:
#   no_vehicle_hh = B25044_003E + B25044_010E
#
# ACS year mapping:
#   Pre-COVID period (2019) → ACS 2019 5-year
#   Post-COVID periods (2022, 2024) → ACS 2022 5-year
#   (2024 block-group 5-year data not yet available as of analysis date)
#
# Outputs:
#   acs_bg_2019.csv
#   acs_bg_2022.csv
# ==============================================================

if 'OUT_DIR' not in globals():
    OUT_DIR = Path("/content/drive/MyDrive/dow_ridership_paper/outputs")

# Optional Census API key — improves rate limits but not required
if 'CENSUS_API_KEY' not in globals():
    CENSUS_API_KEY = None

ACS_YEAR_MAP    = {2019: 2019, 2022: 2022, 2024: 2022}
ACS_YEARS_TO_FETCH = sorted(set(ACS_YEAR_MAP.values()))

NYC_COUNTIES = {
    "061": "Manhattan",  "005": "Bronx",
    "047": "Brooklyn",   "081": "Queens", "085": "Staten Island",
}

ACS_VARS = {
    "B01003_001E": "total_population",
    "B25044_001E": "total_households",
    "B25044_003E": "owner_no_vehicle_hh",
    "B25044_010E": "renter_no_vehicle_hh",
    "B19013_001E": "median_hh_income",
    "B08301_001E": "total_commuters",
    "B08301_010E": "transit_commuters",
}


def fetch_acs_block_groups(acs_year: int,
                            force_refresh: bool = False) -> pd.DataFrame:
    """
    Fetch ACS 5-year block-group data for NYC counties.
    Caches result to Drive. Set force_refresh=True to re-download.
    """
    cache = OUT_DIR / f"acs_bg_{acs_year}.csv"
    if cache.exists() and not force_refresh:
        print(f"  ACS {acs_year}: loaded from cache")
        return pd.read_csv(cache, dtype={"GEOID": str})

    base_url  = f"https://api.census.gov/data/{acs_year}/acs/acs5"
    var_list  = ["NAME"] + list(ACS_VARS.keys())
    all_frames = []

    for county_fips, county_name in NYC_COUNTIES.items():
        params = {
            "get": ",".join(var_list),
            "for": "block group:*",
            "in":  f"state:36 county:{county_fips} tract:*",
        }
        if CENSUS_API_KEY:
            params["key"] = CENSUS_API_KEY

        print(f"  ACS {acs_year}: {county_name} ({county_fips})...", end=" ")
        resp = requests.get(base_url, params=params, timeout=120)
        if resp.status_code != 200:
            raise RuntimeError(f"ACS {acs_year} county {county_fips}: "
                               f"HTTP {resp.status_code}")
        data    = resp.json()
        df_cty  = pd.DataFrame(data[1:], columns=data[0])
        df_cty["county_fips"] = county_fips
        df_cty["county_name"] = county_name
        all_frames.append(df_cty)
        print(f"{len(df_cty):,} BGs")

    df = pd.concat(all_frames, ignore_index=True)

    # Build 12-digit GEOID
    df["GEOID"] = (df["state"].astype(str).str.zfill(2)
                   + df["county"].astype(str).str.zfill(3)
                   + df["tract"].astype(str).str.zfill(6)
                   + df["block group"].astype(str).str.zfill(1))

    df = df.rename(columns=ACS_VARS)

    # Coerce to numeric; ACS missing = negative codes → NaN
    for col in ACS_VARS.values():
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df.loc[df[col] < 0, col] = np.nan

    # Combine owner/renter no-vehicle households
    df["no_vehicle_hh"] = (df["owner_no_vehicle_hh"].fillna(0)
                           + df["renter_no_vehicle_hh"].fillna(0))
    df.loc[df["total_households"].isna(), "no_vehicle_hh"] = np.nan

    keep_cols = ["GEOID","NAME","county_fips","county_name",
                 "total_population","total_households","no_vehicle_hh",
                 "median_hh_income","total_commuters","transit_commuters"]
    df = df[keep_cols].copy()
    df.to_csv(cache, index=False)
    print(f"  ACS {acs_year}: {len(df):,} block groups saved to {cache}")
    return df


acs_data = {}
for acs_year in ACS_YEARS_TO_FETCH:
    print(f"\nFetching ACS {acs_year}...")
    acs_data[acs_year] = fetch_acs_block_groups(acs_year, force_refresh=False)

print("\nACS fetch complete.")
for yr, df in acs_data.items():
    print(f"  ACS {yr}: {len(df):,} block groups | "
          f"no_vehicle_hh nonzero: {(df['no_vehicle_hh']>0).sum():,}")



Fetching ACS 2019...
  ACS 2019: loaded from cache

Fetching ACS 2022...
  ACS 2022: loaded from cache

ACS fetch complete.
  ACS 2019: 6,493 block groups | no_vehicle_hh nonzero: 6,129
  ACS 2022: 6,807 block groups | no_vehicle_hh nonzero: 6,244


In [8]:
# ==============================================================
# Section 1.3b — LEHD/LODES Workplace Job Data
#
# Purpose:
#   Download LODES 8 Workplace Area Characteristics (WAC) files
#   for New York State and aggregate block-level job counts to
#   Census block-group level for spatial joining in Section 5.1.
#
# Data source:
#   US Census LEHD LODES 8
#   https://lehd.ces.census.gov/data/lodes/LODES8/ny/wac/
#   File: ny_wac_S000_JT00_{year}.csv.gz
#
# Job sector variables:
#   C000   total_jobs
#   CNS09  finance_jobs         } combined as
#   CNS12  professional_jobs    } office_jobs
#   CNS07  retail_jobs
#   CNS15  healthcare_jobs      } combined as
#   CNS18  food_service_jobs    } essential_jobs
#
# Year mapping:
#   Pre-COVID (2019) → LODES 2019
#   Post-COVID (2022, 2024) → LODES 2021 (latest available)
#
# Processing:
#   1. Download .csv.gz, decompress in memory.
#   2. Keep required sector columns.
#   3. Derive office_jobs = finance + professional.
#   4. Derive essential_jobs = healthcare + food service.
#   5. Truncate 15-digit block FIPS to 12-digit block-group GEOID.
#   6. Aggregate block-level counts to block-group sums.
#
# Outputs:
#   lodes_wac_2019.csv
#   lodes_wac_2021.csv
# ==============================================================

if 'OUT_DIR' not in globals():
    OUT_DIR = Path("/content/drive/MyDrive/dow_ridership_paper/outputs")

LODES_BASE  = "https://lehd.ces.census.gov/data/lodes/LODES8/ny/wac/"
LODES_YEARS = {2019: 2019, 2022: 2021, 2024: 2021}

LODES_VARS = {
    "w_geocode":  "block_fips",
    "C000":       "total_jobs",
    # Office sector (high telework substitutability)
    "CNS09":      "finance_jobs",
    "CNS12":      "professional_jobs",
    # Retail sector
    "CNS07":      "retail_jobs",
    # Essential / in-person services
    "CNS15":      "healthcare_jobs",
    "CNS18":      "food_service_jobs",
    # Education sector — university/school commuter trip generation
    # Virtual class adoption reduces Mon/Fri trips at station-adjacent campuses
    "CNS13":      "education_jobs",
    # Public administration — government service virtualization
    # Online portals/remote hearings reduce trips to civic center stations
    "CNS20":      "public_admin_jobs",
}


def fetch_lodes_wac(lodes_year: int) -> pd.DataFrame:
    """
    Download, decompress, and aggregate LODES WAC file.
    Caches the block-group aggregation to Drive.
    """
    cache = OUT_DIR / f"lodes_wac_{lodes_year}.csv"
    if cache.exists():
        print(f"  LODES {lodes_year}: loaded from cache")
        return pd.read_csv(cache, dtype={"GEOID": str})

    filename = f"ny_wac_S000_JT00_{lodes_year}.csv.gz"
    url      = LODES_BASE + filename
    print(f"  Downloading: {url}")

    resp = requests.get(url, timeout=120)
    resp.raise_for_status()

    # Read compressed gz directly from memory
    df = pd.read_csv(BytesIO(resp.content), compression="gzip",
                     dtype={"w_geocode": str}, low_memory=False)

    keep = [c for c in LODES_VARS if c in df.columns]
    df = df[keep].rename(columns=LODES_VARS)

    # Validate required columns survived the rename
    required = ["block_fips","total_jobs","finance_jobs","professional_jobs",
                "retail_jobs","healthcare_jobs","food_service_jobs",
                "education_jobs","public_admin_jobs"]
    missing  = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(f"LODES {lodes_year}: missing columns {missing}")

    # Coerce to numeric
    for col in [c for c in required if c != "block_fips"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    # Derived sector aggregations
    df["office_jobs"]    = df["finance_jobs"]    + df["professional_jobs"]
    df["essential_jobs"] = df["healthcare_jobs"] + df["food_service_jobs"]
    # Virtualization exposure composite:
    # education + healthcare + public admin = sectors with highest virtual
    # substitutability of in-person trip purposes beyond office commuting
    df["virtualization_jobs"] = (
        df["education_jobs"].fillna(0)
        + df["healthcare_jobs"].fillna(0)
        + df["public_admin_jobs"].fillna(0)
    )

    # Aggregate to block group (12-digit GEOID)
    df["GEOID"] = df["block_fips"].astype(str).str.zfill(15).str[:12]

    agg_cols = ["total_jobs","finance_jobs","professional_jobs","retail_jobs",
                "healthcare_jobs","food_service_jobs","office_jobs","essential_jobs",
                "education_jobs","public_admin_jobs","virtualization_jobs"]
    bg = df.groupby("GEOID", as_index=False)[agg_cols].sum()
    bg.to_csv(cache, index=False)
    print(f"  LODES {lodes_year}: {len(bg):,} block groups saved to {cache}")
    return bg


lodes_data = {}
for lodes_yr in sorted(set(LODES_YEARS.values())):
    print(f"\nFetching LODES {lodes_yr}...")
    lodes_data[lodes_yr] = fetch_lodes_wac(lodes_yr)

print("\nLODES fetch complete.")
for yr, df in lodes_data.items():
    print(f"  LODES {yr}: {len(df):,} block groups | "
          f"total_jobs sum: {df['total_jobs'].sum():,.0f}")


# ==============================================================
# Section 1.3c — LODES Residence Area Characteristics (RAC)
#
# Purpose:
#   Download LODES RAC to identify workers who LIVE near each
#   station (as opposed to WAC which counts workers who WORK there).
#
# The WAC/RAC ratio is a destination-vs-origin index:
#   High ratio  → station is a destination (workers arrive: office core)
#   Low ratio   → station is an origin (workers depart: residential)
#   ~Equal      → mixed-use or transfer station
#
# This directly captures the suburban commuter flow structure.
# Outer-borough residential stations where many workers live but
# few work will show low WAC/RAC — and their DOW profiles will
# reflect commute departure patterns, not arrival patterns.
#
# Source: LEHD LODES8 ny_rac_S000_JT00_{year}.csv.gz
# ==============================================================

LODES_RAC_BASE  = "https://lehd.ces.census.gov/data/lodes/LODES8/ny/rac/"
LODES_RAC_YEARS = sorted(set(LODES_YEARS.values()))   # same year mapping as WAC

RAC_VARS = {
    "h_geocode": "block_fips",
    "C000":      "total_resident_workers",
    # Resident workers by earnings band — low earners less likely to telework
    "CE01":      "resident_workers_low_earn",    # < $1,250/month
    "CE03":      "resident_workers_high_earn",   # > $3,333/month
}

rac_data = {}

for lodes_yr in LODES_RAC_YEARS:
    cache = OUT_DIR / f"lodes_rac_{lodes_yr}.csv"

    if cache.exists():
        rac_data[lodes_yr] = pd.read_csv(cache, dtype={"GEOID": str})
        print(f"  LODES RAC {lodes_yr}: loaded from cache")
        continue

    filename = f"ny_rac_S000_JT00_{lodes_yr}.csv.gz"
    url      = LODES_RAC_BASE + filename
    print(f"  Downloading LODES RAC {lodes_yr}: {url}")

    resp = requests.get(url, timeout=120)
    resp.raise_for_status()

    df_rac = pd.read_csv(BytesIO(resp.content), compression="gzip",
                         dtype={"h_geocode": str}, low_memory=False)

    keep = [c for c in RAC_VARS if c in df_rac.columns]
    df_rac = df_rac[keep].rename(columns=RAC_VARS)

    for col in ["total_resident_workers","resident_workers_low_earn",
                "resident_workers_high_earn"]:
        if col in df_rac.columns:
            df_rac[col] = pd.to_numeric(df_rac[col], errors="coerce").fillna(0)

    # Derive high-earner share (proxy for telework-eligible resident workers)
    df_rac["pct_high_earn_residents"] = (
        df_rac["resident_workers_high_earn"]
        / df_rac["total_resident_workers"].replace(0, np.nan)
    )

    df_rac["GEOID"] = df_rac["block_fips"].astype(str).str.zfill(15).str[:12]

    agg_rac_cols = [c for c in ["total_resident_workers",
                                "resident_workers_low_earn",
                                "resident_workers_high_earn"]
                   if c in df_rac.columns]
    bg_rac = df_rac.groupby("GEOID", as_index=False)[agg_rac_cols].sum()

    # Recalculate pct_high_earn at block-group level
    bg_rac["pct_high_earn_residents"] = (
        bg_rac["resident_workers_high_earn"]
        / bg_rac["total_resident_workers"].replace(0, np.nan)
    )

    bg_rac.to_csv(cache, index=False)
    rac_data[lodes_yr] = bg_rac
    print(f"  LODES RAC {lodes_yr}: {len(bg_rac):,} block groups saved")

print("\nLODES RAC fetch complete.")
for yr, df in rac_data.items():
    print(f"  RAC {yr}: {len(df):,} block groups | "
          f"resident workers: {df['total_resident_workers'].sum():,.0f}")



Fetching LODES 2019...
  Downloading: https://lehd.ces.census.gov/data/lodes/LODES8/ny/wac/ny_wac_S000_JT00_2019.csv.gz
  LODES 2019: 15,698 block groups saved to /content/drive/MyDrive/dow_ridership_paper/outputs/lodes_wac_2019.csv

Fetching LODES 2021...
  Downloading: https://lehd.ces.census.gov/data/lodes/LODES8/ny/wac/ny_wac_S000_JT00_2021.csv.gz
  LODES 2021: 15,678 block groups saved to /content/drive/MyDrive/dow_ridership_paper/outputs/lodes_wac_2021.csv

LODES fetch complete.
  LODES 2019: 15,698 block groups | total_jobs sum: 9,547,776
  LODES 2021: 15,678 block groups | total_jobs sum: 8,676,609
  LODES RAC 2019: 15,709 block groups saved
  LODES RAC 2021: 15,774 block groups saved

LODES RAC fetch complete.
  RAC 2019: 15,709 block groups | resident workers: 9,144,984
  RAC 2021: 15,774 block groups | resident workers: 8,279,762


### 1.3d Commuter Rail Terminal Proximity

**Purpose:** Compute the distance from each station complex to the five major multimodal transfer terminals that feed suburban commuters into the NYC subway. This is a direct operationalization of the suburban commuter drain mechanism: when Metro-North and LIRR Friday AM trains run virtually empty, the subway stations within walking distance of those terminals lose a concentrated, predictable ridership block.

**Theoretical basis:** <cite>Metro-North 2024 commutation ridership to/from Manhattan is 42% lower than 1990, with commutation now only 31% of East of Hudson ridership compared to 62% in 1990.</cite> The LIRR pattern mirrors this. The suburban commuter loss is not uniformly distributed across the subway — it is spatially concentrated at transfer hubs. A station-level proximity variable captures this geographic concentration.

**Five terminals covered:**
- **Grand Central Terminal** — Metro-North (Hudson, Harlem, New Haven, Port Jervis, Pascack Valley lines)
- **Penn Station** — LIRR (all branches), NJ Transit (all lines), Amtrak
- **Atlantic Terminal** — LIRR (Flatbush Ave branch)
- **Jamaica Station** — LIRR (Jamaica Hub, AirTrain connection)
- **Port Authority Bus Terminal** — NJ Transit express buses, suburban coaches

**Variables produced:**
- `dist_to_nearest_terminal_miles` — Haversine distance to closest terminal
- `is_terminal_adjacent` — binary flag: within 0.25 miles (direct walking transfer zone)
- `log_dist_to_nearest_terminal` — log-transformed distance for regression
- `nearest_terminal_name` — name of closest terminal (for interpretability)


In [9]:
# ==============================================================
# Section 1.3d — Commuter Rail Terminal Proximity
#
# Purpose:
#   Compute station-complex distance to the five major suburban
#   commuter rail terminals. Stations near these terminals absorb
#   the Friday ridership depression caused by hybrid work reducing
#   suburban commuter flows — documented in MTA's own annual
#   ridership reports showing Metro-North commutation 42% below
#   1990 levels and LIRR Friday trains running well below capacity.
#
# Why this matters analytically:
#   Built-environment variables like office job density are PROXIES
#   for hybrid work exposure. Terminal proximity is a DIRECT measure
#   of the mechanism: suburban commuters stop coming on Fridays,
#   and the subway stations that serve as their transfer nodes bear
#   the ridership loss disproportionately.
#
# Detection threshold: 0.25 miles
#   This captures stations in the direct walking transfer zone —
#   riders who arrive at Grand Central and walk to a subway entrance.
#   Beyond 0.25 miles the transfer becomes a bus/taxi trip and the
#   direct ridership link weakens substantially.
#
# Output columns added to station_features in Section 5.2:
#   dist_to_nearest_terminal_miles
#   nearest_terminal_name
#   is_terminal_adjacent          (binary: within 0.25 miles)
#   log_dist_to_nearest_terminal
# ==============================================================

import numpy as np
import pandas as pd

# ── Commuter rail terminal coordinates ──────────────────────────
# Coordinates verified against Google Maps / MTA official sources

COMMUTER_TERMINALS = {
    "Grand_Central":          (40.7527, -73.9772),
    "Penn_Station":           (40.7506, -73.9971),
    "Atlantic_Terminal_LIRR": (40.6841, -73.9766),
    "Jamaica_Station":        (40.7018, -73.8088),
    "Port_Authority_Bus":     (40.7572, -73.9901),
}

TERMINAL_ADJACENT_MILES = 0.25   # walking transfer zone threshold


def haversine_miles(lat1: float, lon1: float,
                    lat2: float, lon2: float) -> float:
    """
    Compute Haversine great-circle distance in miles.
    Accurate to within ~0.1% for NYC-scale distances.
    """
    R = 3958.8   # Earth radius in miles
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = (np.sin(dlat / 2) ** 2
         + np.cos(np.radians(lat1))
         * np.cos(np.radians(lat2))
         * np.sin(dlon / 2) ** 2)
    return R * 2 * np.arcsin(np.sqrt(a))


def compute_terminal_proximity(df: pd.DataFrame,
                                lat_col: str = "complex_lat",
                                lon_col: str = "complex_lon") -> pd.DataFrame:
    """
    For each station complex, find the nearest commuter rail terminal
    and compute distance metrics.

    Parameters
    ----------
    df      : DataFrame with station complex lat/lon
    lat_col : column name for latitude
    lon_col : column name for longitude

    Returns
    -------
    df with new columns:
      dist_to_nearest_terminal_miles
      nearest_terminal_name
      is_terminal_adjacent
      log_dist_to_nearest_terminal
    """
    results = []

    for _, row in df.iterrows():
        lat, lon = row.get(lat_col), row.get(lon_col)

        if pd.isna(lat) or pd.isna(lon):
            results.append({
                "dist_to_nearest_terminal_miles": np.nan,
                "nearest_terminal_name":          "unknown",
                "is_terminal_adjacent":            0,
                "log_dist_to_nearest_terminal":    np.nan,
            })
            continue

        # Compute distance to every terminal
        distances = {
            name: haversine_miles(lat, lon, t_lat, t_lon)
            for name, (t_lat, t_lon) in COMMUTER_TERMINALS.items()
        }

        nearest_name = min(distances, key=distances.get)
        nearest_dist = distances[nearest_name]

        results.append({
            "dist_to_nearest_terminal_miles": round(nearest_dist, 4),
            "nearest_terminal_name":          nearest_name,
            "is_terminal_adjacent":           int(nearest_dist <= TERMINAL_ADJACENT_MILES),
            "log_dist_to_nearest_terminal":   np.log1p(nearest_dist),
        })

    proximity_df = pd.DataFrame(results, index=df.index)
    return pd.concat([df, proximity_df], axis=1)


# ── Session recovery ────────────────────────────────────────────
if "OUT_DIR" not in globals():
    from pathlib import Path
    OUT_DIR = Path("/content/drive/MyDrive/dow_ridership_paper/outputs")

# ── Try to load station_complex_df if available ────────────────
if "station_complex_df" not in globals():
    sc_cache = OUT_DIR / "mta_station_complexes.csv"
    if sc_cache.exists():
        station_complex_df = pd.read_csv(sc_cache)
        station_complex_df.columns = (
            station_complex_df.columns.str.strip().str.lower()
            .str.replace(r"[\s/]+", "_", regex=True)
        )
        print(f"station_complex_df recovered from cache: {len(station_complex_df):,} rows")
    else:
        print("station_complex_df not found — run Section 1.2 first.")
        station_complex_df = None

# ── Compute terminal proximity if station data is available ─────
if station_complex_df is not None:

    # Confirm lat/lon columns exist
    lat_col = next((c for c in ["latitude","complex_lat","lat"]
                    if c in station_complex_df.columns), None)
    lon_col = next((c for c in ["longitude","complex_lon","lon"]
                    if c in station_complex_df.columns), None)

    if lat_col and lon_col:
        station_complex_df = compute_terminal_proximity(
            station_complex_df, lat_col, lon_col
        )

        # Summary
        n_adjacent = station_complex_df["is_terminal_adjacent"].sum()
        print(f"Terminal proximity computed for {len(station_complex_df):,} station complexes.")
        print(f"  Within {TERMINAL_ADJACENT_MILES} miles of a terminal: {n_adjacent} stations")
        print()
        print("Distribution by nearest terminal:")
        print(station_complex_df["nearest_terminal_name"].value_counts().to_string())
        print()
        print("Terminal-adjacent stations:")
        adj_cols = ["stop_name","nearest_terminal_name",
                    "dist_to_nearest_terminal_miles"]
        avail_cols = [c for c in adj_cols if c in station_complex_df.columns]
        print(
            station_complex_df.loc[
                station_complex_df["is_terminal_adjacent"] == 1,
                avail_cols
            ].sort_values("dist_to_nearest_terminal_miles")
            .to_string(index=False)
        )

        # Cache updated station complex file
        prox_cache = OUT_DIR / "station_complexes_with_proximity.csv"
        station_complex_df.to_csv(prox_cache, index=False)
        print(f"\nSaved: {prox_cache}")

    else:
        print(f"Lat/lon columns not found. Available: {list(station_complex_df.columns)}")
else:
    print("Skipping terminal proximity — station_complex_df not available.")
    print("Run Section 1.2 then re-run this cell.")


Terminal proximity computed for 445 station complexes.
  Within 0.25 miles of a terminal: 8 stations

Distribution by nearest terminal:
nearest_terminal_name
Atlantic_Terminal_LIRR    184
Grand_Central             127
Jamaica_Station            57
Port_Authority_Bus         43
Penn_Station               34

Terminal-adjacent stations:
                                 stop_name  nearest_terminal_name  dist_to_nearest_terminal_miles
                       Grand Central-42 St          Grand_Central                            0.05
                  Atlantic Av-Barclays Ctr Atlantic_Terminal_LIRR                            0.06
        Sutphin Blvd-Archer Av-JFK Airport        Jamaica_Station                            0.10
Times Sq-42 St/Port Authority Bus Terminal     Port_Authority_Bus                            0.17
                              Lafayette Av Atlantic_Terminal_LIRR                            0.20
                                 Fulton St Atlantic_Terminal_LIRR          

---
## SECTION 2: Data Processing

Transform raw ridership data into the 7-day normalised ridership vectors used for K-means clustering. Three sub-tasks:

1. **Section 2.1** — Download post-COVID (2022–2024) daily ridership at station-complex level directly from the MTA Hourly Ridership Socrata API. This data is pre-aggregated to complex level, so no turnstile-to-complex crosswalk is needed.

2. **Section 2.2** — Join pre-COVID turnstile station names to station complex IDs using the MTA station complex reference file. Exact normalized-name matching first, then difflib fuzzy matching for non-exact cases.

3. **Section 2.3** — Construct 7-day normalised ridership vectors. For each station complex and study period: pivot mean entries to a 7-column DOW profile, normalise each row to sum to 1.0 (capturing demand *shape* not *scale*), flag low-volume stations. This is the feature matrix for K-means.


### 2.1 Post-COVID Station-Complex Daily Ridership (2022, 2023, 2024)

**Source:** MTA Subway Hourly Ridership dataset (`wujg-7c2s` on data.ny.gov). Already aggregated to station-complex level by the MTA; contains `station_complex_id` natively — no crosswalk needed.

**Query strategy:** Use Socrata SoQL `$select` with `date_trunc_ymd()` and `sum(ridership)` to request daily totals server-side, avoiding downloading millions of hourly rows locally.

**Years acquired:** 2022 (main analysis), 2023 (robustness), 2024 (main analysis).


In [10]:
# ==============================================================
# Section 2.1 — FAST Post-COVID Station-Complex Daily Ridership
#
# Purpose:
#   Downloads daily post-COVID subway ridership at station-complex level
#   from the MTA Subway Hourly Ridership dataset.
#
# Extracted years:
#   2022, 2023, 2024
#
# Main analysis:
#   2022 and 2024
#
# Robustness:
#   2023
#
# Outputs saved to Google Drive / OUT_DIR:
#   station_complex_daily_2022.csv
#   station_complex_daily_2023.csv
#   station_complex_daily_2024.csv
#   station_complex_daily_main_2022_2024.csv
#   station_complex_daily_robustness_2023.csv
# ==============================================================

import time
import io
from pathlib import Path

import pandas as pd
import numpy as np
import requests


# -------------------------------------------------------
# Recover required globals
# -------------------------------------------------------

if "OUT_DIR" not in globals():
    OUT_DIR = Path("/content/drive/MyDrive/dow_ridership_paper/outputs")
    print(f"OUT_DIR recovered: {OUT_DIR}")

OUT_DIR.mkdir(parents=True, exist_ok=True)

if "DOW_MAP" not in globals():
    DOW_MAP = {
        0: "Mon",
        1: "Tue",
        2: "Wed",
        3: "Thu",
        4: "Fri",
        5: "Sat",
        6: "Sun",
    }

if "DOW_LABELS" not in globals():
    DOW_LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]


# -------------------------------------------------------
# Settings
# -------------------------------------------------------

DEBUG = True

def _log(msg):
    if DEBUG:
        print(msg, flush=True)


SESSION = requests.Session()
SESSION.headers.update(
    {
        "User-Agent": "Mozilla/5.0",
        "Accept-Encoding": "gzip, deflate",
    }
)

TIMEOUT = 180
MAX_RETRIES = 6
BACKOFF_BASE_SEC = 2.0

POST_BASE_URL = "https://data.ny.gov/resource/wujg-7c2s.csv"

POST_COVID_YEARS = [2022, 2023, 2024]
MAIN_MODEL_YEARS = [2022, 2024]
ROBUSTNESS_YEARS = [2023]

# -------------------------------------------------------
# One-time refresh setting
# -------------------------------------------------------
# Set True for this one run to redownload/rebuild files.
# After the files are successfully saved, change this to False
# so future notebook runs load from cache.

FORCE_REFRESH_POSTCOVID = False


# -------------------------------------------------------
# Network helper
# -------------------------------------------------------

def _request_csv(url: str, params: dict) -> pd.DataFrame:
    """
    Request a CSV from Data.NY.gov with retries.
    """

    last_err = None

    for attempt in range(MAX_RETRIES):
        try:
            r = SESSION.get(url, params=params, timeout=TIMEOUT)

            if r.status_code in (429, 500, 502, 503, 504):
                sleep_s = BACKOFF_BASE_SEC * (2 ** attempt)
                _log(
                    f"Retryable status {r.status_code}; "
                    f"sleeping {sleep_s:.1f}s"
                )
                time.sleep(sleep_s)
                continue

            if 400 <= r.status_code < 500:
                print("Request URL:")
                print(r.url)
                print("Response preview:")
                print(r.text[:1000])
                r.raise_for_status()

            r.raise_for_status()

            return pd.read_csv(io.StringIO(r.text))

        except requests.HTTPError as e:
            last_err = e
            raise

        except Exception as e:
            last_err = e
            sleep_s = BACKOFF_BASE_SEC * (2 ** attempt)
            _log(
                f"Request failed on attempt {attempt + 1}/{MAX_RETRIES}; "
                f"sleeping {sleep_s:.1f}s"
            )
            time.sleep(sleep_s)

    raise RuntimeError(f"Request failed after retries. Last error: {last_err}")


# -------------------------------------------------------
# Fetch one year
# -------------------------------------------------------

def fetch_station_complex_daily_post(year: int) -> pd.DataFrame:
    """
    Pull already-aggregated daily ridership for one year from
    the post-COVID MTA hourly ridership dataset.

    Output level:
        station_complex_id × date
    """

    cache = OUT_DIR / f"station_complex_daily_{year}.csv"

    if cache.exists() and not FORCE_REFRESH_POSTCOVID:
        print(f"  {year}: loaded station-complex daily from cache")
        return pd.read_csv(cache, parse_dates=["date"])

    if cache.exists() and FORCE_REFRESH_POSTCOVID:
        print(
            f"  {year}: cache exists, but FORCE_REFRESH_POSTCOVID=True, "
            "so redownloading..."
        )

    start_iso = f"{year}-01-01"
    end_iso = f"{year + 1}-01-01"

    params = {
        "$select": (
            "date_trunc_ymd(transit_timestamp) AS date, "
            "station_complex_id, "
            "station_complex, "
            "borough, "
            "sum(ridership) AS net_entries"
        ),
        "$where": (
            f"transit_timestamp >= '{start_iso}T00:00:00.000' "
            f"AND transit_timestamp < '{end_iso}T00:00:00.000'"
        ),
        "$group": "date, station_complex_id, station_complex, borough",
        "$limit": 500000,
    }

    _log(f"POST {year}: fetching grouped daily ridership...")

    df = _request_csv(POST_BASE_URL, params)

    if df.empty:
        raise RuntimeError(f"{year}: no post-COVID ridership rows returned.")

    # -------------------------------------------------------
    # Clean columns
    # -------------------------------------------------------

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[\s/]+", "_", regex=True)
    )

    required_cols = {
        "date",
        "station_complex_id",
        "station_complex",
        "net_entries",
    }

    missing = required_cols - set(df.columns)

    if missing:
        raise RuntimeError(
            f"{year}: missing expected columns: {missing}\n"
            f"Columns found: {list(df.columns)}"
        )

    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    df["station_complex_id"] = pd.to_numeric(
        df["station_complex_id"],
        errors="coerce"
    )

    df["station_complex"] = df["station_complex"].astype(str).str.strip()

    if "borough" in df.columns:
        df["borough"] = df["borough"].astype(str).str.strip()
        df.loc[df["borough"].str.lower() == "nan", "borough"] = np.nan
        df.loc[df["borough"] == "", "borough"] = np.nan
    else:
        df["borough"] = np.nan

    df["net_entries"] = pd.to_numeric(
        df["net_entries"],
        errors="coerce"
    ).fillna(0)

    df = df.dropna(
        subset=["date", "station_complex_id", "station_complex"]
    ).copy()

    # Add day-of-week
    df["dow"] = df["date"].dt.dayofweek
    df["dow_label"] = df["dow"].map(DOW_MAP)

    # Rename for consistency with the rest of the notebook
    df = df.rename(columns={"station_complex": "complex_name"})

    # Final safety aggregation
    df = (
        df.groupby(
            [
                "station_complex_id",
                "complex_name",
                "borough",
                "date",
                "dow",
                "dow_label",
            ],
            as_index=False,
        )
        .agg(
            net_entries=("net_entries", "sum")
        )
    )

    # Save individual year file
    df.to_csv(cache, index=False)

    print(
        f"  {year}: saved {len(df):,} rows | "
        f"{df['station_complex_id'].nunique()} complexes | "
        f"{df['date'].nunique()} days"
    )
    print(f"  Saved file: {cache}")

    return df


# -------------------------------------------------------
# Run extraction for 2022, 2023, and 2024
# -------------------------------------------------------

station_complex_daily = {}

for yr in POST_COVID_YEARS:
    print(f"\nFetching post-COVID daily ridership for {yr}...")
    station_complex_daily[yr] = fetch_station_complex_daily_post(yr)

print("\nPost-COVID station-complex daily ridership extraction complete.")

for yr, df in station_complex_daily.items():
    print(
        f"  {yr}: {df['station_complex_id'].nunique()} complexes, "
        f"{df['date'].nunique()} days, {len(df):,} rows"
    )


# -------------------------------------------------------
# Save main modeling dataset: 2022 and 2024
# -------------------------------------------------------

main_model_daily = pd.concat(
    [
        station_complex_daily[yr].assign(period_year=yr)
        for yr in MAIN_MODEL_YEARS
    ],
    ignore_index=True,
)

main_model_out = OUT_DIR / "station_complex_daily_main_2022_2024.csv"

main_model_daily.to_csv(main_model_out, index=False)

print("\nMain modeling dataset saved:")
print(f"  File: {main_model_out}")
print(f"  Rows: {len(main_model_daily):,}")
print(f"  Years: {sorted(main_model_daily['period_year'].unique())}")
print(f"  Complexes: {main_model_daily['station_complex_id'].nunique()}")


# -------------------------------------------------------
# Save robustness dataset: 2023
# -------------------------------------------------------

robustness_daily_2023 = station_complex_daily[2023].assign(period_year=2023)

robustness_out = OUT_DIR / "station_complex_daily_robustness_2023.csv"

robustness_daily_2023.to_csv(robustness_out, index=False)

print("\nRobustness dataset saved:")
print(f"  File: {robustness_out}")
print(f"  Rows: {len(robustness_daily_2023):,}")
print(f"  Years: {sorted(robustness_daily_2023['period_year'].unique())}")
print(f"  Complexes: {robustness_daily_2023['station_complex_id'].nunique()}")


# -------------------------------------------------------
# Verify files were saved
# -------------------------------------------------------

print("\n================ POST-COVID FILE SAVE CHECK ================\n")

files_to_check = [
    OUT_DIR / "station_complex_daily_2022.csv",
    OUT_DIR / "station_complex_daily_2023.csv",
    OUT_DIR / "station_complex_daily_2024.csv",
    OUT_DIR / "station_complex_daily_main_2022_2024.csv",
    OUT_DIR / "station_complex_daily_robustness_2023.csv",
]

for path in files_to_check:
    print(f"{path.name}: exists={path.exists()} | {path}")


# -------------------------------------------------------
# Quick QA checks
# -------------------------------------------------------

print("\n================ POST-COVID QA CHECKS ================\n")

for yr, df in station_complex_daily.items():
    print(f"\nYear: {yr}")
    print("-" * 50)
    print("Rows:", len(df))
    print("Unique station complexes:", df["station_complex_id"].nunique())
    print("Unique dates:", df["date"].nunique())
    print("Date range:", df["date"].min(), "to", df["date"].max())
    print("Total net entries:", df["net_entries"].sum())

    print("\nMissing values:")
    print(
        df[
            [
                "station_complex_id",
                "complex_name",
                "borough",
                "date",
                "net_entries",
            ]
        ].isna().sum()
    )

print("\nAverage daily net entries by year:")

combined_postcovid_check = pd.concat(
    [
        station_complex_daily[yr].assign(period_year=yr)
        for yr in POST_COVID_YEARS
    ],
    ignore_index=True,
)

print(
    combined_postcovid_check
    .groupby("period_year")["net_entries"]
    .mean()
)


Fetching post-COVID daily ridership for 2022...
  2022: loaded station-complex daily from cache

Fetching post-COVID daily ridership for 2023...
  2023: loaded station-complex daily from cache

Fetching post-COVID daily ridership for 2024...
  2024: loaded station-complex daily from cache

Post-COVID station-complex daily ridership extraction complete.
  2022: 426 complexes, 365 days, 155,248 rows
  2023: 426 complexes, 365 days, 155,238 rows
  2024: 426 complexes, 366 days, 155,634 rows

Main modeling dataset saved:
  File: /content/drive/MyDrive/dow_ridership_paper/outputs/station_complex_daily_main_2022_2024.csv
  Rows: 310,882
  Years: [np.int64(2022), np.int64(2024)]
  Complexes: 426

Robustness dataset saved:
  File: /content/drive/MyDrive/dow_ridership_paper/outputs/station_complex_daily_robustness_2023.csv
  Rows: 155,238
  Years: [np.int64(2023)]
  Complexes: 426

================ POST-COVID FILE SAVE CHECK ================

station_complex_daily_2022.csv: exists=True | /cont

### 2.2 Pre-COVID Station Name → Complex ID Join

**Purpose:** The pre-COVID turnstile data identifies stations by free-text `STATION` strings (e.g., `"GRAND CENTRAL-42 ST"`) while post-COVID data uses numeric `station_complex_id`. This section bridges the gap so all three study periods share a common station-complex universe.

**Methodology:**
1. Build an alias table from the station complex reference file using `display_name`, `stop_name`, and `constituent_station_names` columns.
2. Normalize all names (lowercase, strip punctuation, collapse whitespace).
3. Exact match on normalized name → covers ~85% of stations.
4. difflib fuzzy match (cutoff 0.75) for remaining unmatched names.
5. Aggregate matched pre-COVID data to station-complex × DOW level.

**Note:** Post-COVID 2022/2023/2024 data already has `station_complex_id` and does not need this join.


In [11]:
# ==============================================================
# Section 2.2 — Pre-COVID Station Name → Complex ID Join
#
# Purpose:
#   Converts the 2017–2019 pre-COVID station-level baseline into
#   station-complex-level baseline data.
#
# Input:
#   station_daily_precovid_avg.csv
#
# Required existing object:
#   station_complex_df
#
# Outputs:
#   station_daily_precovid_with_complex.csv
#   station_complex_precovid_avg.csv
#
# Note:
#   Post-COVID 2022, 2023, and 2024 data already comes from the
#   MTA hourly ridership dataset at station-complex level, so it
#   does not need this station-name join.
# ==============================================================

import re
import difflib
import pandas as pd
import numpy as np


# -------------------------------------------------------
# Helper: normalize station names
# -------------------------------------------------------

def normalize_name(name: str) -> str:
    """
    Lowercase, remove punctuation, collapse whitespace.
    """
    name = str(name).lower()
    name = re.sub(r"[^a-z0-9 ]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name


# -------------------------------------------------------
# Build flexible crosswalk
# -------------------------------------------------------

def build_crosswalk(complex_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build a flexible station-name → complex_id crosswalk from the
    station complex file.

    Compatible with columns:
      complex_id
      stop_name
      display_name
      constituent_station_names
      latitude
      longitude

    The matching keys are built from:
      1. display_name
      2. stop_name
      3. constituent_station_names
    """

    df = complex_df.copy()
    df.columns = df.columns.str.strip().str.lower()

    required = {"complex_id", "display_name", "latitude", "longitude"}
    missing = required - set(df.columns)

    if missing:
        raise RuntimeError(
            f"Missing required crosswalk columns: {missing}\n"
            f"Columns found: {list(df.columns)}"
        )

    alias_rows = []

    for _, row in df.iterrows():
        complex_id = row.get("complex_id")
        display_name = row.get("display_name")
        stop_name = row.get("stop_name")
        constituent_names = row.get("constituent_station_names")
        lat = row.get("latitude")
        lon = row.get("longitude")

        aliases = []

        # Official display/complex name
        if pd.notna(display_name):
            aliases.append(str(display_name))

        # Stop-level name variant
        if "stop_name" in df.columns and pd.notna(stop_name):
            aliases.append(str(stop_name))

        # Additional names inside the complex
        if "constituent_station_names" in df.columns and pd.notna(constituent_names):
            parts = re.split(r"[;,/]", str(constituent_names))
            aliases.extend([p.strip() for p in parts if p.strip()])

        # Remove blanks
        aliases = [a.strip() for a in aliases if str(a).strip()]

        for alias in aliases:
            alias_rows.append(
                {
                    "station_complex_id": complex_id,
                    "complex_name": display_name,
                    "complex_lat": lat,
                    "complex_lon": lon,
                    "alias_name": alias,
                    "key": normalize_name(alias),
                }
            )

    xwalk = pd.DataFrame(alias_rows)

    if xwalk.empty:
        raise RuntimeError("Crosswalk alias table is empty. Check station_complex_df.")

    xwalk["station_complex_id"] = pd.to_numeric(
        xwalk["station_complex_id"],
        errors="coerce"
    )

    xwalk["complex_lat"] = pd.to_numeric(
        xwalk["complex_lat"],
        errors="coerce"
    )

    xwalk["complex_lon"] = pd.to_numeric(
        xwalk["complex_lon"],
        errors="coerce"
    )

    xwalk["complex_name"] = xwalk["complex_name"].astype(str).str.strip()
    xwalk["alias_name"] = xwalk["alias_name"].astype(str).str.strip()

    xwalk = xwalk.dropna(subset=["station_complex_id"])
    xwalk = xwalk[xwalk["key"].notna() & (xwalk["key"] != "")].copy()

    # Keep one row per normalized alias key
    xwalk = xwalk.drop_duplicates(subset=["key"])

    print(f"Crosswalk built: {len(xwalk):,} unique station-name keys")
    print(f"Complexes in crosswalk: {xwalk['station_complex_id'].nunique()}")

    return xwalk


# -------------------------------------------------------
# Join station-level pre-COVID baseline to complex IDs
# -------------------------------------------------------

def join_complex(
    daily_df: pd.DataFrame,
    xwalk: pd.DataFrame,
    fuzzy_cutoff: float = 0.75
) -> pd.DataFrame:
    """
    Join station-level pre-COVID baseline to station-complex crosswalk.
    Uses exact normalized-name matching first, then fuzzy matching.
    """

    daily_df = daily_df.copy()

    if "station_name" not in daily_df.columns:
        raise RuntimeError(
            "Input dataframe must contain station_name.\n"
            f"Columns found: {list(daily_df.columns)}"
        )

    daily_df["key"] = daily_df["station_name"].apply(normalize_name)

    exact_map = (
        xwalk
        .set_index("key")[["station_complex_id", "complex_name", "complex_lat", "complex_lon"]]
        .to_dict("index")
    )

    unmatched_keys = set(daily_df["key"].unique()) - set(exact_map.keys())
    all_xwalk_keys = list(exact_map.keys())

    fuzzy_map = {}

    for ukey in unmatched_keys:
        matches = difflib.get_close_matches(
            ukey,
            all_xwalk_keys,
            n=1,
            cutoff=fuzzy_cutoff
        )

        if matches:
            fuzzy_map[ukey] = exact_map[matches[0]]

    combined_map = {**exact_map, **fuzzy_map}

    still_unmatched = set(daily_df["key"].unique()) - set(combined_map.keys())

    if still_unmatched:
        print(f"\nWarning: {len(still_unmatched)} station names unmatched.")
        print("First 25 unmatched station keys:")
        for s in sorted(still_unmatched)[:25]:
            print(f"  {s}")

    daily_df["station_complex_id"] = daily_df["key"].map(
        lambda k: combined_map.get(k, {}).get("station_complex_id")
    )

    daily_df["complex_name"] = daily_df["key"].map(
        lambda k: combined_map.get(k, {}).get("complex_name")
    )

    daily_df["complex_lat"] = daily_df["key"].map(
        lambda k: combined_map.get(k, {}).get("complex_lat")
    )

    daily_df["complex_lon"] = daily_df["key"].map(
        lambda k: combined_map.get(k, {}).get("complex_lon")
    )

    before = len(daily_df)

    daily_df = daily_df.dropna(subset=["station_complex_id"]).copy()

    after = len(daily_df)

    print("\nJoin results:")
    print(f"  Rows before join drop: {before:,}")
    print(f"  Rows after join drop:  {after:,}")
    print(f"  Rows dropped:          {before - after:,}")
    print(f"  Complexes retained:    {daily_df['station_complex_id'].nunique()}")

    daily_df = daily_df.drop(columns=["key"])

    return daily_df


# -------------------------------------------------------
# Build crosswalk
# -------------------------------------------------------

xwalk_df = build_crosswalk(station_complex_df)


# -------------------------------------------------------
# Load pre-COVID station-level baseline
# -------------------------------------------------------

precovid_avg_path = OUT_DIR / "station_daily_precovid_avg.csv"

if not precovid_avg_path.exists():
    raise FileNotFoundError(
        f"Missing file: {precovid_avg_path}\n"
        "Run the pre-COVID 2017–2019 baseline creation section first."
    )

precovid_avg = pd.read_csv(precovid_avg_path)

print("\nPre-COVID station-level baseline loaded:")
print(f"  File: {precovid_avg_path}")
print(f"  Rows: {len(precovid_avg):,}")
print(f"  Station names: {precovid_avg['station_name'].nunique()}")


# -------------------------------------------------------
# Join pre-COVID baseline to station complex
# -------------------------------------------------------

precovid_with_complex = join_complex(
    precovid_avg,
    xwalk_df,
    fuzzy_cutoff=0.75
)

precovid_with_complex_out = OUT_DIR / "station_daily_precovid_with_complex.csv"

precovid_with_complex.to_csv(
    precovid_with_complex_out,
    index=False
)

print("\nPre-COVID station-level baseline with complex IDs saved:")
print(f"  File: {precovid_with_complex_out}")
print(f"  Rows: {len(precovid_with_complex):,}")
print(f"  Complexes: {precovid_with_complex['station_complex_id'].nunique()}")


# -------------------------------------------------------
# Aggregate pre-COVID baseline to station-complex × DOW
# -------------------------------------------------------

required_precovid_cols = [
    "mean_entries_2017",
    "mean_entries_2018",
    "mean_entries_2019",
    "mean_entries_precovid",
]

missing_precovid_cols = [
    c for c in required_precovid_cols
    if c not in precovid_with_complex.columns
]

if missing_precovid_cols:
    raise RuntimeError(
        f"Missing expected pre-COVID mean-entry columns: {missing_precovid_cols}\n"
        f"Columns found: {list(precovid_with_complex.columns)}"
    )

required_group_cols = [
    "station_complex_id",
    "complex_name",
    "complex_lat",
    "complex_lon",
    "dow",
    "dow_label",
]

missing_group_cols = [
    c for c in required_group_cols
    if c not in precovid_with_complex.columns
]

if missing_group_cols:
    raise RuntimeError(
        f"Missing expected grouping columns: {missing_group_cols}\n"
        f"Columns found: {list(precovid_with_complex.columns)}"
    )

precovid_complex_avg = (
    precovid_with_complex
    .groupby(
        [
            "station_complex_id",
            "complex_name",
            "complex_lat",
            "complex_lon",
            "dow",
            "dow_label",
        ],
        as_index=False
    )
    .agg(
        mean_entries_2017=("mean_entries_2017", "sum"),
        mean_entries_2018=("mean_entries_2018", "sum"),
        mean_entries_2019=("mean_entries_2019", "sum"),
        mean_entries_precovid=("mean_entries_precovid", "sum"),
        n_station_names=("station_name", "nunique"),
    )
)

precovid_complex_avg_out = OUT_DIR / "station_complex_precovid_avg.csv"

precovid_complex_avg.to_csv(
    precovid_complex_avg_out,
    index=False
)

print("\nPre-COVID baseline aggregated to station-complex level:")
print(f"  File: {precovid_complex_avg_out}")
print(f"  Rows: {len(precovid_complex_avg):,}")
print(f"  Complexes: {precovid_complex_avg['station_complex_id'].nunique()}")
print(f"  DOW values: {sorted(precovid_complex_avg['dow_label'].dropna().unique())}")


# -------------------------------------------------------
# QA checks
# -------------------------------------------------------

print("\n================ PRE-COVID COMPLEX BASELINE QA ================\n")

print("Rows by DOW:")
print(
    precovid_complex_avg
    .groupby(["dow", "dow_label"])
    .size()
    .reset_index(name="rows")
    .sort_values("dow")
)

print("\nAverage pre-COVID entries by DOW:")
print(
    precovid_complex_avg
    .groupby(["dow", "dow_label"])["mean_entries_precovid"]
    .mean()
    .reset_index()
    .sort_values("dow")
)

print("\nTop 10 complexes by pre-COVID average entries:")
display(
    precovid_complex_avg
    .groupby(["station_complex_id", "complex_name"], as_index=False)
    ["mean_entries_precovid"]
    .mean()
    .sort_values("mean_entries_precovid", ascending=False)
    .head(10)
)

Crosswalk built: 840 unique station-name keys
Complexes in crosswalk: 445

Pre-COVID station-level baseline loaded:
  File: /content/drive/MyDrive/dow_ridership_paper/outputs/station_daily_precovid_avg.csv
  Rows: 2,662
  Station names: 381

First 25 unmatched station keys:
  116 st columbia
  14th street
  42 st port auth
  47 50 sts rock
  81 st museum
  86 st 2 ave
  96 st 2 ave
  9th street
  atl av barclay
  bushwick av
  central pk n110
  christopher st
  city bus
  eastn pkwy musm
  exchange place
  flatbush av b c
  grove street
  harrison
  howard bch jfk
  hoyt scher
  jfk jamaica ct1
  journal square
  kew gardens
  lackawanna
  newark bm bw

Join results:
  Rows before join drop: 2,662
  Rows after join drop:  2,380
  Rows dropped:          282
  Complexes retained:    319

Pre-COVID station-level baseline with complex IDs saved:
  File: /content/drive/MyDrive/dow_ridership_paper/outputs/station_daily_precovid_with_complex.csv
  Rows: 2,380
  Complexes: 319

Pre-COVID basel

,station_complex_id,complex_name,mean_entries_precovid
177,318.00,"34 St-Penn Station (1,2,3)","153,442.17"
297,610.00,"Grand Central-42 St (4,5,6,7,S)","130,699.45"
294,607.00,"34 St-Herald Sq (B,D,F,M,N,Q,R,W)","105,947.41"
178,320.00,23 St (1),"103,970.80"
289,602.00,"14 St-Union Sq (4,5,6,L,N,Q,R,W)","97,367.47"
298,611.00,"Times Sq-42 St/Port Authority Bus Terminal (1,...","93,004.57"
314,628.00,"Fulton St (2,3,4,5,A,C,J,Z)","87,463.15"
284,476.00,86 St (Q),"79,034.74"
308,622.00,"Brooklyn Bridge-City Hall/Chambers St (4,5,6,J,Z)","72,727.86"
261,439.00,"125 St (2,3)","70,943.52"


### 2.3 7-Day Ridership Vector Construction

**The core analytical unit** is a 7-dimensional normalized ridership vector per station complex per study period.

**Normalization rationale:** Dividing each station's daily entries by its weekly total converts absolute ridership volumes into *demand shape shares*. This is critical because:
- A large hub (500k weekly entries) and a small neighbourhood station (10k weekly entries) can share the same demand regime *shape* — both show the same Tue/Thu peak and Mon/Fri trough.
- Without normalization, K-means would cluster by volume rather than temporal pattern.
- Normalization also makes the framework transferable to other cities or data-scarce environments (relevant for the Africa research agenda).

**Low-volume flag:** Stations with average weekday entries < 50 are flagged but retained in the dataset. They are excluded from K-means clustering (noisy profiles) but retained in the transition analysis.


In [12]:
# ==============================================================
# Section 2.3 — 7-Day Ridership Vector Construction
#
# Purpose:
#   Converts station-complex ridership data to normalized 7-day
#   ridership vectors for K-means clustering.
#
# Inputs:
#   Pre-COVID:
#     station_complex_precovid_avg.csv
#
#   Post-COVID:
#     station_complex_daily_2022.csv
#     station_complex_daily_2023.csv
#     station_complex_daily_2024.csv
#
# Outputs:
#   dow_vectors_precovid.csv
#   dow_raw_precovid.csv
#   dow_vectors_2022.csv
#   dow_raw_2022.csv
#   dow_vectors_2023.csv
#   dow_raw_2023.csv
#   dow_vectors_2024.csv
#   dow_raw_2024.csv
# ==============================================================

import pandas as pd
import numpy as np


# -------------------------------------------------------
# DOW label setup
# -------------------------------------------------------

DOW_MAP_FULL = {
    0: "Mon",
    1: "Tue",
    2: "Wed",
    3: "Thu",
    4: "Fri",
    5: "Sat",
    6: "Sun",
}

DOW_LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]


# -------------------------------------------------------
# Optional: delete old/bad post-COVID DOW cache files
# -------------------------------------------------------
# This is important because the earlier version created empty files
# for 2022, 2023, and 2024.

DELETE_BAD_POSTCOVID_DOW_CACHE = True

if DELETE_BAD_POSTCOVID_DOW_CACHE:
    for yr in [2022, 2023, 2024]:
        for prefix in ["dow_raw", "dow_vectors"]:
            path = OUT_DIR / f"{prefix}_{yr}.csv"
            if path.exists():
                path.unlink()
                print(f"Deleted old post-COVID DOW cache: {path}")


# -------------------------------------------------------
# Build DOW vectors from post-COVID daily data
# -------------------------------------------------------

def build_dow_vectors_from_daily(
    daily_df: pd.DataFrame,
    label: str
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Build normalized and raw day-of-week ridership vectors from
    station-complex daily data.

    Expected input columns:
      station_complex_id
      complex_name
      dow
      net_entries

    Optional:
      borough
    """

    cache_norm = OUT_DIR / f"dow_vectors_{label}.csv"
    cache_raw = OUT_DIR / f"dow_raw_{label}.csv"

    if cache_norm.exists() and cache_raw.exists():
        print(f"  {label}: loaded DOW vectors from cache")
        return (
            pd.read_csv(cache_norm),
            pd.read_csv(cache_raw),
        )

    df = daily_df.copy()

    required_cols = {
        "station_complex_id",
        "complex_name",
        "dow",
        "net_entries",
    }

    missing = required_cols - set(df.columns)

    if missing:
        raise RuntimeError(
            f"{label}: missing required columns: {missing}\n"
            f"Columns found: {list(df.columns)}"
        )

    df["station_complex_id"] = pd.to_numeric(
        df["station_complex_id"],
        errors="coerce"
    )

    df["complex_name"] = df["complex_name"].astype(str).str.strip()

    df["dow"] = pd.to_numeric(
        df["dow"],
        errors="coerce"
    )

    df["net_entries"] = pd.to_numeric(
        df["net_entries"],
        errors="coerce"
    ).fillna(0)

    if "borough" not in df.columns:
        df["borough"] = "Unknown"

    df["borough"] = df["borough"].fillna("Unknown").astype(str).str.strip()

    df = df.dropna(
        subset=["station_complex_id", "complex_name", "dow"]
    ).copy()

    # Keep only valid DOW values
    df = df[df["dow"].isin(range(7))].copy()
    df["dow"] = df["dow"].astype(int)

    # -------------------------------------------------------
    # IMPORTANT:
    # Do not group by complex_lat/complex_lon here.
    # The post-COVID files may not have them, and grouping on
    # all-NaN coordinate columns makes pandas drop every row.
    # -------------------------------------------------------

    group_cols = [
        "station_complex_id",
        "complex_name",
        "borough",
        "dow",
    ]

    agg = (
        df
        .groupby(group_cols, as_index=False)
        .agg(mean_entries=("net_entries", "mean"))
    )

    raw = (
        agg
        .pivot_table(
            index=[
                "station_complex_id",
                "complex_name",
                "borough",
            ],
            columns="dow",
            values="mean_entries",
        )
        .reset_index()
    )

    raw = raw.rename(columns=DOW_MAP_FULL)

    for col in DOW_LABELS:
        if col not in raw.columns:
            raw[col] = np.nan

    raw = raw.dropna(subset=DOW_LABELS).copy()

    norm = raw.copy()

    row_sums = norm[DOW_LABELS].sum(axis=1)

    nonzero = row_sums > 0
    raw = raw.loc[nonzero].copy()
    norm = norm.loc[nonzero].copy()
    row_sums = row_sums.loc[nonzero]

    norm[DOW_LABELS] = norm[DOW_LABELS].div(row_sums, axis=0)

    weekday_cols = ["Mon", "Tue", "Wed", "Thu", "Fri"]

    raw["avg_weekday_entries"] = raw[weekday_cols].mean(axis=1)
    raw["low_volume_flag"] = raw["avg_weekday_entries"] < 50

    norm["avg_weekday_entries"] = raw["avg_weekday_entries"].values
    norm["low_volume_flag"] = raw["low_volume_flag"].values

    raw.to_csv(cache_raw, index=False)
    norm.to_csv(cache_norm, index=False)

    print(
        f"  {label}: {len(norm):,} complexes | "
        f"{int(norm['low_volume_flag'].sum())} low-volume flagged"
    )

    return norm, raw


# -------------------------------------------------------
# Build DOW vectors from pre-COVID complex average
# -------------------------------------------------------

def build_dow_vectors_from_precovid_avg(
    precovid_df: pd.DataFrame,
    label: str = "precovid"
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Build normalized and raw DOW vectors from the pre-COVID station-complex
    average file.

    Expected input columns:
      station_complex_id
      complex_name
      complex_lat
      complex_lon
      dow
      mean_entries_precovid
    """

    cache_norm = OUT_DIR / f"dow_vectors_{label}.csv"
    cache_raw = OUT_DIR / f"dow_raw_{label}.csv"

    if cache_norm.exists() and cache_raw.exists():
        print(f"  {label}: loaded DOW vectors from cache")
        return (
            pd.read_csv(cache_norm),
            pd.read_csv(cache_raw),
        )

    df = precovid_df.copy()

    required_cols = {
        "station_complex_id",
        "complex_name",
        "complex_lat",
        "complex_lon",
        "dow",
        "mean_entries_precovid",
    }

    missing = required_cols - set(df.columns)

    if missing:
        raise RuntimeError(
            f"{label}: missing required columns: {missing}\n"
            f"Columns found: {list(df.columns)}"
        )

    df["station_complex_id"] = pd.to_numeric(
        df["station_complex_id"],
        errors="coerce"
    )

    df["complex_name"] = df["complex_name"].astype(str).str.strip()

    df["complex_lat"] = pd.to_numeric(
        df["complex_lat"],
        errors="coerce"
    )

    df["complex_lon"] = pd.to_numeric(
        df["complex_lon"],
        errors="coerce"
    )

    df["dow"] = pd.to_numeric(
        df["dow"],
        errors="coerce"
    )

    df["mean_entries_precovid"] = pd.to_numeric(
        df["mean_entries_precovid"],
        errors="coerce"
    ).fillna(0)

    df = df.dropna(
        subset=[
            "station_complex_id",
            "complex_name",
            "complex_lat",
            "complex_lon",
            "dow",
        ]
    ).copy()

    df = df[df["dow"].isin(range(7))].copy()
    df["dow"] = df["dow"].astype(int)

    raw = (
        df
        .pivot_table(
            index=[
                "station_complex_id",
                "complex_name",
                "complex_lat",
                "complex_lon",
            ],
            columns="dow",
            values="mean_entries_precovid",
        )
        .reset_index()
    )

    raw = raw.rename(columns=DOW_MAP_FULL)

    for col in DOW_LABELS:
        if col not in raw.columns:
            raw[col] = np.nan

    raw = raw.dropna(subset=DOW_LABELS).copy()

    norm = raw.copy()

    row_sums = norm[DOW_LABELS].sum(axis=1)

    nonzero = row_sums > 0
    raw = raw.loc[nonzero].copy()
    norm = norm.loc[nonzero].copy()
    row_sums = row_sums.loc[nonzero]

    norm[DOW_LABELS] = norm[DOW_LABELS].div(row_sums, axis=0)

    weekday_cols = ["Mon", "Tue", "Wed", "Thu", "Fri"]

    raw["avg_weekday_entries"] = raw[weekday_cols].mean(axis=1)
    raw["low_volume_flag"] = raw["avg_weekday_entries"] < 50

    norm["avg_weekday_entries"] = raw["avg_weekday_entries"].values
    norm["low_volume_flag"] = raw["low_volume_flag"].values

    raw.to_csv(cache_raw, index=False)
    norm.to_csv(cache_norm, index=False)

    print(
        f"  {label}: {len(norm):,} complexes | "
        f"{int(norm['low_volume_flag'].sum())} low-volume flagged"
    )

    return norm, raw


# -------------------------------------------------------
# Load required inputs if needed
# -------------------------------------------------------

precovid_path = OUT_DIR / "station_complex_precovid_avg.csv"

if "precovid_complex_avg" not in globals():
    if not precovid_path.exists():
        raise FileNotFoundError(
            f"Missing file: {precovid_path}\n"
            "Run Section 2.2 first to create station_complex_precovid_avg.csv."
        )

    precovid_complex_avg = pd.read_csv(precovid_path)

if "station_complex_daily" not in globals():
    station_complex_daily = {}

for yr in [2022, 2023, 2024]:
    if yr not in station_complex_daily:
        path = OUT_DIR / f"station_complex_daily_{yr}.csv"

        if not path.exists():
            raise FileNotFoundError(
                f"Missing file: {path}\n"
                "Run Section 2.1 first to create post-COVID station-complex daily files."
            )

        station_complex_daily[yr] = pd.read_csv(path, parse_dates=["date"])


# -------------------------------------------------------
# Run DOW vector construction
# -------------------------------------------------------

dow_vectors = {}
dow_raw = {}

print("Building DOW vectors for pre-COVID baseline...")
dow_vectors["precovid"], dow_raw["precovid"] = build_dow_vectors_from_precovid_avg(
    precovid_complex_avg,
    label="precovid"
)

for yr in [2022, 2023, 2024]:
    print(f"Building DOW vectors for {yr}...")
    dow_vectors[yr], dow_raw[yr] = build_dow_vectors_from_daily(
        station_complex_daily[yr],
        label=str(yr)
    )

print("\nDOW vector construction complete.")

print("\nDOW raw shapes:")
for k, df in dow_raw.items():
    print(f"  {k}: {df.shape}")

print("\nDOW vector shapes:")
for k, df in dow_vectors.items():
    print(f"  {k}: {df.shape}")


# -------------------------------------------------------
# QA checks
# -------------------------------------------------------

print("\n================ DOW VECTOR QA CHECKS ================\n")

for k, df in dow_raw.items():
    missing_dow = [c for c in DOW_LABELS if c not in df.columns]
    print(f"\nPeriod: {k}")
    print("-" * 40)
    print("Rows:", len(df))
    print("Missing DOW columns:", missing_dow)

    if len(df) > 0:
        print("Mean weekday entries:")
        print(df[["Mon", "Tue", "Wed", "Thu", "Fri"]].mean().round(2))
        print("Mean weekend entries:")
        print(df[["Sat", "Sun"]].mean().round(2))

print("\nPreview: pre-COVID")
display(dow_raw["precovid"].head(3))

print("\nPreview: 2024")
display(dow_raw[2024].head(3))

Deleted old post-COVID DOW cache: /content/drive/MyDrive/dow_ridership_paper/outputs/dow_raw_2022.csv
Deleted old post-COVID DOW cache: /content/drive/MyDrive/dow_ridership_paper/outputs/dow_vectors_2022.csv
Deleted old post-COVID DOW cache: /content/drive/MyDrive/dow_ridership_paper/outputs/dow_raw_2023.csv
Deleted old post-COVID DOW cache: /content/drive/MyDrive/dow_ridership_paper/outputs/dow_vectors_2023.csv
Deleted old post-COVID DOW cache: /content/drive/MyDrive/dow_ridership_paper/outputs/dow_raw_2024.csv
Deleted old post-COVID DOW cache: /content/drive/MyDrive/dow_ridership_paper/outputs/dow_vectors_2024.csv
Building DOW vectors for pre-COVID baseline...
  precovid: loaded DOW vectors from cache
Building DOW vectors for 2022...
  2022: 426 complexes | 0 low-volume flagged
Building DOW vectors for 2023...
  2023: 426 complexes | 0 low-volume flagged
Building DOW vectors for 2024...
  2024: 426 complexes | 0 low-volume flagged

DOW vector construction complete.

DOW raw shapes:
 

,station_complex_id,complex_name,complex_lat,complex_lon,Mon,Tue,Wed,Thu,Fri,Sat,Sun,avg_weekday_entries,low_volume_flag
0,1.00,"Astoria-Ditmars Blvd (N,W)",40.78,-73.91,"15,914.11","17,173.65","17,506.63","17,547.30","16,878.52","14,451.71","5,914.22","17,004.04",False
1,2.00,"Astoria Blvd (N,W)",40.77,-73.92,"11,383.81","13,129.56","12,738.90","12,734.85","12,554.63","9,992.70","4,638.11","12,508.35",False
2,3.00,"30 Av (N,W)",40.77,-73.92,"14,857.85","16,240.34","16,335.78","16,543.54","15,678.84","11,283.69","4,924.64","15,931.27",False



Preview: 2024


dow,station_complex_id,complex_name,borough,Mon,Tue,Wed,Thu,Fri,Sat,Sun,avg_weekday_entries,low_volume_flag
0,1.00,"Astoria-Ditmars Blvd (N,W)",Queens,"9,619.51","11,051.34","11,225.00","11,034.44","10,376.27","6,615.58","4,967.23","10,661.31",False
1,2.00,"Astoria Blvd (N,W)",Queens,"6,998.40","8,032.98","8,118.65","8,174.02","7,691.65","4,763.71","3,690.79","7,803.14",False
2,3.00,"30 Av (N,W)",Queens,"8,824.77","10,089.75","10,312.90","10,151.60","9,510.63","6,048.73","4,515.79","9,777.93",False


---
## SECTION 3: Exploratory Data Analysis

Three sub-tasks establish the empirical foundation before clustering:

- **3.1** produces **Table 1** — system-wide mean daily entries by DOW for all study periods, with % change vs. the pre-COVID average. This is the paper's primary descriptive table.

- **3.2** produces **Figure 2** — overlaid 7-day profiles comparing the pre-COVID average to 2024 for two contrasting station groups (high office-density Manhattan vs. outer-borough residential). This is the paper's opening visual argument.

- **3.3** quantifies the **Mon/Fri gap** — the headline finding that Mon/Fri recovery lags Tue–Thu recovery at office-type stations, providing the empirical basis for the paper's central claim.


### 3.1 Summary Statistics: Day-of-Week × Period (Table 1)

**Outputs:** `table1_summary_stats.csv` (main: pre-COVID avg, 2022, 2024) and `table1_summary_stats_with_2023_robustness.csv` (includes 2023 transitional year).


In [13]:
# ==============================================================
# Section 3.1 — Summary Statistics: Day-of-Week × Period
#
# Purpose:
#   Computes system-wide mean daily entries by day of week
#   for each study period, and calculates % change from the
#   2017–2019 pre-COVID average baseline.
#
# Main periods:
#   Pre-COVID Avg = 2017–2019 average
#   2022 = Early Recovery
#   2024 = Stabilized New Normal
#
# Robustness:
#   2023 = Transitional Recovery
#
# Output:
#   table1_summary_stats.csv
#   table1_summary_stats_with_2023_robustness.csv
# ==============================================================

import pandas as pd
import numpy as np

DOW_LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

PERIOD_LABELS = {
    "precovid": "Pre-COVID Avg. (2017–2019)",
    2022: "Early Recovery (2022)",
    2023: "Transitional Recovery (2023)",
    2024: "Stabilized New Normal (2024)",
}

MAIN_PERIODS = ["precovid", 2022, 2024]
ALL_PERIODS_WITH_ROBUSTNESS = ["precovid", 2022, 2023, 2024]

# -------------------------------------------------------
# Validation checks
# -------------------------------------------------------

required_periods = set(ALL_PERIODS_WITH_ROBUSTNESS)
available_periods = set(dow_raw.keys())
missing_periods = required_periods - available_periods

if missing_periods:
    raise RuntimeError(
        f"dow_raw is missing these periods: {missing_periods}\n"
        f"Available dow_raw keys: {list(dow_raw.keys())}\n"
        "Run Section 2.3 first to build DOW vectors for pre-COVID, 2022, 2023, and 2024."
    )

for period in ALL_PERIODS_WITH_ROBUSTNESS:
    missing_cols = [c for c in DOW_LABELS if c not in dow_raw[period].columns]
    if missing_cols:
        raise RuntimeError(
            f"dow_raw[{period!r}] is missing DOW columns: {missing_cols}\n"
            f"Columns found: {list(dow_raw[period].columns)}"
        )


# -------------------------------------------------------
# Helper function to build summary table
# -------------------------------------------------------

def build_summary_stats(periods, output_name, title_suffix=""):
    """
    Build day-of-week summary statistics for selected periods.

    Uses dow_raw['precovid'] as the baseline for percentage change.
    """

    summary_rows = []

    for period in periods:
        df = dow_raw[period].copy()

        for day in DOW_LABELS:
            summary_rows.append(
                {
                    "period": period,
                    "period_label": PERIOD_LABELS[period],
                    "day_of_week": day,
                    "mean_entries": df[day].mean(),
                    "median_entries": df[day].median(),
                    "std_entries": df[day].std(),
                }
            )

    summary_df = pd.DataFrame(summary_rows)

    baseline = (
        summary_df[summary_df["period"] == "precovid"]
        .set_index("day_of_week")["mean_entries"]
    )

    summary_df["pct_change_vs_precovid_avg"] = summary_df.apply(
        lambda r: (
            (r["mean_entries"] - baseline[r["day_of_week"]])
            / baseline[r["day_of_week"]]
            * 100
        ),
        axis=1,
    )

    summary_df.loc[
        summary_df["period"] == "precovid",
        "pct_change_vs_precovid_avg"
    ] = 0.0

    table = summary_df.pivot_table(
        index="day_of_week",
        columns="period_label",
        values=["mean_entries", "pct_change_vs_precovid_avg"],
    ).round(1)

    table = table.reindex(DOW_LABELS)

    print(f"\nTable 1: System-Wide Mean Daily Entries by Day of Week{title_suffix}")
    print("Baseline: 2017–2019 Pre-COVID Average")
    print(table.to_string())

    out_path = OUT_DIR / output_name
    summary_df.to_csv(out_path, index=False)

    print(f"\nSaved: {out_path}")

    return summary_df, table


# -------------------------------------------------------
# Main Table 1: Pre-COVID Avg, 2022, 2024
# -------------------------------------------------------

summary_df, table1 = build_summary_stats(
    periods=MAIN_PERIODS,
    output_name="table1_summary_stats.csv",
    title_suffix=" — Main Analysis"
)


# -------------------------------------------------------
# Robustness version: includes 2023
# -------------------------------------------------------

summary_df_with_2023, table1_with_2023 = build_summary_stats(
    periods=ALL_PERIODS_WITH_ROBUSTNESS,
    output_name="table1_summary_stats_with_2023_robustness.csv",
    title_suffix=" — With 2023 Robustness"
)


Table 1: System-Wide Mean Daily Entries by Day of Week — Main Analysis
Baseline: 2017–2019 Pre-COVID Average
                      mean_entries                                                         pct_change_vs_precovid_avg                                                        
period_label Early Recovery (2022) Pre-COVID Avg. (2017–2019) Stabilized New Normal (2024)      Early Recovery (2022) Pre-COVID Avg. (2017–2019) Stabilized New Normal (2024)
day_of_week                                                                                                                                                                  
Mon                       6,720.80                  15,029.70                     7,998.10                     -55.30                       0.00                       -46.80
Tue                       7,638.50                  16,187.50                     9,029.30                     -52.80                       0.00                       -44.20
Wed                 

### 3.2 Figure 2: Overlaid 7-Day Profiles — Pre-COVID Average vs. 2024

**Methodology:** For two representative station groups (high office-density and outer-borough residential), compute the mean normalized DOW profile across matched stations for the pre-COVID average and 2024. Plot overlaid line charts. Monday and Friday are lightly shaded to highlight the hybrid-work-relevant shoulder days.

**Station matching:** Partial case-insensitive string matching against `complex_name`. The `show_name_matches()` helper prints what was matched before plotting — always review this output to confirm representative stations were found.


In [14]:
# ==============================================================
# Section 3.2 — Figure 2: Overlaid 7-Day Profiles
# Pre-COVID Average vs 2024
#
# Purpose:
#   Visualizes the structural shift in day-of-week ridership
#   profiles between the 2017-2019 pre-COVID average and 2024
#   for two contrasting station types:
#     (a) High office-density Manhattan stations
#     (b) Outer-borough residential stations
#
# Interpretation:
#   The figure highlights whether post-pandemic ridership is more
#   concentrated in the midweek and weaker on Fridays/weekends,
#   consistent with hybrid-work and changed non-work travel patterns.
#
# Station group design:
#   Every search term must match >=1 station in BOTH the pre-COVID
#   and 2024 datasets so group composition is symmetric across periods.
#
#   OFFICE_STATIONS:
#     "47-50" replaces "Rockefeller" -- the MTA renamed this station
#     between data vintages. "47-50" matches "47-50 Sts-Rockefeller Ctr"
#     in both periods. "Rockefeller" alone returns no match pre-COVID.
#
#     "Fulton St" matches ONE complex pre-COVID and TWO complexes in
#     2024 ("Fulton St (G)" and "Fulton St (A,C,J,Z,2,3,4,5)"). Both
#     2024 complexes are included -- they represent the same geographic
#     catchment as the single pre-COVID complex. get_profile() averages
#     across all matched rows, so this is handled correctly.
#
#   RESIDENTIAL_STATIONS:
#     "Flatbush Av" replaces "Flatbush" -- the more specific term
#     targets Flatbush Av-Brooklyn College and avoids spurious matches
#     with other Flatbush-named streets in the dataset.
#
# Output:
#   figure2_dow_profile_shift_precovid_vs_2024.png
# ==============================================================

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import numpy as np
import re


# -------------------------------------------------------
# Required DOW labels
# -------------------------------------------------------

DOW_LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]


# -------------------------------------------------------
# Representative station search terms
# Partial case-insensitive matches against complex_name.
#
# OFFICE_STATIONS:
#   "Grand Central"  -- Grand Central-42 St (both periods)
#   "Times Sq"       -- Times Sq-42 St (both periods)
#   "47-50"          -- 47-50 Sts-Rockefeller Ctr (both periods)
#                       NOTE: do NOT use "Rockefeller" -- absent pre-COVID
#   "Herald Sq"      -- 34 St-Herald Sq (both periods)
#   "Fulton St"      -- 1 complex pre-COVID, 2 complexes 2024 (all included)
#
# RESIDENTIAL_STATIONS:
#   "Pelham Bay"     -- Pelham Bay Park (both periods)
#   "Jamaica-179"    -- Jamaica-179 St (both periods)
#   "Bay Ridge"      -- Bay Ridge Av + Bay Ridge-95 St (both periods)
#   "Norwood"        -- Norwood Av + Norwood-205 St (both periods)
#   "Flatbush Av"    -- Flatbush Av-Brooklyn College (both periods)
#                       NOTE: do NOT use "Flatbush" -- too broad
# -------------------------------------------------------

OFFICE_STATIONS = [
    "Grand Central",   # Grand Central-42 St -- confirmed both periods
    "Times Sq",        # Times Sq-42 St -- confirmed both periods
    "47-50",           # 47-50 Sts-Rockefeller Ctr -- confirmed both periods
    "Herald Sq",       # 34 St-Herald Sq -- confirmed both periods
    "Fulton St",       # 1 complex pre-COVID, 2 complexes 2024 -- all included
]

RESIDENTIAL_STATIONS = [
    "Pelham Bay",      # Pelham Bay Park -- confirmed both periods
    "Jamaica-179",     # Jamaica-179 St -- confirmed both periods
    "Bay Ridge",       # Bay Ridge Av + Bay Ridge-95 St -- confirmed both periods
    "Norwood",         # Norwood Av + Norwood-205 St -- confirmed both periods
    "Flatbush Av",     # Flatbush Av-Brooklyn College -- confirmed both periods
]


# -------------------------------------------------------
# Fallback color palette
# -------------------------------------------------------

if "PALETTE" not in globals():
    PALETTE = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]


# -------------------------------------------------------
# Validate required inputs
# -------------------------------------------------------

required_periods = ["precovid", 2024]

missing_periods = [p for p in required_periods if p not in dow_raw]

if missing_periods:
    raise RuntimeError(
        "dow_raw is missing required periods: " + str(missing_periods) + "\n"
        + "Available keys: " + str(list(dow_raw.keys())) + "\n"
        + "Run Section 2.3 first."
    )

for period in required_periods:
    required_cols = ["station_complex_id", "complex_name"] + DOW_LABELS
    missing_cols = [c for c in required_cols if c not in dow_raw[period].columns]

    if missing_cols:
        raise RuntimeError(
            "dow_raw[" + repr(period) + "] is missing required columns: "
            + str(missing_cols) + "\n"
            + "Columns found: " + str(list(dow_raw[period].columns))
        )


# -------------------------------------------------------
# Helper: inspect available complex names
# -------------------------------------------------------

def show_name_matches(search_terms, df_key="precovid", max_matches=10):
    """
    Print possible complex-name matches for each search term.
    Useful for confirming that representative stations exist.
    """
    df = dow_raw[df_key].copy()
    print("\nChecking names in dow_raw[" + repr(df_key) + "]")

    for term in search_terms:
        pattern = re.escape(term)
        matches = (
            df.loc[
                df["complex_name"]
                .astype(str)
                .str.contains(pattern, case=False, na=False, regex=True),
                "complex_name"
            ]
            .drop_duplicates()
            .head(max_matches)
            .tolist()
        )
        print("\nSearch term: " + term)
        if matches:
            for m in matches:
                print("  " + m)
        else:
            print("  No matches found.")


# -------------------------------------------------------
# Helper: get matched station rows
# -------------------------------------------------------

def get_matched_station_rows(period, station_terms):
    """
    Returns rows from dow_raw[period] whose complex_name partially matches
    any of the search terms. Deduplicates on station_complex_id so a station
    matched by multiple terms is counted only once.
    """
    df = dow_raw[period].copy()
    matched_frames = []

    for term in station_terms:
        pattern = re.escape(term)
        sub = df.loc[
            df["complex_name"]
            .astype(str)
            .str.contains(pattern, case=False, na=False, regex=True),
            ["station_complex_id", "complex_name"] + DOW_LABELS
        ].copy()

        if not sub.empty:
            sub["matched_term"] = term
            matched_frames.append(sub)

    if not matched_frames:
        return pd.DataFrame(
            columns=["station_complex_id", "complex_name", "matched_term"] + DOW_LABELS
        )

    matched = (
        pd.concat(matched_frames, ignore_index=True)
        .drop_duplicates(subset=["station_complex_id", "complex_name"])
        .reset_index(drop=True)
    )
    return matched


# -------------------------------------------------------
# Helper: get normalized DOW profile
# -------------------------------------------------------

def get_profile(period, station_terms, group_label):
    """
    Get normalized mean DOW profile across a set of station complexes.
    Uses partial case-insensitive matching.
    Returns a Series of 7 normalized shares summing to 1.0.
    """
    matched = get_matched_station_rows(period, station_terms)

    if matched.empty:
        print("\nWarning: no matching stations for " + group_label
              + " in period " + str(period))
        print("Search terms: " + str(station_terms))
        return pd.Series(index=DOW_LABELS, dtype=float)

    print("\nMatched stations for " + group_label
          + ", period " + str(period) + ":")
    for name in matched["complex_name"].drop_duplicates().tolist():
        print("  " + name)

    profile = matched[DOW_LABELS].mean()
    total   = profile.sum()

    if total <= 0 or pd.isna(total):
        print("\nWarning: profile total is zero/NaN for "
              + group_label + ", period " + str(period))
        return pd.Series(index=DOW_LABELS, dtype=float)

    return profile / total


# -------------------------------------------------------
# Symmetry check -- verify all terms match in both periods
# before plotting. Raises RuntimeError if any term is absent
# from either period, preventing an asymmetric comparison.
# -------------------------------------------------------

print("\n================ FIGURE 2 SYMMETRY CHECK ================\n")
print("Every search term must match >=1 station in BOTH periods.")
print("MISSING in either period = asymmetric group = fix required.\n")

all_symmetric = True

for group_name, terms in [
    ("Office-Density",  OFFICE_STATIONS),
    ("Residential",     RESIDENTIAL_STATIONS),
]:
    print("--- " + group_name + " ---")
    for term in terms:
        pre_mask  = dow_raw["precovid"]["complex_name"].astype(str).str.contains(
            re.escape(term), case=False, na=False, regex=True)
        post_mask = dow_raw[2024]["complex_name"].astype(str).str.contains(
            re.escape(term), case=False, na=False, regex=True)

        pre_names  = dow_raw["precovid"].loc[pre_mask,  "complex_name"].unique().tolist()
        post_names = dow_raw[2024].loc[post_mask, "complex_name"].unique().tolist()

        ok = len(pre_names) > 0 and len(post_names) > 0
        if not ok:
            all_symmetric = False

        status = "OK" if ok else "MISSING"
        print("  [" + status + "] '" + term + "'")
        print("    precovid (" + str(len(pre_names)) + "): " + str(pre_names))
        print("    2024     (" + str(len(post_names)) + "): " + str(post_names))
    print()

if not all_symmetric:
    raise RuntimeError(
        "Asymmetric station terms detected -- one or more search terms "
        "matched in one period but not the other.\n"
        "Fix OFFICE_STATIONS or RESIDENTIAL_STATIONS above so every "
        "term appears in both the pre-COVID and 2024 datasets, then re-run."
    )

print("All terms symmetric. Proceeding to name match detail and plot.\n")


# -------------------------------------------------------
# Optional name checks (full detail)
# -------------------------------------------------------

print("================ FIGURE 2 NAME MATCH CHECKS ================\n")

show_name_matches(
    OFFICE_STATIONS + RESIDENTIAL_STATIONS,
    df_key="precovid"
)

show_name_matches(
    OFFICE_STATIONS + RESIDENTIAL_STATIONS,
    df_key=2024
)


# -------------------------------------------------------
# Build profiles
# -------------------------------------------------------

office_pre = get_profile(
    "precovid",
    OFFICE_STATIONS,
    group_label="high office-density stations"
)

office_2024 = get_profile(
    2024,
    OFFICE_STATIONS,
    group_label="high office-density stations"
)

res_pre = get_profile(
    "precovid",
    RESIDENTIAL_STATIONS,
    group_label="outer-borough residential stations"
)

res_2024 = get_profile(
    2024,
    RESIDENTIAL_STATIONS,
    group_label="outer-borough residential stations"
)


# -------------------------------------------------------
# Build figure
# -------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)

panel_specs = [
    {
        "ax":    axes[0],
        "p_pre": office_pre,
        "p2024": office_2024,
        "title": "(a) High Office-Density Stations",
    },
    {
        "ax":    axes[1],
        "p_pre": res_pre,
        "p2024": res_2024,
        "title": "(b) Outer-Borough Residential Stations",
    },
]

for spec in panel_specs:
    ax = spec["ax"]
    x  = range(len(DOW_LABELS))

    ax.plot(
        x,
        spec["p_pre"],
        marker="o",
        linewidth=2.5,
        color=PALETTE[0],
        label="Pre-COVID Avg. (2017-2019)",
    )

    ax.plot(
        x,
        spec["p2024"],
        marker="s",
        linewidth=2.5,
        color=PALETTE[1],
        linestyle="--",
        label="2024 (Stabilized New Normal)",
    )

    ax.set_xticks(list(x))
    ax.set_xticklabels(DOW_LABELS)

    ax.set_title(
        spec["title"],
        fontsize=12,
        fontweight="bold",
        pad=10,
    )

    ax.set_ylabel("Share of Weekly Ridership", fontsize=10)

    ax.yaxis.set_major_formatter(
        mticker.PercentFormatter(xmax=1, decimals=0)
    )

    # Lightly shade Monday and Friday to highlight hybrid-work shoulder days
    ax.axvspan(-0.15, 0.5, alpha=0.05, color="gray")
    ax.axvspan(3.5, 4.5, alpha=0.05, color="gray")

    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)


# -------------------------------------------------------
# Title and figure note
# -------------------------------------------------------

fig.suptitle(
    "Figure 2: Day-of-Week Ridership Profile Shift, Pre-COVID Avg. vs. 2024",
    fontsize=14,
    fontweight="bold",
    y=1.03,
)

fig.text(
    0.5,
    -0.03,
    (
        "Note: High office-density group: Grand Central-42 St, Times Sq-42 St, "
        "47-50 Sts-Rockefeller Ctr, 34 St-Herald Sq, Fulton St (all complexes). "
        "Outer-borough residential group: Pelham Bay Park, Jamaica-179 St, "
        "Bay Ridge Av, Bay Ridge-95 St, Norwood Av, Norwood-205 St, "
        "Flatbush Av-Brooklyn College. "
        "Profiles are normalized to each station group's weekly ridership total. "
        "Pre-COVID = 2017-2019 average. 2024 = Stabilized New Normal."
    ),
    ha="center",
    va="top",
    fontsize=8,
)


plt.tight_layout()


# -------------------------------------------------------
# Save figure
# -------------------------------------------------------

fig_out = OUT_DIR / "figure2_dow_profile_shift_precovid_vs_2024.png"

plt.savefig(
    fig_out,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("\nFigure 2 saved: " + str(fig_out))



================ FIGURE 2 SYMMETRY CHECK ================

Every search term must match >=1 station in BOTH periods.
MISSING in either period = asymmetric group = fix required.

--- Office-Density ---
  [OK] 'Grand Central'
    precovid (1): ['Grand Central-42 St (4,5,6,7,S)']
    2024     (1): ['Grand Central-42 St (S,4,5,6,7)']
  [OK] 'Times Sq'
    precovid (1): ['Times Sq-42 St/Port Authority Bus Terminal (1,2,3,7,A,C,E,N,Q,R,W,S)']
    2024     (1): ['Times Sq-42 St (N,Q,R,W,S,1,2,3,7)/42 St (A,C,E)']
  [MISSING] '47-50'
    precovid (0): []
    2024     (1): ['47-50 Sts-Rockefeller Ctr (B,D,F,M)']
  [OK] 'Herald Sq'
    precovid (1): ['34 St-Herald Sq (B,D,F,M,N,Q,R,W)']
    2024     (1): ['34 St-Herald Sq (B,D,F,M,N,Q,R,W)']
  [OK] 'Fulton St'
    precovid (1): ['Fulton St (2,3,4,5,A,C,J,Z)']
    2024     (2): ['Fulton St (G)', 'Fulton St (A,C,J,Z,2,3,4,5)']

--- Residential ---
  [OK] 'Pelham Bay'
    precovid (1): ['Pelham Bay Park (6)']
    2024     (1): ['Pelham Bay Park (6

RuntimeError: Asymmetric station terms detected -- one or more search terms matched in one period but not the other.
Fix OFFICE_STATIONS or RESIDENTIAL_STATIONS above so every term appears in both the pre-COVID and 2024 datasets, then re-run.

### 3.3 Monday/Friday vs. Tuesday–Thursday Recovery Gap

**The headline quantitative finding.** Computes recovery rates (post-COVID mean / pre-COVID average mean) for each day of week and each study period. The Mon/Fri gap — the difference between Tue–Thu average recovery and Mon/Fri average recovery — is the key metric that motivates the paper's policy argument.

**Outputs:** `recovery_rates_by_dow.csv`, `recovery_gap_summary.csv`


In [ ]:
# ==============================================================
# Section 3.3 — Monday/Friday vs. Tuesday–Thursday Recovery Gap
#
# Purpose:
#   Quantifies the "Mon/Fri gap" — the structural divergence
#   between Monday/Friday recovery and Tuesday–Thursday recovery.
#
# Main metric:
#   Recovery Rate = period mean entries / pre-COVID average mean entries
#
# Baseline:
#   Pre-COVID Avg. = 2017–2019 average
#
# Main comparison:
#   2022 = Early Recovery
#   2024 = Stabilized New Normal
#
# Robustness:
#   2023 = Transitional Recovery
#
# Outputs:
#   recovery_rates_by_dow.csv
#   recovery_gap_summary.csv
# ==============================================================

import pandas as pd
import numpy as np


# -------------------------------------------------------
# Required DOW labels and periods
# -------------------------------------------------------

DOW_LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

PERIOD_LABELS = {
    "precovid": "Pre-COVID Avg. (2017–2019)",
    2022: "Early Recovery (2022)",
    2023: "Transitional Recovery (2023)",
    2024: "Stabilized New Normal (2024)",
}

RECOVERY_PERIODS = [2022, 2023, 2024]


# -------------------------------------------------------
# Validate inputs
# -------------------------------------------------------

required_periods = ["precovid"] + RECOVERY_PERIODS

missing_periods = [p for p in required_periods if p not in dow_raw]

if missing_periods:
    raise RuntimeError(
        f"dow_raw is missing required periods: {missing_periods}\n"
        f"Available keys: {list(dow_raw.keys())}\n"
        "Run Section 2.3 first."
    )

for period in required_periods:
    missing_cols = [c for c in DOW_LABELS if c not in dow_raw[period].columns]

    if missing_cols:
        raise RuntimeError(
            f"dow_raw[{period!r}] is missing DOW columns: {missing_cols}\n"
            f"Columns found: {list(dow_raw[period].columns)}"
        )


# -------------------------------------------------------
# Build recovery table by day of week
# -------------------------------------------------------

recovery_rows = []

for day in DOW_LABELS:
    baseline_mean = dow_raw["precovid"][day].mean()

    row = {
        "day": day,
        "baseline_mean_precovid": baseline_mean,
    }

    for period in RECOVERY_PERIODS:
        period_mean = dow_raw[period][day].mean()

        row[f"mean_{period}"] = period_mean

        row[f"recovery_{period}"] = (
            period_mean / baseline_mean
            if baseline_mean > 0
            else np.nan
        )

        row[f"pct_change_vs_precovid_{period}"] = (
            (period_mean - baseline_mean) / baseline_mean * 100
            if baseline_mean > 0
            else np.nan
        )

    recovery_rows.append(row)

recovery_df = (
    pd.DataFrame(recovery_rows)
    .set_index("day")
    .reindex(DOW_LABELS)
)


# -------------------------------------------------------
# Print recovery table with simple text bars
# -------------------------------------------------------

print("Recovery Rates by Day of Week")
print("Baseline: 2017–2019 Pre-COVID Average")
print("-" * 70)

for day, row in recovery_df.iterrows():
    print(f"\n{day}")

    for period in RECOVERY_PERIODS:
        recovery_value = row[f"recovery_{period}"]

        if pd.isna(recovery_value):
            bar = ""
            display_value = "NA"
        else:
            bar = "█" * int(recovery_value * 20)
            display_value = f"{recovery_value:5.1%}"

        print(
            f"  {period}: {display_value} {bar}"
        )


# -------------------------------------------------------
# Compute Mon/Fri vs Tue–Thu recovery gaps
# -------------------------------------------------------

gap_rows = []

for period in RECOVERY_PERIODS:
    monfri_recovery = recovery_df.loc[
        ["Mon", "Fri"],
        f"recovery_{period}"
    ].mean()

    tuethu_recovery = recovery_df.loc[
        ["Tue", "Wed", "Thu"],
        f"recovery_{period}"
    ].mean()

    weekend_recovery = recovery_df.loc[
        ["Sat", "Sun"],
        f"recovery_{period}"
    ].mean()

    gap = tuethu_recovery - monfri_recovery

    gap_rows.append(
        {
            "period": period,
            "period_label": PERIOD_LABELS[period],
            "monfri_recovery": monfri_recovery,
            "tuethu_recovery": tuethu_recovery,
            "weekend_recovery": weekend_recovery,
            "midweek_premium_over_monfri": gap,
        }
    )

gap_summary = pd.DataFrame(gap_rows)


# -------------------------------------------------------
# Print key findings
# -------------------------------------------------------

print("\n\nMon/Fri vs Tuesday–Thursday Recovery Gap")
print("Baseline: 2017–2019 Pre-COVID Average")
print("-" * 70)

for _, row in gap_summary.iterrows():
    print(f"\n{row['period_label']}")
    print(f"  Mon/Fri average recovery:       {row['monfri_recovery']:.1%}")
    print(f"  Tue–Thu average recovery:       {row['tuethu_recovery']:.1%}")
    print(f"  Weekend average recovery:       {row['weekend_recovery']:.1%}")
    print(
        f"  Midweek premium over Mon/Fri:   "
        f"{row['midweek_premium_over_monfri']:.1%}"
    )


# -------------------------------------------------------
# Save outputs
# -------------------------------------------------------

recovery_out = OUT_DIR / "recovery_rates_by_dow.csv"
gap_out = OUT_DIR / "recovery_gap_summary.csv"

recovery_df.to_csv(recovery_out)
gap_summary.to_csv(gap_out, index=False)

print("\nSaved outputs:")
print(f"  {recovery_out}")
print(f"  {gap_out}")


# -------------------------------------------------------
# Display clean tables
# -------------------------------------------------------

print("\nRecovery table preview:")
display(
    recovery_df[
        [
            "baseline_mean_precovid",
            "mean_2022",
            "recovery_2022",
            "mean_2023",
            "recovery_2023",
            "mean_2024",
            "recovery_2024",
        ]
    ].round(3)
)

print("\nGap summary:")
display(gap_summary.round(3))

---
## SECTION 4: Cluster Analysis

K-means clustering on normalized 7-day ridership vectors identifies distinct **demand regime typologies** that transcend the weekday/weekend binary. Five sub-tasks:

- **4.1** K-selection diagnostics (elbow + silhouette) → **Figure 1**
- **4.2** Final K-means fit and cluster assignment for all periods
- **4.3** Cluster profile analysis and labeling → **Table 2**
- **4.4** Radar chart visualization → **Figure 3**
- **4.5** Cluster transition matrix (pre-COVID → 2024) → **Table 3**

**Critical scaler discipline:** The StandardScaler is fit *only* on 2024 data. Pre-COVID, 2022, and 2023 vectors are *transformed* using the 2024-fitted scaler before cluster assignment. This ensures that cluster assignments across periods are comparable on the same scale — fitting a separate scaler per period would make cross-period comparisons invalid.

**K_FINAL is set in Section 0.3 and referenced here.** Do not redefine it in this section.


### 4.1 K-Selection: Elbow Method + Silhouette Score (Figure 1)

**Methodology:** Sweep K from 2 to 8. For each K, fit K-means on the 2024 normalized DOW vectors (low-volume stations excluded), record inertia (within-cluster sum of squares) and silhouette score. Select K at the elbow of the inertia curve, validated by silhouette.

**Decision note:** K=2 typically has the highest silhouette but produces only a weekday/weekend split — precisely the binary we are arguing against. K=4 yields four interpretable demand regimes while maintaining good silhouette scores and avoiding over-fragmentation at K≥5.


In [ ]:
# ==============================================================
# Section 4.1 — K-Selection: Elbow + Silhouette
#
# Purpose:
#   Determines the optimal number of clusters K for K-means
#   applied to normalized 7-day DOW ridership vectors.
#
# Method:
#   - Primary clustering target: 2024 Stabilized New Normal
#   - Sweep K = 2..8
#   - Record inertia for elbow method
#   - Record silhouette score for each K
#   - Select K_FINAL = 4 based on interpretability and diagnostics
#
# Input:
#   dow_vectors[2024]
#
# Outputs:
#   figure1_k_selection_2024.png
#   k_selection_diagnostics_2024.csv
# ==============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


# -------------------------------------------------------
# Settings
# -------------------------------------------------------

CLUSTER_YEAR = 2024

DOW_LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

K_RANGE = range(2, 9)

# Final selected K
# K=2 has the highest silhouette, but K=4 provides a more interpretable
# demand-regime solution while avoiding over-fragmentation at K>=5.
# K_FINAL is defined in Section 0.3 — do not override here.
# After running the diagnostics below, update K_FINAL in Section 0.3
# if the elbow/silhouette suggest a different value.
if 'K_FINAL' not in globals():
    K_FINAL = 4   # fallback only — should come from Section 0.3

if "KMEANS_INIT" not in globals():
    KMEANS_INIT = 50

if "KMEANS_SEED" not in globals():
    KMEANS_SEED = 42

if "PALETTE" not in globals():
    PALETTE = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]


# -------------------------------------------------------
# Validate inputs
# -------------------------------------------------------

if "dow_vectors" not in globals():
    raise RuntimeError(
        "dow_vectors not found. Run Section 2.3 first."
    )

if CLUSTER_YEAR not in dow_vectors:
    raise RuntimeError(
        f"dow_vectors[{CLUSTER_YEAR}] not found.\n"
        f"Available keys: {list(dow_vectors.keys())}\n"
        "Run Section 2.3 first."
    )

cluster_df = dow_vectors[CLUSTER_YEAR].copy()

missing_cols = [c for c in DOW_LABELS if c not in cluster_df.columns]

if missing_cols:
    raise RuntimeError(
        f"dow_vectors[{CLUSTER_YEAR}] is missing DOW columns: {missing_cols}\n"
        f"Columns found: {list(cluster_df.columns)}"
    )

if "low_volume_flag" not in cluster_df.columns:
    print("Warning: low_volume_flag not found. Creating default False flag.")
    cluster_df["low_volume_flag"] = False


# -------------------------------------------------------
# Prepare feature matrix
# -------------------------------------------------------

before_filter = len(cluster_df)

cluster_df = cluster_df[
    ~cluster_df["low_volume_flag"].fillna(False).astype(bool)
].copy()

after_filter = len(cluster_df)

if after_filter < 10:
    raise RuntimeError(
        f"Too few station complexes available for clustering: {after_filter}.\n"
        "Check Section 2.3 outputs."
    )

X = cluster_df[DOW_LABELS].astype(float).values

print("K-selection input summary")
print("-" * 50)
print(f"Analysis year: {CLUSTER_YEAR}")
print(f"Rows before low-volume filter: {before_filter:,}")
print(f"Rows after low-volume filter:  {after_filter:,}")
print(f"Feature matrix shape:          {X.shape}")


# -------------------------------------------------------
# Standardize features
# -------------------------------------------------------

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# -------------------------------------------------------
# K sweep
# -------------------------------------------------------

inertias = []
silhouettes = []
valid_k_values = []

print("\nK-selection diagnostics")
print("-" * 50)

for k in K_RANGE:
    if k >= len(cluster_df):
        print(f"Skipping K={k}: K must be less than number of observations.")
        continue

    km = KMeans(
        n_clusters=k,
        n_init=KMEANS_INIT,
        random_state=KMEANS_SEED,
        max_iter=500,
    )

    labels = km.fit_predict(X_scaled)

    inertia = km.inertia_
    silhouette = silhouette_score(X_scaled, labels)

    valid_k_values.append(k)
    inertias.append(inertia)
    silhouettes.append(silhouette)

    print(
        f"  K={k:2d}  "
        f"inertia={inertia:10.1f}  "
        f"silhouette={silhouette:.4f}"
    )


# -------------------------------------------------------
# Save diagnostics table
# -------------------------------------------------------

k_diag = pd.DataFrame(
    {
        "k": valid_k_values,
        "inertia": inertias,
        "silhouette": silhouettes,
        "selected_k": [k == K_FINAL for k in valid_k_values],
    }
)

k_diag_out = OUT_DIR / f"k_selection_diagnostics_{CLUSTER_YEAR}.csv"
k_diag.to_csv(k_diag_out, index=False)


# -------------------------------------------------------
# Plot diagnostics
# -------------------------------------------------------

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(
    k_diag["k"],
    k_diag["inertia"],
    "o-",
    color=PALETTE[0],
    linewidth=2,
)

ax1.axvline(
    K_FINAL,
    color=PALETTE[1],
    linestyle="--",
    label=f"Selected K={K_FINAL}",
)

ax1.set_xlabel("Number of Clusters (K)")
ax1.set_ylabel("Inertia (WCSS)")
ax1.set_title("Elbow Method", fontweight="bold")
ax1.legend()
ax1.grid(axis="y", alpha=0.3)


ax2.plot(
    k_diag["k"],
    k_diag["silhouette"],
    "s-",
    color=PALETTE[2],
    linewidth=2,
)

ax2.axvline(
    K_FINAL,
    color=PALETTE[1],
    linestyle="--",
    label=f"Selected K={K_FINAL}",
)

ax2.set_xlabel("Number of Clusters (K)")
ax2.set_ylabel("Silhouette Score")
ax2.set_title("Silhouette Score", fontweight="bold")
ax2.legend()
ax2.grid(axis="y", alpha=0.3)


plt.suptitle(
    f"Figure 1: K-Means Cluster Selection Diagnostics ({CLUSTER_YEAR} Data)",
    fontweight="bold",
)

plt.tight_layout()

fig_out = OUT_DIR / f"figure1_k_selection_{CLUSTER_YEAR}.png"

plt.savefig(
    fig_out,
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# -------------------------------------------------------
# Report selected K
# -------------------------------------------------------

best_silhouette_row = k_diag.loc[k_diag["silhouette"].idxmax()]

print("\nK-selection summary")
print("-" * 50)
print(f"Selected K_FINAL:                 {K_FINAL}")
print(
    f"Highest silhouette K:             "
    f"{int(best_silhouette_row['k'])} "
    f"(score={best_silhouette_row['silhouette']:.4f})"
)
print(f"Diagnostics saved:                {k_diag_out}")
print(f"Figure saved:                     {fig_out}")

print(
    "\nInterpretation note: K=2 has the highest silhouette score, "
    "but K=4 is selected as a more interpretable demand-regime solution. "
    "It preserves meaningful variation in day-of-week profiles while avoiding "
    "the weaker silhouette and likely over-fragmentation at K>=5."
)

### 4.2 Final K-Means Fit

**Scaler discipline:** Fit `StandardScaler` and `KMeans` on 2024 data only. Then use `scaler.transform()` (not `fit_transform()`) for all other periods. This is the correct cross-period comparison approach.

**Output files:** `cluster_assignments_{period}.csv` for pre-COVID, 2022, 2023, 2024.


In [ ]:
# ==============================================================
# Section 4.2 — Final K-Means Fit
#
# Purpose:
#   Fits K-means with K=K_FINAL on normalized 7-day vectors
#   using 2024 as the primary clustering/training period.
#
# Important:
#   - Scaler is fit ONLY on 2024 data.
#   - K-means is fit ONLY on 2024 data.
#   - Pre-COVID, 2022, and 2023 vectors are transformed using
#     the same 2024-fitted scaler and assigned to the 2024
#     cluster structure for cross-period comparison.
#
# Baseline:
#   Pre-COVID Avg. = 2017–2019 average
#
# Main periods:
#   Pre-COVID Avg., 2022, 2024
#
# Robustness:
#   2023
#
# Outputs:
#   cluster_assignments_precovid.csv
#   cluster_assignments_2022.csv
#   cluster_assignments_2023.csv
#   cluster_assignments_2024.csv
#   cluster_centers_2024.csv
#   cluster_assignment_summary.csv
# ==============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


# -------------------------------------------------------
# Settings
# -------------------------------------------------------

CLUSTER_YEAR = 2024
ASSIGNMENT_PERIODS = ["precovid", 2022, 2023, 2024]

DOW_LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

if "K_FINAL" not in globals():
    K_FINAL = 4   # fallback: must match Section 0.3 / Section 4.1 selection

if "KMEANS_INIT" not in globals():
    KMEANS_INIT = 50

if "KMEANS_SEED" not in globals():
    KMEANS_SEED = 42


# -------------------------------------------------------
# Validate inputs
# -------------------------------------------------------

if "dow_vectors" not in globals():
    raise RuntimeError(
        "dow_vectors not found. Run Section 2.3 first."
    )

missing_periods = [p for p in ASSIGNMENT_PERIODS if p not in dow_vectors]

if missing_periods:
    raise RuntimeError(
        f"dow_vectors is missing required periods: {missing_periods}\n"
        f"Available keys: {list(dow_vectors.keys())}\n"
        "Run Section 2.3 first."
    )

for period in ASSIGNMENT_PERIODS:
    missing_cols = [c for c in DOW_LABELS if c not in dow_vectors[period].columns]

    if missing_cols:
        raise RuntimeError(
            f"dow_vectors[{period!r}] is missing DOW columns: {missing_cols}\n"
            f"Columns found: {list(dow_vectors[period].columns)}"
        )

    if "low_volume_flag" not in dow_vectors[period].columns:
        print(f"Warning: low_volume_flag missing for {period}; setting to False.")
        dow_vectors[period]["low_volume_flag"] = False


# -------------------------------------------------------
# Prepare 2024 training data
# -------------------------------------------------------

train_df = dow_vectors[CLUSTER_YEAR].copy()

before_train = len(train_df)

train_df = train_df[
    ~train_df["low_volume_flag"].fillna(False).astype(bool)
].copy()

after_train = len(train_df)

if after_train < K_FINAL:
    raise RuntimeError(
        f"Too few 2024 station complexes for K={K_FINAL}: {after_train} rows."
    )

X_train = train_df[DOW_LABELS].astype(float).values

print("Final K-means training summary")
print("-" * 60)
print(f"Training period:              {CLUSTER_YEAR}")
print(f"K_FINAL:                      {K_FINAL}")
print(f"Rows before low-volume filter: {before_train:,}")
print(f"Rows after low-volume filter:  {after_train:,}")
print(f"Feature matrix shape:          {X_train.shape}")


# -------------------------------------------------------
# Fit scaler and final K-means on 2024 only
# -------------------------------------------------------

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

km_final = KMeans(
    n_clusters=K_FINAL,
    n_init=KMEANS_INIT,
    random_state=KMEANS_SEED,
    max_iter=1000,
)

km_final.fit(X_train_scaled)

print("\nFinal K-means model fit complete.")


# -------------------------------------------------------
# Save 2024 cluster centers in original DOW-share scale
# -------------------------------------------------------
# Cluster centers are fitted in standardized space. Convert them back
# to the original normalized DOW-share scale for interpretation.

centers_scaled = km_final.cluster_centers_
centers_original = scaler.inverse_transform(centers_scaled)

cluster_centers = pd.DataFrame(
    centers_original,
    columns=DOW_LABELS,
)

cluster_centers.insert(0, "cluster", range(K_FINAL))

# Normalize centers again for safety because inverse transform can create
# very small floating-point deviations from row sum = 1.
row_sums = cluster_centers[DOW_LABELS].sum(axis=1)

cluster_centers[DOW_LABELS] = cluster_centers[DOW_LABELS].div(
    row_sums,
    axis=0,
)

cluster_centers["weekday_share"] = cluster_centers[
    ["Mon", "Tue", "Wed", "Thu", "Fri"]
].sum(axis=1)

cluster_centers["weekend_share"] = cluster_centers[
    ["Sat", "Sun"]
].sum(axis=1)

cluster_centers["monfri_share"] = cluster_centers[
    ["Mon", "Fri"]
].sum(axis=1)

cluster_centers["tuethu_share"] = cluster_centers[
    ["Tue", "Wed", "Thu"]
].sum(axis=1)

cluster_centers["midweek_minus_monfri"] = (
    cluster_centers["tuethu_share"] -
    cluster_centers["monfri_share"]
)

centers_out = OUT_DIR / f"cluster_centers_{CLUSTER_YEAR}.csv"
cluster_centers.to_csv(centers_out, index=False)


# -------------------------------------------------------
# Assign clusters to each period
# -------------------------------------------------------

cluster_assignments = {}
summary_rows = []

for period in ASSIGNMENT_PERIODS:
    df_period = dow_vectors[period].copy()

    df_period = df_period[
        ~df_period["low_volume_flag"].fillna(False).astype(bool)
    ].copy()

    if df_period.empty:
        print(f"\nWarning: no rows available for period {period}. Skipping.")
        continue

    X_period = df_period[DOW_LABELS].astype(float).values
    X_period_scaled = scaler.transform(X_period)

    df_period["cluster"] = km_final.predict(X_period_scaled)
    df_period["period"] = period
    df_period["period_label"] = (
        "Pre-COVID Avg. (2017–2019)"
        if period == "precovid"
        else str(period)
    )

    # Add helpful profile metrics
    df_period["weekday_share"] = df_period[
        ["Mon", "Tue", "Wed", "Thu", "Fri"]
    ].sum(axis=1)

    df_period["weekend_share"] = df_period[
        ["Sat", "Sun"]
    ].sum(axis=1)

    df_period["monfri_share"] = df_period[
        ["Mon", "Fri"]
    ].sum(axis=1)

    df_period["tuethu_share"] = df_period[
        ["Tue", "Wed", "Thu"]
    ].sum(axis=1)

    df_period["midweek_minus_monfri"] = (
        df_period["tuethu_share"] -
        df_period["monfri_share"]
    )

    # Save assignment file
    period_file = "precovid" if period == "precovid" else str(period)
    cache = OUT_DIR / f"cluster_assignments_{period_file}.csv"

    df_period.to_csv(cache, index=False)

    cluster_assignments[period] = df_period

    dist = (
        df_period["cluster"]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    print(f"\n{period}: cluster assignments saved")
    print(f"  File: {cache}")
    print(f"  Rows: {len(df_period):,}")
    print(f"  Distribution: {dist}")

    for cluster_id, count in dist.items():
        summary_rows.append(
            {
                "period": period,
                "period_label": df_period["period_label"].iloc[0],
                "cluster": cluster_id,
                "n_station_complexes": count,
                "share_of_period": count / len(df_period),
            }
        )


# -------------------------------------------------------
# Save cluster assignment summary
# -------------------------------------------------------

cluster_assignment_summary = pd.DataFrame(summary_rows)

summary_out = OUT_DIR / "cluster_assignment_summary.csv"

cluster_assignment_summary.to_csv(summary_out, index=False)

print("\nK-means fitting and cluster assignment complete.")
print(f"Cluster centers saved:      {centers_out}")
print(f"Assignment summary saved:   {summary_out}")


# -------------------------------------------------------
# Display summaries
# -------------------------------------------------------

print("\nCluster centers, original DOW-share scale:")
display(cluster_centers.round(4))

print("\nCluster assignment summary:")
display(cluster_assignment_summary.round(4))

### 4.3 Cluster Profile Analysis and Labeling (Table 2)

**Methodology:** Compute mean normalized DOW shares for each cluster in 2024. Add interpretive metrics (weekday share, Mon/Fri share, Tue–Thu share, midweek premium). Apply rule-based provisional labels, then override with manual labels based on profile shape and representative stations.

**Manual labels must be updated** after you inspect the actual cluster profiles from your data run. The defaults reflect the expected K=4 solution based on domain knowledge.


In [ ]:
# ==============================================================
# Section 4.3 — Cluster Profile Analysis and Labeling
#
# Purpose:
#   Computes mean normalized DOW profile for each 2024 cluster
#   and generates interpretive labels based on profile shape and
#   representative stations.
#
# This produces Table 2 in the TRR paper.
#
# Important:
#   - Clusters were trained on 2024 in Section 4.2.
#   - Manual labels below are based on the observed 2024 cluster
#     profiles and representative high-volume station complexes.
#
# Outputs:
#   table2_cluster_profiles.csv
#   table2_cluster_profiles_labeled.csv
#   cluster_label_mapping.csv
#   cluster_assignments_{period}_labeled.csv
#   representative_stations_by_cluster.csv
# ==============================================================

import pandas as pd
import numpy as np


# -------------------------------------------------------
# Required setup
# -------------------------------------------------------

DOW_LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

ASSIGNMENT_PERIODS = ["precovid", 2022, 2023, 2024]

if "cluster_assignments" not in globals():
    raise RuntimeError(
        "cluster_assignments not found. Run Section 4.2 first."
    )

if 2024 not in cluster_assignments:
    raise RuntimeError(
        f"cluster_assignments[2024] not found. "
        f"Available keys: {list(cluster_assignments.keys())}"
    )

for period in ASSIGNMENT_PERIODS:
    if period not in cluster_assignments:
        raise RuntimeError(
            f"cluster_assignments[{period!r}] not found. "
            f"Available keys: {list(cluster_assignments.keys())}\n"
            "Run Section 4.2 first."
        )


# -------------------------------------------------------
# Use 2024 cluster assignments for profile labeling
# -------------------------------------------------------

df_2024_clusters = cluster_assignments[2024].copy()

missing_cols = [
    c for c in ["cluster", "complex_name", "avg_weekday_entries"] + DOW_LABELS
    if c not in df_2024_clusters.columns
]

if missing_cols:
    raise RuntimeError(
        f"2024 cluster assignment data is missing required columns: {missing_cols}\n"
        f"Columns found: {list(df_2024_clusters.columns)}"
    )


# -------------------------------------------------------
# Compute cluster profiles in original normalized DOW-share space
# -------------------------------------------------------

cluster_profiles = (
    df_2024_clusters
    .groupby("cluster")[DOW_LABELS]
    .mean()
    .round(4)
)

cluster_sizes = (
    df_2024_clusters["cluster"]
    .value_counts()
    .sort_index()
    .rename("n_station_complexes")
)

cluster_profiles = cluster_profiles.join(cluster_sizes)


# -------------------------------------------------------
# Add interpretable profile metrics
# -------------------------------------------------------

cluster_profiles["weekday_share"] = cluster_profiles[
    ["Mon", "Tue", "Wed", "Thu", "Fri"]
].sum(axis=1)

cluster_profiles["weekend_share"] = cluster_profiles[
    ["Sat", "Sun"]
].sum(axis=1)

cluster_profiles["monfri_share"] = cluster_profiles[
    ["Mon", "Fri"]
].sum(axis=1)

cluster_profiles["tuethu_share"] = cluster_profiles[
    ["Tue", "Wed", "Thu"]
].sum(axis=1)

cluster_profiles["midweek_minus_monfri"] = (
    cluster_profiles["tuethu_share"] -
    cluster_profiles["monfri_share"]
)

cluster_profiles["fri_minus_tuethu_avg"] = (
    cluster_profiles["Fri"] -
    cluster_profiles[["Tue", "Wed", "Thu"]].mean(axis=1)
)

cluster_profiles["sat_sun_ratio"] = (
    cluster_profiles["Sat"] /
    cluster_profiles["Sun"].replace(0, np.nan)
)

cluster_profiles["peak_day"] = cluster_profiles[DOW_LABELS].idxmax(axis=1)
cluster_profiles["lowest_day"] = cluster_profiles[DOW_LABELS].idxmin(axis=1)

cluster_profiles["profile_range"] = (
    cluster_profiles[DOW_LABELS].max(axis=1) -
    cluster_profiles[DOW_LABELS].min(axis=1)
)


# -------------------------------------------------------
# Rule-based provisional cluster labels
# -------------------------------------------------------

def assign_cluster_label(row):
    """
    Assign provisional descriptive labels based on 2024 DOW profile shape.
    Manual labels below override these provisional labels.
    """

    midweek_premium = row["midweek_minus_monfri"]
    weekend_share = row["weekend_share"]
    weekday_share = row["weekday_share"]
    profile_range = row["profile_range"]
    peak_day = row["peak_day"]
    friday_gap = row["fri_minus_tuethu_avg"]

    # Strong Tue-Thu concentration and weak Friday/Monday
    if midweek_premium >= 0.04 and friday_gap <= -0.01:
        return "Hybrid Office Core"

    # Weekend-heavy or leisure/amenity profile
    if weekend_share >= 0.30 or peak_day in ["Sat", "Sun"]:
        return "Weekend / Amenity-Oriented"

    # Very flat profile across the week
    if profile_range <= 0.035:
        return "Stable All-Week / Residential-Essential"

    # Strong weekday but not necessarily hybrid-office shape
    if weekday_share >= 0.75 and midweek_premium > 0:
        return "Weekday-Oriented Commuter"

    # Moderate shape, mixed-use
    return "Mixed-Use / Balanced"


cluster_profiles["cluster_label_provisional"] = cluster_profiles.apply(
    assign_cluster_label,
    axis=1
)


# -------------------------------------------------------
# Manual label override for K=4 solution
# -------------------------------------------------------
# Based on observed cluster profiles and representative stations:
#
# Cluster 0:
#   Highest weekend share among the four clusters; includes major mixed-use
#   hubs such as Union Square, Jackson Heights, Canal St, Atlantic Av.
#
# Cluster 1:
#   Weekday-heavy but generally lower-volume neighborhood/residential
#   commuter stations.
#
# Cluster 2:
#   Largest regional/Midtown-oriented hubs such as Times Square,
#   Herald Sq, Penn Station, Columbus Circle.
#
# Cluster 3:
#   Strongest Tue–Thu concentration and office-core representatives:
#   Grand Central, Fulton St, WTC, Lexington Av-53 St, Rockefeller Ctr.

MANUAL_CLUSTER_LABELS = {
    0: "Balanced Hub / Mixed-Use",
    1: "Neighborhood Weekday Commuter",
    2: "Major Midtown / Regional Core",
    3: "Hybrid Office Core",
}

cluster_profiles["cluster_label"] = cluster_profiles.index.map(
    lambda c: MANUAL_CLUSTER_LABELS.get(
        c,
        cluster_profiles.loc[c, "cluster_label_provisional"]
    )
)


# -------------------------------------------------------
# Save Table 2 outputs
# -------------------------------------------------------

table2_out = OUT_DIR / "table2_cluster_profiles.csv"
table2_labeled_out = OUT_DIR / "table2_cluster_profiles_labeled.csv"
label_map_out = OUT_DIR / "cluster_label_mapping.csv"

cluster_profiles[DOW_LABELS + ["n_station_complexes"]].to_csv(table2_out)

cluster_profiles.to_csv(table2_labeled_out)

cluster_label_mapping = (
    cluster_profiles[["cluster_label"]]
    .reset_index()
    .rename(columns={"index": "cluster"})
)

cluster_label_mapping.to_csv(label_map_out, index=False)


# -------------------------------------------------------
# Apply labels to all period assignment files
# -------------------------------------------------------

CLUSTER_LABELS = cluster_profiles["cluster_label"].to_dict()

for period in ASSIGNMENT_PERIODS:
    df_period = cluster_assignments[period].copy()

    df_period["cluster_label"] = df_period["cluster"].map(CLUSTER_LABELS)

    cluster_assignments[period] = df_period

    period_file = "precovid" if period == "precovid" else str(period)

    labeled_out = OUT_DIR / f"cluster_assignments_{period_file}_labeled.csv"

    df_period.to_csv(labeled_out, index=False)

    print(f"Saved labeled cluster assignments for {period}: {labeled_out}")


# -------------------------------------------------------
# Print Table 2
# -------------------------------------------------------

print("\nTable 2: Cluster Profiles — Mean Normalized Day-of-Week Ridership Share (2024)")
print("-" * 90)

display_cols = (
    DOW_LABELS
    + [
        "n_station_complexes",
        "weekday_share",
        "weekend_share",
        "monfri_share",
        "tuethu_share",
        "midweek_minus_monfri",
        "fri_minus_tuethu_avg",
        "peak_day",
        "lowest_day",
        "profile_range",
        "cluster_label_provisional",
        "cluster_label",
    ]
)

display(cluster_profiles[display_cols].round(4))


print("\nCluster sizes:")
print(cluster_sizes.to_string())


# -------------------------------------------------------
# Print representative stations per cluster
# -------------------------------------------------------

print("\nRepresentative Stations by Cluster")
print("Top 5 by average weekday entries within each 2024 cluster")
print("-" * 90)

for c_id in sorted(cluster_profiles.index):
    c_label = cluster_profiles.loc[c_id, "cluster_label"]

    members = (
        df_2024_clusters[df_2024_clusters["cluster"] == c_id]
        .sort_values("avg_weekday_entries", ascending=False)
        .head(5)
    )

    cols_to_show = ["complex_name", "avg_weekday_entries"]

    if "borough" in members.columns:
        cols_to_show = ["complex_name", "borough", "avg_weekday_entries"]

    print(f"\nCluster {c_id}: {c_label}")
    print(members[cols_to_show].to_string(index=False))


# -------------------------------------------------------
# Save representative stations table
# -------------------------------------------------------

rep_rows = []

for c_id in sorted(cluster_profiles.index):
    c_label = cluster_profiles.loc[c_id, "cluster_label"]

    members = (
        df_2024_clusters[df_2024_clusters["cluster"] == c_id]
        .sort_values("avg_weekday_entries", ascending=False)
        .head(10)
        .copy()
    )

    members["cluster_label"] = c_label
    members["rank_within_cluster"] = range(1, len(members) + 1)

    keep_cols = [
        "cluster",
        "cluster_label",
        "rank_within_cluster",
        "complex_name",
        "avg_weekday_entries",
    ]

    if "borough" in members.columns:
        keep_cols.insert(4, "borough")

    rep_rows.append(members[keep_cols])

representative_stations = pd.concat(rep_rows, ignore_index=True)

rep_out = OUT_DIR / "representative_stations_by_cluster.csv"

representative_stations.to_csv(rep_out, index=False)


# -------------------------------------------------------
# Save compact cluster label interpretation table
# -------------------------------------------------------

cluster_interpretation = cluster_profiles[
    [
        "cluster_label",
        "cluster_label_provisional",
        "n_station_complexes",
        "weekday_share",
        "weekend_share",
        "monfri_share",
        "tuethu_share",
        "midweek_minus_monfri",
        "peak_day",
        "lowest_day",
        "profile_range",
    ]
].reset_index()

interpretation_out = OUT_DIR / "cluster_interpretation_summary.csv"

cluster_interpretation.to_csv(interpretation_out, index=False)


# -------------------------------------------------------
# Final output summary
# -------------------------------------------------------

print("\nSaved outputs:")
print(f"  {table2_out}")
print(f"  {table2_labeled_out}")
print(f"  {label_map_out}")
print(f"  {rep_out}")
print(f"  {interpretation_out}")

print("\nFinal cluster label mapping:")
for c_id, label in CLUSTER_LABELS.items():
    print(f"  Cluster {c_id}: {label}")

### 4.4 Figure 3: Radar/Spider Chart of Cluster Profiles

**Visualization:** Each cluster's 7-day profile plotted as a filled radar chart on a common radial scale. Station count (n=) shown in the center of each radar. Saves both PNG (300 DPI for journal submission) and PDF.


In [ ]:
# ==============================================================
# Section 4.4 — Figure 3: Radar Chart of Cluster Profiles
#
# Purpose:
#   Visualizes the 7-day ridership profile for each cluster
#   as a radar/spider chart.
#
# Analysis period:
#   2024 Stabilized New Normal
#
# This becomes Figure 3 in the TRR paper.
#
# Outputs:
#   figure3_radar_cluster_profiles.png
#   figure3_radar_cluster_profiles.pdf
# ==============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


# -------------------------------------------------------
# Required setup
# -------------------------------------------------------

DOW_LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

if "cluster_profiles" not in globals():
    raise RuntimeError(
        "cluster_profiles not found. Run Section 4.3 first."
    )

if "CLUSTER_LABELS" not in globals():
    if "cluster_label" in cluster_profiles.columns:
        CLUSTER_LABELS = cluster_profiles["cluster_label"].to_dict()
    else:
        raise RuntimeError(
            "CLUSTER_LABELS not found and cluster_profiles has no cluster_label column. "
            "Run Section 4.3 first."
        )

if "cluster_sizes" not in globals():
    if "n_station_complexes" in cluster_profiles.columns:
        cluster_sizes = cluster_profiles["n_station_complexes"]
    else:
        raise RuntimeError(
            "cluster_sizes not found and cluster_profiles has no n_station_complexes column. "
            "Run Section 4.3 first."
        )

missing_cols = [c for c in DOW_LABELS if c not in cluster_profiles.columns]

if missing_cols:
    raise RuntimeError(
        f"cluster_profiles is missing DOW columns: {missing_cols}\n"
        f"Columns found: {list(cluster_profiles.columns)}"
    )

if "PALETTE" not in globals():
    PALETTE = [
        "#1f77b4",
        "#ff7f0e",
        "#2ca02c",
        "#d62728",
        "#9467bd",
        "#8c564b",
        "#e377c2",
        "#7f7f7f",
    ]


# -------------------------------------------------------
# Prepare radar geometry
# -------------------------------------------------------

categories = DOW_LABELS
N = len(categories)

angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

cluster_ids = sorted(cluster_profiles.index.tolist())
n_clusters = len(cluster_ids)

# Use a common radial scale across all clusters
max_value = cluster_profiles[DOW_LABELS].max().max()
radial_max = np.ceil(max_value * 100) / 100

# Add a small buffer
radial_max = max(radial_max + 0.01, 0.20)


# -------------------------------------------------------
# Create subplot grid
# -------------------------------------------------------

n_cols = min(n_clusters, 5)
n_rows = int(np.ceil(n_clusters / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(4.2 * n_cols, 4.2 * n_rows),
    subplot_kw=dict(polar=True),
)

axes = np.array(axes).reshape(-1)


# -------------------------------------------------------
# Plot each cluster
# -------------------------------------------------------

for idx, c_id in enumerate(cluster_ids):
    ax = axes[idx]

    c_label = CLUSTER_LABELS.get(c_id, f"Cluster {c_id}")

    values = cluster_profiles.loc[c_id, DOW_LABELS].astype(float).tolist()
    values += values[:1]

    color = PALETTE[idx % len(PALETTE)]

    ax.plot(
        angles,
        values,
        linewidth=2.5,
        color=color,
    )

    ax.fill(
        angles,
        values,
        alpha=0.20,
        color=color,
    )

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=9)

    ax.set_ylim(0, radial_max)

    ax.yaxis.set_major_formatter(
        mticker.PercentFormatter(xmax=1, decimals=0)
    )

    ax.tick_params(axis="y", labelsize=7)

    n_stations = int(cluster_sizes.get(c_id, 0))

    ax.set_title(
        f"Cluster {c_id}\n{c_label}",
        fontsize=10,
        fontweight="bold",
        pad=18,
    )

    ax.text(
        0,
        0,
        f"n={n_stations}",
        ha="center",
        va="center",
        fontsize=8,
        color="gray",
        fontweight="bold",
    )

    # Optional subtle grid styling
    ax.grid(alpha=0.35)


# Hide unused axes if K does not fill the grid
for j in range(n_clusters, len(axes)):
    axes[j].set_visible(False)


# -------------------------------------------------------
# Title and note
# -------------------------------------------------------

fig.suptitle(
    "Figure 3: Day-of-Week Demand Regime Profiles by Cluster (2024)",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)

fig.text(
    0.5,
    -0.02,
    (
        "Note: Values represent each cluster's mean normalized share of weekly ridership "
        "by day of week. Clusters are fitted on 2024 station-complex DOW profiles."
    ),
    ha="center",
    va="top",
    fontsize=9,
)


plt.tight_layout()


# -------------------------------------------------------
# Save figure
# -------------------------------------------------------

fig_png_out = OUT_DIR / "figure3_radar_cluster_profiles.png"
fig_pdf_out = OUT_DIR / "figure3_radar_cluster_profiles.pdf"

plt.savefig(
    fig_png_out,
    dpi=300,
    bbox_inches="tight",
)

plt.savefig(
    fig_pdf_out,
    bbox_inches="tight",
)

plt.show()

print("Figure 3 saved:")
print(f"  {fig_png_out}")
print(f"  {fig_pdf_out}")

### 4.5 Cluster Transition Analysis — Pre-COVID → 2024 (Table 3)

**Methodology:** Inner-join stations present in both the pre-COVID and 2024 cluster assignments. Cross-tabulate to produce the transition matrix (rows = pre-COVID cluster, columns = 2024 cluster). Report stability metrics and identify high-volume stations that migrated between regimes.

**This is a novel contribution** — prior clustering studies classify stations at a single point in time. Showing *how* demand regimes shifted is analytically distinct from showing *that* ridership changed.


In [ ]:
# ==============================================================
# Section 4.5 — Cluster Transition Analysis
# Pre-COVID Avg. → 2024
#
# Purpose:
#   Tracks how station complexes migrated between demand-regime
#   clusters from the 2017–2019 pre-COVID baseline to the
#   stabilized new normal in 2024.
#
# This produces Table 3 in the TRR paper.
#
# Method:
#   - Inner join station complexes present in both pre-COVID and 2024
#   - Cross-tabulate cluster assignments
#   - Report transition counts, row percentages, and stability metrics
#
# Inputs:
#   cluster_assignments["precovid"]
#   cluster_assignments[2024]
#
# Outputs:
#   table3_transition_matrix_counts.csv
#   table3_transition_matrix_row_pct.csv
#   station_transitions_precovid_to_2024.csv
#   top_cluster_migrants_precovid_to_2024.csv
# ==============================================================

import pandas as pd
import numpy as np


# -------------------------------------------------------
# Required setup
# -------------------------------------------------------

BASELINE_PERIOD = "precovid"
TARGET_PERIOD = 2024

DOW_LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

if "cluster_assignments" not in globals():
    raise RuntimeError(
        "cluster_assignments not found. Run Section 4.2 and Section 4.3 first."
    )

for period in [BASELINE_PERIOD, TARGET_PERIOD]:
    if period not in cluster_assignments:
        raise RuntimeError(
            f"cluster_assignments[{period!r}] not found. "
            f"Available keys: {list(cluster_assignments.keys())}\n"
            "Run Section 4.2 first, then Section 4.3 for labels."
        )

required_cols = ["station_complex_id", "complex_name", "cluster", "cluster_label"]

for period in [BASELINE_PERIOD, TARGET_PERIOD]:
    missing_cols = [
        c for c in required_cols
        if c not in cluster_assignments[period].columns
    ]

    if missing_cols:
        raise RuntimeError(
            f"cluster_assignments[{period!r}] is missing required columns: {missing_cols}\n"
            f"Columns found: {list(cluster_assignments[period].columns)}\n"
            "Run Section 4.3 to add cluster labels."
        )


# -------------------------------------------------------
# Prepare baseline and 2024 assignment tables
# -------------------------------------------------------

df_base = (
    cluster_assignments[BASELINE_PERIOD][
        [
            "station_complex_id",
            "complex_name",
            "cluster",
            "cluster_label",
        ]
    ]
    .copy()
    .rename(
        columns={
            "complex_name": "complex_name_precovid",
            "cluster": "cluster_precovid",
            "cluster_label": "label_precovid",
        }
    )
)

df_target = (
    cluster_assignments[TARGET_PERIOD][
        [
            "station_complex_id",
            "complex_name",
            "cluster",
            "cluster_label",
        ]
    ]
    .copy()
    .rename(
        columns={
            "complex_name": "complex_name_2024",
            "cluster": "cluster_2024",
            "cluster_label": "label_2024",
        }
    )
)


# -------------------------------------------------------
# Inner join common station-complex universe
# -------------------------------------------------------

transitions = df_base.merge(
    df_target,
    on="station_complex_id",
    how="inner",
)

if transitions.empty:
    raise RuntimeError(
        "No overlapping station_complex_id values between pre-COVID and 2024 "
        "cluster assignments. Check Section 2.2 station-complex join."
    )

# Prefer 2024 name where available, but keep both for QA
transitions["complex_name"] = transitions["complex_name_2024"].fillna(
    transitions["complex_name_precovid"]
)

transitions["cluster_changed"] = (
    transitions["cluster_precovid"] != transitions["cluster_2024"]
)

transitions["transition_label"] = (
    transitions["label_precovid"].astype(str)
    + " → "
    + transitions["label_2024"].astype(str)
)


# -------------------------------------------------------
# Transition matrix: counts
# -------------------------------------------------------

transition_matrix_counts = pd.crosstab(
    transitions["label_precovid"],
    transitions["label_2024"],
    margins=True,
    margins_name="Total",
)


# -------------------------------------------------------
# Transition matrix: row percentages
# -------------------------------------------------------

transition_matrix_row_pct = pd.crosstab(
    transitions["label_precovid"],
    transitions["label_2024"],
    normalize="index",
) * 100

transition_matrix_row_pct = transition_matrix_row_pct.round(1)


# -------------------------------------------------------
# Stability metrics
# -------------------------------------------------------

n_total = len(transitions)
n_stable = (~transitions["cluster_changed"]).sum()
n_migrated = transitions["cluster_changed"].sum()

pct_stable = n_stable / n_total * 100
pct_migrated = n_migrated / n_total * 100


# -------------------------------------------------------
# Merge 2024 volume data for top movers
# -------------------------------------------------------

if "dow_raw" in globals() and 2024 in dow_raw and "avg_weekday_entries" in dow_raw[2024].columns:
    volume_2024 = dow_raw[2024][
        ["station_complex_id", "avg_weekday_entries"]
    ].copy()

    transitions = transitions.merge(
        volume_2024,
        on="station_complex_id",
        how="left",
    )
else:
    transitions["avg_weekday_entries"] = np.nan


# -------------------------------------------------------
# Identify high-volume migrants
# -------------------------------------------------------

migrants = transitions[transitions["cluster_changed"]].copy()

top_migrants = (
    migrants
    .sort_values("avg_weekday_entries", ascending=False, na_position="last")
    .head(15)
    [
        [
            "station_complex_id",
            "complex_name",
            "label_precovid",
            "label_2024",
            "cluster_precovid",
            "cluster_2024",
            "avg_weekday_entries",
        ]
    ]
)


# -------------------------------------------------------
# Transition-pair summary
# -------------------------------------------------------

transition_pair_summary = (
    transitions
    .groupby(
        [
            "label_precovid",
            "label_2024",
            "cluster_changed",
        ],
        as_index=False,
    )
    .agg(
        n_station_complexes=("station_complex_id", "nunique"),
        mean_2024_weekday_entries=("avg_weekday_entries", "mean"),
    )
    .sort_values(
        ["n_station_complexes", "mean_2024_weekday_entries"],
        ascending=[False, False],
    )
)


# -------------------------------------------------------
# Print results
# -------------------------------------------------------

print("Table 3: Cluster Transition Matrix")
print("Pre-COVID Avg. (2017–2019) → 2024 Stabilized New Normal")
print("Rows = pre-COVID cluster | Columns = 2024 cluster")
print("-" * 90)
print(transition_matrix_counts.to_string())

print("\nTable 3 Row Percentages")
print("Rows sum to 100%, excluding Total row")
print("-" * 90)
print(transition_matrix_row_pct.to_string())

print("\nCluster Stability Metrics")
print("-" * 90)
print(f"Common station-complex universe:          {n_total:,}")
print(f"Same cluster in both periods:             {n_stable:,} ({pct_stable:.1f}%)")
print(f"Migrated to a different cluster:          {n_migrated:,} ({pct_migrated:.1f}%)")

print("\nTop High-Volume Stations That Migrated Between Clusters")
print("-" * 90)

if top_migrants.empty:
    print("No migrated stations found.")
else:
    print(top_migrants.to_string(index=False))

print("\nTransition Pair Summary")
print("-" * 90)
display(transition_pair_summary)


# -------------------------------------------------------
# Save outputs
# -------------------------------------------------------

counts_out = OUT_DIR / "table3_transition_matrix_counts.csv"
row_pct_out = OUT_DIR / "table3_transition_matrix_row_pct.csv"
transitions_out = OUT_DIR / "station_transitions_precovid_to_2024.csv"
top_migrants_out = OUT_DIR / "top_cluster_migrants_precovid_to_2024.csv"
pair_summary_out = OUT_DIR / "cluster_transition_pair_summary.csv"

transition_matrix_counts.to_csv(counts_out)
transition_matrix_row_pct.to_csv(row_pct_out)
transitions.to_csv(transitions_out, index=False)
top_migrants.to_csv(top_migrants_out, index=False)
transition_pair_summary.to_csv(pair_summary_out, index=False)

print("\nSaved outputs:")
print(f"  {counts_out}")
print(f"  {row_pct_out}")
print(f"  {transitions_out}")
print(f"  {top_migrants_out}")
print(f"  {pair_summary_out}")

---
## SECTION 5: Regression Analysis

Explain 2024 cluster membership using built-environment features via multinomial logistic regression. This section answers: *What structural characteristics predict which demand regime a station belongs to?*

Four sub-tasks:
- **5.1** Spatial join: 0.5-mile station buffers × Census block groups (GeoPandas)
- **5.2** Feature engineering: log transforms, share variables, density features
- **5.3** Multinomial logit (statsmodels MNLogit) + sklearn 5-fold CV
- **5.4** Publication-ready Table 4 formatting

**Note on Section 5.3:** The notebook contains a simpler version (without explicit reference category) and the recommended version with `REFERENCE_LABEL`. **Only the version with explicit reference category is included here.** The reference category is `"Major Midtown / Regional Core"` — the most theoretically appropriate baseline since it represents the traditional CBD commuter pattern.


### 5.1 Spatial Join: Built Environment → Station Complexes

**Methodology:** Buffer each station complex by 0.5 miles in NYC State Plane coordinates (EPSG:2263 — feet). Intersect with Census block group geometries. Area-weight demographic variables (summing population, jobs, households proportional to the share of each block group inside the buffer). Weighted average for median income.

**Dependencies:** Requires `cluster_assignments[2024]` (from Section 4.3) for station coordinates and cluster labels. Also requires `station_complex_df` (from Section 1.2) as a coordinate fallback.

**Runtime note:** The `gpd.overlay()` spatial intersection is the slowest step (~5–10 minutes on Colab CPU). The output is cached to Drive.


In [ ]:
# ==============================================================
# Section 5.1 — Spatial Join: Built Environment → Stations
#
# Purpose:
#   Aggregates ACS and LODES data to station-complex-level features
#   using 0.5-mile station-complex buffers.
#
# Requires:
#   geopandas, shapely, requests
#
# Processing Steps:
#   1. Recover station-complex coordinates.
#   2. Project station coordinates to NYC local CRS (EPSG:2263).
#   3. Buffer each station complex by 0.5 miles.
#   4. Load NYC Census block group geometries.
#   5. Join ACS and LODES attributes to block groups.
#   6. Intersect block groups with station buffers.
#   7. Compute area-weighted ACS and LODES features.
#   8. Save station_features.csv.
#
# Output:
#   station_features.csv
# ==============================================================

import pandas as pd
import numpy as np
import geopandas as gpd
import requests

from pathlib import Path
from shapely.geometry import Point


# -------------------------------------------------------
# Required settings
# -------------------------------------------------------

if "OUT_DIR" not in globals():
    OUT_DIR = Path("/content/drive/MyDrive/dow_ridership_paper/outputs")

OUT_DIR.mkdir(parents=True, exist_ok=True)

if "CATCHMENT_MILES" not in globals():
    CATCHMENT_MILES = 0.5

BUFFER_FT = CATCHMENT_MILES * 5280

NYC_COUNTIES = ["061", "005", "047", "081", "085"]

ACS_FEATURE_YEAR = 2022
LODES_FEATURE_YEAR = 2021

ACS_FILE = OUT_DIR / f"acs_bg_{ACS_FEATURE_YEAR}.csv"
LODES_FILE = OUT_DIR / f"lodes_wac_{LODES_FEATURE_YEAR}.csv"

TIGER_URL = "https://www2.census.gov/geo/tiger/TIGER2022/BG/tl_2022_36_bg.zip"
tiger_cache = OUT_DIR / "tl_2022_36_bg.zip"


# -------------------------------------------------------
# Validate cluster assignments
# -------------------------------------------------------

if "cluster_assignments" not in globals():
    raise RuntimeError(
        "cluster_assignments not found. Run Sections 4.2 and 4.3 first."
    )

if 2024 not in cluster_assignments:
    raise RuntimeError(
        f"cluster_assignments[2024] not found. Available keys: {list(cluster_assignments.keys())}"
    )


# -------------------------------------------------------
# Recover station-complex coordinates
# -------------------------------------------------------

station_base = cluster_assignments[2024].copy()

required_base_cols = ["station_complex_id", "complex_name"]

missing_base_cols = [
    c for c in required_base_cols
    if c not in station_base.columns
]

if missing_base_cols:
    raise RuntimeError(
        f"cluster_assignments[2024] is missing required columns: {missing_base_cols}\n"
        f"Columns found: {list(station_base.columns)}"
    )

station_geo_df = station_base[
    ["station_complex_id", "complex_name"]
].drop_duplicates().copy()

station_geo_df["station_complex_id"] = pd.to_numeric(
    station_geo_df["station_complex_id"],
    errors="coerce"
)

# Try to use coordinates already in cluster_assignments[2024]
if {"complex_lat", "complex_lon"}.issubset(station_base.columns):
    coord_df = (
        station_base[
            [
                "station_complex_id",
                "complex_lat",
                "complex_lon",
            ]
        ]
        .drop_duplicates(subset=["station_complex_id"])
        .copy()
    )

    station_geo_df = station_geo_df.merge(
        coord_df,
        on="station_complex_id",
        how="left",
    )

else:
    station_geo_df["complex_lat"] = np.nan
    station_geo_df["complex_lon"] = np.nan


# Fallback: recover coordinates from station_complex_df
if station_geo_df[["complex_lat", "complex_lon"]].isna().any(axis=None):
    if "station_complex_df" not in globals():
        raise RuntimeError(
            "Missing coordinates in cluster_assignments[2024], and station_complex_df "
            "is not available. Reload the MTA station complex file before Section 5.1."
        )

    sc = station_complex_df.copy()
    sc.columns = sc.columns.str.strip().str.lower()

    required_sc_cols = {"complex_id", "latitude", "longitude"}

    missing_sc_cols = required_sc_cols - set(sc.columns)

    if missing_sc_cols:
        raise RuntimeError(
            f"station_complex_df is missing required columns: {missing_sc_cols}\n"
            f"Columns found: {list(sc.columns)}"
        )

    coord_lookup = (
        sc[
            [
                "complex_id",
                "latitude",
                "longitude",
            ]
        ]
        .copy()
        .rename(
            columns={
                "complex_id": "station_complex_id",
                "latitude": "complex_lat_lookup",
                "longitude": "complex_lon_lookup",
            }
        )
    )

    coord_lookup["station_complex_id"] = pd.to_numeric(
        coord_lookup["station_complex_id"],
        errors="coerce"
    )

    coord_lookup["complex_lat_lookup"] = pd.to_numeric(
        coord_lookup["complex_lat_lookup"],
        errors="coerce"
    )

    coord_lookup["complex_lon_lookup"] = pd.to_numeric(
        coord_lookup["complex_lon_lookup"],
        errors="coerce"
    )

    coord_lookup = coord_lookup.dropna(
        subset=[
            "station_complex_id",
            "complex_lat_lookup",
            "complex_lon_lookup",
        ]
    )

    coord_lookup = coord_lookup.drop_duplicates(
        subset=["station_complex_id"]
    )

    station_geo_df = station_geo_df.merge(
        coord_lookup,
        on="station_complex_id",
        how="left",
    )

    station_geo_df["complex_lat"] = station_geo_df["complex_lat"].fillna(
        station_geo_df["complex_lat_lookup"]
    )

    station_geo_df["complex_lon"] = station_geo_df["complex_lon"].fillna(
        station_geo_df["complex_lon_lookup"]
    )

    station_geo_df = station_geo_df.drop(
        columns=[
            c for c in ["complex_lat_lookup", "complex_lon_lookup"]
            if c in station_geo_df.columns
        ]
    )


station_geo_df["complex_lat"] = pd.to_numeric(
    station_geo_df["complex_lat"],
    errors="coerce"
)

station_geo_df["complex_lon"] = pd.to_numeric(
    station_geo_df["complex_lon"],
    errors="coerce"
)

before_coord_drop = len(station_geo_df)

station_geo_df = station_geo_df.dropna(
    subset=[
        "station_complex_id",
        "complex_lat",
        "complex_lon",
    ]
).copy()

after_coord_drop = len(station_geo_df)

print("Station coordinate recovery")
print("-" * 60)
print(f"Station complexes before coordinate drop: {before_coord_drop:,}")
print(f"Station complexes after coordinate drop:  {after_coord_drop:,}")
print(f"Dropped due to missing coordinates:       {before_coord_drop - after_coord_drop:,}")

if station_geo_df.empty:
    raise RuntimeError(
        "No station-complex coordinates available. Cannot build spatial features."
    )


# -------------------------------------------------------
# Build station GeoDataFrame and buffers
# -------------------------------------------------------

station_gdf = gpd.GeoDataFrame(
    station_geo_df,
    geometry=gpd.points_from_xy(
        station_geo_df["complex_lon"],
        station_geo_df["complex_lat"],
    ),
    crs="EPSG:4326",
)

station_gdf = station_gdf.to_crs("EPSG:2263")

station_buffers = station_gdf.copy()
station_buffers["geometry"] = station_gdf.geometry.buffer(BUFFER_FT)

print(
    f"\nStation buffers created: {len(station_buffers):,} station complexes "
    f"@ {CATCHMENT_MILES}-mile radius"
)


# -------------------------------------------------------
# Load NYC Census block group geometries
# -------------------------------------------------------

if not tiger_cache.exists():
    print("\nDownloading TIGER block group file...")
    resp = requests.get(TIGER_URL, timeout=120)
    resp.raise_for_status()
    tiger_cache.write_bytes(resp.content)
    print(f"Saved TIGER file: {tiger_cache}")
else:
    print(f"\nLoaded TIGER file from cache: {tiger_cache}")

bg_gdf = gpd.read_file(f"zip://{tiger_cache}")

bg_gdf = bg_gdf[
    (bg_gdf["STATEFP"] == "36")
    & (bg_gdf["COUNTYFP"].isin(NYC_COUNTIES))
].copy()

bg_gdf["GEOID"] = bg_gdf["GEOID"].astype(str).str.zfill(12)

bg_gdf = bg_gdf.to_crs("EPSG:2263")

bg_gdf["bg_area"] = bg_gdf.geometry.area

print(f"NYC block groups loaded: {len(bg_gdf):,}")


# -------------------------------------------------------
# Load ACS and LODES attributes
# -------------------------------------------------------

if not ACS_FILE.exists():
    raise FileNotFoundError(
        f"Missing ACS file: {ACS_FILE}\n"
        "Run Section 1.3a first to create ACS block group data."
    )

if not LODES_FILE.exists():
    raise FileNotFoundError(
        f"Missing LODES file: {LODES_FILE}\n"
        "Run Section 1.3b first to create LODES block group data."
    )

acs_df = pd.read_csv(
    ACS_FILE,
    dtype={"GEOID": str},
)

lodes_df = pd.read_csv(
    LODES_FILE,
    dtype={"GEOID": str},
)

acs_df["GEOID"] = acs_df["GEOID"].astype(str).str.zfill(12)
lodes_df["GEOID"] = lodes_df["GEOID"].astype(str).str.zfill(12)

print("\nLoaded built-environment attributes:")
print(f"  ACS {ACS_FEATURE_YEAR}:   {len(acs_df):,} block groups")
print(f"  LODES {LODES_FEATURE_YEAR}: {len(lodes_df):,} block groups")


# -------------------------------------------------------
# Join ACS + LODES to block group geometries
# -------------------------------------------------------

bg_attr = bg_gdf.merge(
    acs_df,
    on="GEOID",
    how="left",
)

bg_attr = bg_attr.merge(
    lodes_df,
    on="GEOID",
    how="left",
    suffixes=("", "_lodes"),
)

# Fill missing numeric attributes with 0 for sums;
# median income remains NaN-aware during weighted average.
numeric_cols = bg_attr.select_dtypes(include=[np.number]).columns.tolist()

for col in numeric_cols:
    if col not in ["median_hh_income", "bg_area"]:
        bg_attr[col] = bg_attr[col].fillna(0)


# -------------------------------------------------------
# Spatial intersection: station buffers × block groups
# -------------------------------------------------------

print("\nIntersecting station buffers with block groups...")

intersections = gpd.overlay(
    station_buffers[
        [
            "station_complex_id",
            "complex_name",
            "complex_lat",
            "complex_lon",
            "geometry",
        ]
    ],
    bg_attr,
    how="intersection",
)

if intersections.empty:
    raise RuntimeError(
        "Spatial overlay produced no intersections. Check CRS and station coordinates."
    )

intersections["intersect_area"] = intersections.geometry.area

intersections["area_weight"] = (
    intersections["intersect_area"] /
    intersections["bg_area"].replace(0, np.nan)
)

intersections["area_weight"] = intersections["area_weight"].clip(
    lower=0,
    upper=1,
)

print(f"Intersections created: {len(intersections):,}")


# -------------------------------------------------------
# Identify ACS and LODES columns
# -------------------------------------------------------

acs_sum_cols = [
    c for c in [
        "total_population",
        "total_households",
        "no_vehicle_hh",
        "total_commuters",
        "transit_commuters",
    ]
    if c in intersections.columns
]

acs_weighted_mean_cols = [
    c for c in [
        "median_hh_income",
    ]
    if c in intersections.columns
]

lodes_sum_cols = [
    c for c in [
        "total_jobs",
        "finance_jobs",
        "professional_jobs",
        "retail_jobs",
        "healthcare_jobs",
        "food_service_jobs",
        "office_jobs",
        "essential_jobs",
    ]
    if c in intersections.columns
]

print("\nColumns available for spatial aggregation:")
print(f"  ACS sum columns:           {acs_sum_cols}")
print(f"  ACS weighted-mean columns: {acs_weighted_mean_cols}")
print(f"  LODES sum columns:         {lodes_sum_cols}")


# -------------------------------------------------------
# Aggregate area-weighted features to station buffers
# -------------------------------------------------------

feature_rows = []

for sid, group in intersections.groupby("station_complex_id"):
    row = {
        "station_complex_id": sid,
        "complex_name": group["complex_name"].iloc[0],
        "complex_lat": group["complex_lat"].iloc[0],
        "complex_lon": group["complex_lon"].iloc[0],
        "n_intersecting_block_groups": group["GEOID"].nunique(),
    }

    # Area-weighted ACS counts
    for col in acs_sum_cols:
        row[col] = (group[col] * group["area_weight"]).sum()

    # Area-weighted LODES job counts
    for col in lodes_sum_cols:
        row[col] = (group[col] * group["area_weight"]).sum()

    # Weighted mean / median-style attributes
    for col in acs_weighted_mean_cols:
        valid = group[col].notna() & group["area_weight"].notna()

        if valid.any() and group.loc[valid, "area_weight"].sum() > 0:
            row[col] = np.average(
                group.loc[valid, col],
                weights=group.loc[valid, "area_weight"],
            )
        else:
            row[col] = np.nan

    feature_rows.append(row)

station_features = pd.DataFrame(feature_rows)


# -------------------------------------------------------
# Derived built-environment features
# -------------------------------------------------------

def safe_divide(num, den):
    return np.where(
        den > 0,
        num / den,
        np.nan,
    )


if {"no_vehicle_hh", "total_households"}.issubset(station_features.columns):
    station_features["no_vehicle_share"] = safe_divide(
        station_features["no_vehicle_hh"],
        station_features["total_households"],
    )

if {"transit_commuters", "total_commuters"}.issubset(station_features.columns):
    station_features["transit_commute_share"] = safe_divide(
        station_features["transit_commuters"],
        station_features["total_commuters"],
    )

if {"office_jobs", "total_jobs"}.issubset(station_features.columns):
    station_features["office_job_share"] = safe_divide(
        station_features["office_jobs"],
        station_features["total_jobs"],
    )

if {"essential_jobs", "total_jobs"}.issubset(station_features.columns):
    station_features["essential_job_share"] = safe_divide(
        station_features["essential_jobs"],
        station_features["total_jobs"],
    )

if {"retail_jobs", "total_jobs"}.issubset(station_features.columns):
    station_features["retail_job_share"] = safe_divide(
        station_features["retail_jobs"],
        station_features["total_jobs"],
    )

# Density features per square mile of station buffer
buffer_area_sqmi = np.pi * (CATCHMENT_MILES ** 2)

if "total_population" in station_features.columns:
    station_features["population_density"] = (
        station_features["total_population"] / buffer_area_sqmi
    )

if "total_jobs" in station_features.columns:
    station_features["job_density"] = (
        station_features["total_jobs"] / buffer_area_sqmi
    )

if "office_jobs" in station_features.columns:
    station_features["office_job_density"] = (
        station_features["office_jobs"] / buffer_area_sqmi
    )

if "essential_jobs" in station_features.columns:
    station_features["essential_job_density"] = (
        station_features["essential_jobs"] / buffer_area_sqmi
    )


# -------------------------------------------------------
# Merge station features with 2024 cluster assignments
# -------------------------------------------------------

cluster_cols = [
    c for c in [
        "station_complex_id",
        "cluster",
        "cluster_label",
        "avg_weekday_entries",
    ]
    if c in cluster_assignments[2024].columns
]

station_features = station_features.merge(
    cluster_assignments[2024][cluster_cols].drop_duplicates(
        subset=["station_complex_id"]
    ),
    on="station_complex_id",
    how="left",
)


# -------------------------------------------------------
# Save and inspect output
# -------------------------------------------------------

station_features_out = OUT_DIR / "station_features.csv"

station_features.to_csv(station_features_out, index=False)

print("\nStation built-environment features saved:")
print(f"  File: {station_features_out}")
print(f"  Rows: {len(station_features):,}")
print(f"  Station complexes with cluster label: {station_features['cluster_label'].notna().sum():,}")

print("\nFeature columns:")
print(station_features.columns.tolist())

print("\nPreview:")
display(station_features.head())

### 5.1b ACS QA Check

Quick quality-assurance check on the ACS data before proceeding to feature engineering. Confirms that household and commuter variables are present and non-zero.


In [ ]:
# ==============================================================
# Section 5.1b — ACS QA Check
#
# Purpose:
#   Quick validation of ACS block-group data before spatial join.
#   Confirms expected columns and non-zero coverage.
#   Run this immediately after Section 5.1 to catch issues early.
# ==============================================================

if 'OUT_DIR' not in globals():
    OUT_DIR = Path("/content/drive/MyDrive/dow_ridership_paper/outputs")

acs_check_path = OUT_DIR / "acs_bg_2022.csv"

if not acs_check_path.exists():
    print("acs_bg_2022.csv not found — run Section 1.3a first.")
else:
    acs_check = pd.read_csv(acs_check_path, dtype={"GEOID": str})

    print("ACS 2022 columns:")
    print(acs_check.columns.tolist())

    check_cols = [
        "total_population", "total_households", "no_vehicle_hh",
        "total_commuters",  "transit_commuters","median_hh_income",
    ]

    print("\nACS 2022 descriptive summary:")
    avail = [c for c in check_cols if c in acs_check.columns]
    print(acs_check[avail].describe().round(2).to_string())

    print("\nNon-zero counts:")
    for c in check_cols:
        if c in acs_check.columns:
            print(f"  {c}: {(acs_check[c] > 0).sum():,} / {len(acs_check):,}")


### 5.2 Feature Engineering

**Purpose:** Transform raw spatial join outputs into modeling-ready variables. This section now incorporates the full set of trip-purpose virtualization predictors developed in Sections 1.3b–1.3d.

**Variable groups:**

**Group 1 — Core land-use features (original)**
Share variables and log-transformed densities for office, retail, and essential-service jobs; transit commute share; no-vehicle household share; population and job density.

**Group 2 — Commuter rail access (new)**
- `dist_to_nearest_terminal_miles` — Haversine distance to nearest of five major multimodal terminals. Stations close to Grand Central or Penn Station absorb the Friday suburban commuter drain documented in MTA's commuter rail data.
- `is_terminal_adjacent` — binary flag: within 0.25-mile walking-transfer zone.
- `log_dist_to_nearest_terminal` — log-transformed for regression linearity.

**Group 3 — WAC/RAC destination-vs-origin ratio (new)**
- `wac_rac_ratio` — ratio of jobs (workers arriving) to resident workers (workers departing). High = destination station. Low = origin/residential station. This directly encodes the commuter flow directionality that explains why outer-borough residential and Midtown office-core stations have fundamentally different demand regime shapes.
- `pct_high_earn_residents` — share of high-earning resident workers near the station, a proxy for the telework-eligible residential population.

**Group 4 — Trip-purpose virtualization exposure (new)**
Three sectors with the highest virtual substitutability of in-person trip purposes *beyond* office commuting:
- `education_job_share` / `log_education_jobs` — university and school job concentration. Post-COVID hybrid course schedules suppress Mon/Fri trips at campus-adjacent stations (Hunter, NYU, Columbia, Baruch, Fordham). The DOW signature mirrors hybrid office work but has a distinct seasonal rhythm (absent in summer).
- `healthcare_job_share` / `log_healthcare_job_density` — medical facility concentration. Telehealth has substituted for routine appointments, suppressing mid-day weekday ridership at stations serving the East Side medical corridor, NYU Langone, and Montefiore. Research confirms telehealth served as a substitute for in-person visits with total outpatient volume declining 14%+ post-pandemic.
- `govt_job_share` — public administration concentration. Online service portals and remote court hearings have reduced trips to civic-center stations (Borough Hall, Chambers Street, 161st Street Bronx court complex).
- `virtualization_exposure_index` — composite sum of the three shares above. Summarizes a station's total exposure to trip-purpose virtualization beyond the office commute.

**Log transformations:** Applied via `log1p()` to all right-skewed count variables. This reduces the leverage of extreme outliers at major hubs (Grand Central, Times Square) on the regression coefficients.

**Backward compatibility:** Column aliases ensure Section 5.3 feature names resolve regardless of whether spatial join columns arrive from 5.1 with alternate naming.


In [ ]:
# ==============================================================
# Section 5.2 — Feature Assembly and Engineering
#
# Purpose:
#   Loads station-level built-environment features from Section 5.1
#   and creates modeling-ready variables including the extended
#   trip-purpose virtualization predictor set.
#
# Inputs:
#   station_features.csv
#   lodes_rac_{year}.csv       (resident worker counts)
#   station_complexes_with_proximity.csv  (terminal distances)
#
# Feature groups:
#
# GROUP 1 — Core land-use (original)
#   pct_no_vehicle           — no-vehicle household share
#   pct_transit_commute      — transit commute share
#   office_job_share         — finance + professional / total jobs
#   retail_job_share         — retail / total jobs
#   essential_job_share      — healthcare + food service / total jobs
#   log_office_jobs          — log1p(office_jobs)
#   log_total_jobs           — log1p(total_jobs)
#   log_pop_density          — log1p(population per sq mile)
#   log_median_income        — log1p(median_hh_income)
#
# GROUP 2 — Commuter rail access (new)
#   dist_to_nearest_terminal_miles  — Haversine distance to nearest
#                                     of 5 major multimodal terminals
#   is_terminal_adjacent            — within 0.25-mile walking zone
#   log_dist_to_nearest_terminal    — log-transformed distance
#
# GROUP 3 — WAC/RAC destination-vs-origin ratio (new)
#   wac_rac_ratio            — jobs (arrivals) / resident workers (departures)
#   log_wac_rac_ratio        — log-transformed ratio
#   pct_high_earn_residents  — high-earner share of resident workers
#                              (proxy for telework-eligible residents)
#
# GROUP 4 — Trip-purpose virtualization exposure (new)
#   education_job_share      — education jobs / total jobs
#   log_education_jobs       — log1p(education_jobs)
#   healthcare_job_share     — healthcare / total jobs
#   log_healthcare_job_density — log1p(healthcare jobs per sq mile)
#   govt_job_share           — public admin / total jobs
#   virtualization_exposure_index — education + healthcare + govt shares
#
# Outputs:
#   station_features_engineered.csv
# ==============================================================

import pandas as pd
import numpy as np


# -------------------------------------------------------
# Load station features from Section 5.1
# -------------------------------------------------------

features_cache = OUT_DIR / "station_features.csv"

if not features_cache.exists():
    raise FileNotFoundError(
        f"Missing file: {features_cache}\n"
        "Run Section 5.1 first to create station_features.csv."
    )

station_features = pd.read_csv(features_cache)

print("Station features loaded.")
print(f"  File: {features_cache}")
print(f"  Rows: {len(station_features):,}")
print(f"  Columns: {len(station_features.columns):,}")


# -------------------------------------------------------
# Basic validation
# -------------------------------------------------------

required_id_cols = [
    "station_complex_id",
    "complex_name",
]

missing_id_cols = [
    c for c in required_id_cols
    if c not in station_features.columns
]

if missing_id_cols:
    raise RuntimeError(
        f"station_features is missing required ID columns: {missing_id_cols}\n"
        f"Columns found: {list(station_features.columns)}"
    )


# -------------------------------------------------------
# Ensure numeric columns are numeric
# -------------------------------------------------------

numeric_candidate_cols = [
    "total_population",
    "total_households",
    "no_vehicle_hh",
    "total_commuters",
    "transit_commuters",
    "median_hh_income",
    "total_jobs",
    "finance_jobs",
    "professional_jobs",
    "retail_jobs",
    "healthcare_jobs",
    "food_service_jobs",
    "office_jobs",
    "essential_jobs",
    "population_density",
    "job_density",
    "office_job_density",
    "essential_job_density",
    "retail_job_density",
    "avg_weekday_entries",
]

for col in numeric_candidate_cols:
    if col in station_features.columns:
        station_features[col] = pd.to_numeric(
            station_features[col],
            errors="coerce"
        )


# -------------------------------------------------------
# Helper for safe division
# -------------------------------------------------------

def safe_divide(num, den):
    """
    Vectorized safe division. Returns NaN when denominator is zero or missing.
    """
    num = pd.to_numeric(num, errors="coerce")
    den = pd.to_numeric(den, errors="coerce")

    return np.where(
        den > 0,
        num / den,
        np.nan,
    )


# -------------------------------------------------------
# Harmonize feature names
# -------------------------------------------------------
# Section 5.1 may have created no_vehicle_share and
# transit_commute_share. For modeling, also create pct_* aliases.

if {"no_vehicle_hh", "total_households"}.issubset(station_features.columns):
    station_features["pct_no_vehicle"] = safe_divide(
        station_features["no_vehicle_hh"],
        station_features["total_households"],
    )

elif "no_vehicle_share" in station_features.columns:
    station_features["pct_no_vehicle"] = station_features["no_vehicle_share"]


if {"transit_commuters", "total_commuters"}.issubset(station_features.columns):
    station_features["pct_transit_commute"] = safe_divide(
        station_features["transit_commuters"],
        station_features["total_commuters"],
    )

elif "transit_commute_share" in station_features.columns:
    station_features["pct_transit_commute"] = station_features["transit_commute_share"]


# -------------------------------------------------------
# Job share features
# -------------------------------------------------------

if {"office_jobs", "total_jobs"}.issubset(station_features.columns):
    station_features["office_job_share"] = safe_divide(
        station_features["office_jobs"],
        station_features["total_jobs"],
    )

if {"retail_jobs", "total_jobs"}.issubset(station_features.columns):
    station_features["retail_job_share"] = safe_divide(
        station_features["retail_jobs"],
        station_features["total_jobs"],
    )

if {"essential_jobs", "total_jobs"}.issubset(station_features.columns):
    station_features["essential_job_share"] = safe_divide(
        station_features["essential_jobs"],
        station_features["total_jobs"],
    )


# -------------------------------------------------------
# Density features
# -------------------------------------------------------
# Section 5.1 creates densities per square mile.
# If missing, recompute from CATCHMENT_MILES.

if "CATCHMENT_MILES" not in globals():
    CATCHMENT_MILES = 0.5

buffer_area_sqmi = np.pi * (CATCHMENT_MILES ** 2)

if "population_density" not in station_features.columns:
    if "total_population" in station_features.columns:
        station_features["population_density"] = (
            station_features["total_population"] / buffer_area_sqmi
        )

if "job_density" not in station_features.columns:
    if "total_jobs" in station_features.columns:
        station_features["job_density"] = (
            station_features["total_jobs"] / buffer_area_sqmi
        )

if "office_job_density" not in station_features.columns:
    if "office_jobs" in station_features.columns:
        station_features["office_job_density"] = (
            station_features["office_jobs"] / buffer_area_sqmi
        )

if "retail_job_density" not in station_features.columns:
    if "retail_jobs" in station_features.columns:
        station_features["retail_job_density"] = (
            station_features["retail_jobs"] / buffer_area_sqmi
        )

if "essential_job_density" not in station_features.columns:
    if "essential_jobs" in station_features.columns:
        station_features["essential_job_density"] = (
            station_features["essential_jobs"] / buffer_area_sqmi
        )


# -------------------------------------------------------
# Log transformations for skewed variables
# -------------------------------------------------------

log_source_cols = [
    "office_jobs",
    "total_jobs",
    "retail_jobs",
    "essential_jobs",
    "population_density",
    "job_density",
    "office_job_density",
    "retail_job_density",
    "essential_job_density",
    "median_hh_income",
]

for col in log_source_cols:
    if col in station_features.columns:
        safe_values = station_features[col].clip(lower=0)
        station_features[f"log_{col}"] = np.log1p(safe_values)


# Backward-compatible aliases used in older notebook sections
if "log_population_density" in station_features.columns:
    station_features["log_pop_density"] = station_features["log_population_density"]
elif "population_density" in station_features.columns:
    station_features["log_pop_density"] = np.log1p(
        station_features["population_density"].clip(lower=0)
    )

if "log_median_hh_income" in station_features.columns:
    station_features["log_median_income"] = station_features["log_median_hh_income"]
elif "median_hh_income" in station_features.columns:
    station_features["log_median_income"] = np.log1p(
        station_features["median_hh_income"].clip(lower=0)
    )


# -------------------------------------------------------
# Clean impossible or extreme share values
# -------------------------------------------------------

share_cols = [
    "pct_no_vehicle",
    "pct_transit_commute",
    "office_job_share",
    "retail_job_share",
    "essential_job_share",
    "no_vehicle_share",
    "transit_commute_share",
]

for col in share_cols:
    if col in station_features.columns:
        station_features[col] = pd.to_numeric(
            station_features[col],
            errors="coerce"
        )
        station_features[col] = station_features[col].clip(lower=0, upper=1)


# -------------------------------------------------------
# Add cluster labels if missing
# -------------------------------------------------------

if "cluster_label" not in station_features.columns:
    if "cluster_assignments" in globals() and 2024 in cluster_assignments:
        cluster_cols = [
            c for c in [
                "station_complex_id",
                "cluster",
                "cluster_label",
                "avg_weekday_entries",
            ]
            if c in cluster_assignments[2024].columns
        ]

        station_features = station_features.merge(
            cluster_assignments[2024][cluster_cols].drop_duplicates(
                subset=["station_complex_id"]
            ),
            on="station_complex_id",
            how="left",
        )




# -------------------------------------------------------
# GROUP 2: Commuter Rail Terminal Proximity
# -------------------------------------------------------
# Merge proximity features from Section 1.3d if available.
# These capture the suburban commuter drain mechanism:
# stations near transfer terminals absorb Friday depression
# when Metro-North/LIRR commuter flows decline.

prox_cache = OUT_DIR / "station_complexes_with_proximity.csv"

if prox_cache.exists():
    prox_df = pd.read_csv(prox_cache)
    prox_df.columns = (
        prox_df.columns.str.strip().str.lower()
        .str.replace(r"[\s/]+", "_", regex=True)
    )

    # Identify the complex_id join key in the proximity file
    prox_id_col = next(
        (c for c in ["complex_id","station_complex_id","stop_id"]
         if c in prox_df.columns), None
    )

    proximity_cols = [c for c in [
        "dist_to_nearest_terminal_miles",
        "nearest_terminal_name",
        "is_terminal_adjacent",
        "log_dist_to_nearest_terminal",
    ] if c in prox_df.columns]

    if prox_id_col and proximity_cols:
        station_features = station_features.merge(
            prox_df[[prox_id_col] + proximity_cols]
            .drop_duplicates(subset=[prox_id_col]),
            left_on="station_complex_id",
            right_on=prox_id_col,
            how="left",
        )
        if prox_id_col != "station_complex_id":
            station_features = station_features.drop(columns=[prox_id_col], errors="ignore")

        n_matched = station_features["dist_to_nearest_terminal_miles"].notna().sum()
        print(f"Terminal proximity merged: {n_matched:,} / {len(station_features):,} stations")
    else:
        print("Warning: proximity columns not found in cache file — check Section 1.3d output")
else:
    # Compute on-the-fly if cache missing but station coords available
    print("Proximity cache not found — computing inline...")
    if all(c in station_features.columns for c in ["complex_lat","complex_lon"]):
        COMMUTER_TERMINALS = {
            "Grand_Central":          (40.7527, -73.9772),
            "Penn_Station":           (40.7506, -73.9971),
            "Atlantic_Terminal_LIRR": (40.6841, -73.9766),
            "Jamaica_Station":        (40.7018, -73.8088),
            "Port_Authority_Bus":     (40.7572, -73.9901),
        }
        def _hav(lat1, lon1, lat2, lon2):
            R = 3958.8
            dlat, dlon = np.radians(lat2-lat1), np.radians(lon2-lon1)
            a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1))*np.cos(np.radians(lat2))*np.sin(dlon/2)**2
            return R * 2 * np.arcsin(np.sqrt(a))

        dists, names = [], []
        for _, row in station_features.iterrows():
            d = {n: _hav(row.complex_lat, row.complex_lon, t[0], t[1])
                 for n, t in COMMUTER_TERMINALS.items()}
            nearest = min(d, key=d.get)
            dists.append(d[nearest])
            names.append(nearest)

        station_features["dist_to_nearest_terminal_miles"] = dists
        station_features["nearest_terminal_name"]          = names
        station_features["is_terminal_adjacent"]           = (
            station_features["dist_to_nearest_terminal_miles"] <= 0.25
        ).astype(int)
        station_features["log_dist_to_nearest_terminal"]  = np.log1p(
            station_features["dist_to_nearest_terminal_miles"]
        )
        print(f"Terminal proximity computed inline: "
              f"{station_features['is_terminal_adjacent'].sum()} adjacent stations")
    else:
        print("complex_lat/complex_lon not available — skipping terminal proximity.")


# -------------------------------------------------------
# GROUP 3: WAC/RAC Destination-vs-Origin Ratio
# -------------------------------------------------------
# Merge resident worker counts from LODES RAC.
# WAC = workers arriving (jobs at location)
# RAC = resident workers departing (workers living at location)
# Ratio >> 1 → destination/office-core
# Ratio << 1 → origin/residential
# Ratio ≈ 1  → mixed-use or transfer hub

# Use the post-COVID LODES year (2021 WAC → 2021 RAC)
rac_year = 2021
rac_cache = OUT_DIR / f"lodes_rac_{rac_year}.csv"

if rac_cache.exists():
    rac_bg = pd.read_csv(rac_cache, dtype={"GEOID": str})

    # The spatial join in Section 5.1 should have aggregated RAC
    # to station buffers. If not, do a simple block-group lookup here.
    if "total_resident_workers" not in station_features.columns:
        # Approximate: sum RAC within same block groups intersected in 5.1
        # This requires the bg_intersection table from 5.1 in memory.
        # If not available, flag for re-run of 5.1 with RAC included.
        if "bg_intersection" in globals() and bg_intersection is not None:
            rac_agg = (
                bg_intersection
                .merge(rac_bg[["GEOID","total_resident_workers",
                               "pct_high_earn_residents"]],
                       on="GEOID", how="left")
                .groupby("station_complex_id")
                .agg(
                    total_resident_workers=("total_resident_workers","sum"),
                    pct_high_earn_residents=("pct_high_earn_residents","mean"),
                )
                .reset_index()
            )
            station_features = station_features.merge(
                rac_agg, on="station_complex_id", how="left"
            )
            print(f"RAC merged from bg_intersection: "
                  f"{station_features['total_resident_workers'].notna().sum():,} stations")
        else:
            print("bg_intersection not in memory — re-run Section 5.1 to include RAC.")
            print("WAC/RAC ratio features will be skipped for this run.")
            station_features["total_resident_workers"] = np.nan
            station_features["pct_high_earn_residents"] = np.nan

    # Compute WAC/RAC ratio regardless of source
    if "total_resident_workers" in station_features.columns:
        station_features["wac_rac_ratio"] = (
            station_features["total_jobs"]
            / station_features["total_resident_workers"].replace(0, np.nan)
        ).clip(upper=500)   # cap extreme outliers (e.g. Times Square)

        station_features["log_wac_rac_ratio"] = np.log1p(
            station_features["wac_rac_ratio"].fillna(0)
        )

        n_valid = station_features["wac_rac_ratio"].notna().sum()
        print(f"WAC/RAC ratio computed: {n_valid:,} stations")
        print(f"  Top 5 destination stations (highest ratio):")
        top5 = (station_features[["complex_name","wac_rac_ratio"]]
                .dropna()
                .nlargest(5, "wac_rac_ratio"))
        print(top5.to_string(index=False))
        print(f"  Top 5 origin stations (lowest ratio):")
        bot5 = (station_features[["complex_name","wac_rac_ratio"]]
                .dropna()
                .nsmallest(5, "wac_rac_ratio"))
        print(bot5.to_string(index=False))
else:
    print(f"LODES RAC cache not found: {rac_cache}")
    print("Run Section 1.3b (RAC download section) then re-run this cell.")
    station_features["wac_rac_ratio"]       = np.nan
    station_features["log_wac_rac_ratio"]   = np.nan
    station_features["pct_high_earn_residents"] = np.nan


# -------------------------------------------------------
# GROUP 4: Trip-Purpose Virtualization Exposure
# -------------------------------------------------------
# Three sectors with high virtual substitutability of in-person
# trip purposes beyond the office commute:
#
# EDUCATION (CNS13):
#   NYC has ~280,000 CUNY students, majority commuting by subway.
#   Hybrid course scheduling post-COVID concentrates in-person
#   days Tue–Thu, suppressing Mon/Fri trips at campus stations.
#   Seasonal: absent in summer, concentrated in fall/spring.
#   Key affected stations: 68th St–Hunter, 23rd St (Baruch),
#   8th St–NYU, 116th St–Columbia, Fordham Rd, Flatbush Av–Brooklyn College.
#
# HEALTHCARE (CNS15):
#   Telehealth adoption substituted for in-person visits across
#   primary care, mental health, and chronic disease management.
#   Medicare data shows 14%+ decline in outpatient visit volume.
#   Mid-day weekday suppression at medical corridor stations:
#   68th–96th Sts (East Side), 168th St (NewYork-Presbyterian).
#
# PUBLIC ADMINISTRATION (CNS20):
#   Online service portals, remote court hearings, and virtual
#   government appointments have reduced trips to civic centers.
#   NYC Housing Court shifted many appearances to remote formats.
#   Affected stations: Borough Hall, Chambers St, 161st St–Bronx.

# Education sector
if "education_jobs" in station_features.columns:
    station_features["education_job_share"] = (
        station_features["education_jobs"]
        / station_features["total_jobs"].replace(0, np.nan)
    ).fillna(0).clip(lower=0, upper=1)

    station_features["log_education_jobs"] = np.log1p(
        station_features["education_jobs"].fillna(0)
    )

    station_features["log_education_job_density"] = np.log1p(
        station_features["education_jobs"].fillna(0)
        / station_features.get("buffer_area_sqmi", pd.Series(1, index=station_features.index))
    )
    print(f"Education features built: "
          f"mean share={station_features['education_job_share'].mean():.3f}")
else:
    print("education_jobs not found — re-run Section 1.3b with updated LODES_VARS")
    for col in ["education_job_share","log_education_jobs","log_education_job_density"]:
        station_features[col] = np.nan

# Healthcare density (extends existing healthcare_job_share)
if "healthcare_jobs" in station_features.columns:
    if "healthcare_job_share" not in station_features.columns:
        station_features["healthcare_job_share"] = (
            station_features["healthcare_jobs"]
            / station_features["total_jobs"].replace(0, np.nan)
        ).fillna(0).clip(lower=0, upper=1)

    station_features["log_healthcare_job_density"] = np.log1p(
        station_features["healthcare_jobs"].fillna(0)
        / station_features.get("buffer_area_sqmi", pd.Series(1, index=station_features.index))
    )
    print(f"Healthcare density feature built: "
          f"mean share={station_features['healthcare_job_share'].mean():.3f}")
else:
    print("healthcare_jobs not found in station_features.")
    for col in ["healthcare_job_share","log_healthcare_job_density"]:
        station_features[col] = np.nan

# Government / public administration sector
if "public_admin_jobs" in station_features.columns:
    station_features["govt_job_share"] = (
        station_features["public_admin_jobs"]
        / station_features["total_jobs"].replace(0, np.nan)
    ).fillna(0).clip(lower=0, upper=1)

    station_features["log_govt_jobs"] = np.log1p(
        station_features["public_admin_jobs"].fillna(0)
    )
    print(f"Government sector features built: "
          f"mean share={station_features['govt_job_share'].mean():.3f}")
else:
    print("public_admin_jobs not found — re-run Section 1.3b with updated LODES_VARS")
    for col in ["govt_job_share","log_govt_jobs"]:
        station_features[col] = np.nan

# Composite virtualization exposure index
virt_cols = [c for c in [
    "education_job_share",
    "healthcare_job_share",
    "govt_job_share",
] if c in station_features.columns and station_features[c].notna().any()]

if virt_cols:
    station_features["virtualization_exposure_index"] = (
        station_features[virt_cols].fillna(0).sum(axis=1)
    )
    print(f"\nVirtualization exposure index: "
          f"built from {virt_cols}")
    print(f"  Mean: {station_features['virtualization_exposure_index'].mean():.3f}")
    print(f"  Top 5 stations by virtualization exposure:")
    top_virt = (station_features[["complex_name","virtualization_exposure_index"]]
                .dropna()
                .nlargest(5, "virtualization_exposure_index"))
    print(top_virt.to_string(index=False))
else:
    station_features["virtualization_exposure_index"] = np.nan
    print("Virtualization exposure index: component columns missing")


# -------------------------------------------------------
# Modeling feature list — extended with new predictors
# -------------------------------------------------------
# Features are organized by group. All groups are included
# as candidates; Section 5.3 selects on availability.
#
# NOTE: New features (Groups 2–4) are listed after core
# features. If any are missing due to data unavailability,
# Section 5.3 silently drops them and notes what was used.

MODEL_FEATURES = [
    # Group 1 — Core land-use (original)
    "pct_no_vehicle",
    "pct_transit_commute",
    "office_job_share",
    "retail_job_share",
    "essential_job_share",
    "log_office_jobs",
    "log_total_jobs",
    "log_retail_jobs",
    "log_essential_jobs",
    "log_pop_density",
    "log_median_income",
    "log_job_density",
    "log_office_job_density",
    "log_retail_job_density",
    "log_essential_job_density",
    # Group 2 — Commuter rail access
    "log_dist_to_nearest_terminal",
    "is_terminal_adjacent",
    # Group 3 — WAC/RAC destination-vs-origin
    "log_wac_rac_ratio",
    "pct_high_earn_residents",
    # Group 4 — Trip-purpose virtualization
    "education_job_share",
    "log_education_jobs",
    "healthcare_job_share",
    "log_healthcare_job_density",
    "govt_job_share",
    "virtualization_exposure_index",
]

available_model_features = [
    c for c in MODEL_FEATURES
    if c in station_features.columns
]

missing_model_features = [
    c for c in MODEL_FEATURES
    if c not in station_features.columns
]


# -------------------------------------------------------
# QA summary
# -------------------------------------------------------

print("\n================ FEATURE ASSEMBLY QA ================\n")

print("Available modeling features:")
for c in available_model_features:
    print(f"  {c}")

if missing_model_features:
    print("\nMissing optional modeling features:")
    for c in missing_model_features:
        print(f"  {c}")

print("\nMissing values in available modeling features:")
print(
    station_features[available_model_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nDescriptive statistics:")
display(
    station_features[available_model_features]
    .describe()
    .T
    .round(3)
)


# -------------------------------------------------------
# Save engineered feature table
# -------------------------------------------------------

features_engineered_out = OUT_DIR / "station_features_engineered.csv"

station_features.to_csv(features_engineered_out, index=False)

print("\nStation engineered features saved:")
print(f"  File: {features_engineered_out}")
print(f"  Rows: {len(station_features):,}")
print(f"  Columns: {len(station_features.columns):,}")

print("\nPreview:")
display(station_features.head())

### 5.3 Regression: Multinomial Logistic with Explicit Reference Category

**Model:** Multinomial logistic regression (statsmodels MNLogit). Dependent variable: 2024 cluster assignment (4 categories). Reference category: `"Major Midtown / Regional Core"` — the traditional CBD commuter pattern provides the most theoretically meaningful comparison baseline.

**Extended feature set (this version):**

*Group 1 — Core land-use:* Log-transformed job counts and densities, demographic shares, transit commute share. These are the standard built-environment predictors in transit demand modeling.

*Group 2 — Commuter rail access:* Log-distance to nearest commuter rail terminal, terminal-adjacency flag. These capture the suburban commuter drain documented in MTA annual reports — Metro-North commutation 42% below 1990 levels, Friday trains running near-empty.

*Group 3 — WAC/RAC destination-vs-origin:* The ratio of jobs (workers arriving) to resident workers (workers departing) encodes flow directionality that built-environment variables alone cannot capture. A station surrounded by offices but near no homes is structurally different from a station surrounded by homes but near no offices, even if their job counts are similar.

*Group 4 — Trip-purpose virtualization:* Education, healthcare, and public administration job shares capture the portion of 2024 ridership suppression that is *not* attributable to hybrid office work — university hybrid scheduling, telehealth substitution, and government service virtualization. Including these tests whether the demand regime structure is better explained by a broad activity-virtualization framework than by the office-commute hypothesis alone.

**Feature selection:** All candidate features are listed; unavailable features are silently dropped with a summary printed at runtime.

**Validation:** 5-fold stratified cross-validation (sklearn) for out-of-sample accuracy assessment.

**Output:** `table4_regression_results.csv`, `regression_dataset_2024.csv`, `regression_cv_results.csv`


In [ ]:
# ==============================================================
# Section 5.3 — Regression Model: Multinomial Logistic
#               with Explicit Reference Category
#
# Purpose:
#   Explains 2024 cluster membership using built-environment,
#   commuter rail access, WAC/RAC destination-vs-origin, and
#   trip-purpose virtualization features via multinomial logistic
#   regression.
#
# Reference category:
#   "Major Midtown / Regional Core" — the traditional CBD commuter
#   pattern provides the most theoretically meaningful baseline.
#   Change REFERENCE_LABEL below to use a different reference.
#
# Dependent variable:
#   2024 cluster assignment (recoded: reference = 0, others = 1..K-1)
#
# Estimation strategy — TWO TRACKS:
#
#   TRACK A — Full 21-feature L2-regularized sklearn logit
#     Purpose: CV accuracy, permutation importance, coefficient
#              signs across all feature groups. Always converges.
#     Table:   Supplementary table or paper appendix.
#
#   TRACK B — Parsimonious 8-feature unregularized statsmodels MNLogit
#     Purpose: Proper SEs and p-values for Table 4 in the paper.
#     Rationale: Hybrid Office Core has n=32 observations.
#              Rule of thumb: min class n >= 10-20 × features.
#              32 obs / 3 equations = ~10 obs per equation.
#              8 features keeps parameter count within range for
#              stable covariance matrix estimation.
#     Features: Selected by domain priority — new structural
#              variables (commuter rail, WAC/RAC) given priority
#              over land-use proxies they directly instantiate.
#
# Key objects produced (required by Section 5.4):
#   result               — fitted MNLogit object (Track B)
#   reg_df               — regression dataset with cluster labels
#   regression_results   — tidy coefficient DataFrame with
#                          reference_label and outcome_label cols
#   FEATURE_COLS         — parsimonious feature list (Track B)
#   X_raw                — unstandardized feature matrix (Track B)
#   y_reg                — recoded outcome (reference = 0)
#   MODEL_CODE_LABELS    — dict: model_code → cluster_label
#   model_code_to_cluster — dict: model_code → original cluster int
#   coef_df_a            — Track A full coefficient DataFrame
#   perm_df              — permutation importance DataFrame
#
# Outputs:
#   table4_regression_results.csv         (Track B — for paper)
#   regression_dataset_2024.csv
#   regression_outcome_reference_mapping.csv
#   regression_cv_results.csv             (Track A — full model CV)
#   regression_tracka_coefficients.csv    (Track A — all 21 features)
#   regression_permutation_importance.csv (Track A — feature ranking)
# ==============================================================

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import statsmodels.api as sm

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.inspection import permutation_importance


# -------------------------------------------------------
# Configuration
# -------------------------------------------------------

# Reference category — cluster against which all others are compared.
# "Major Midtown / Regional Core" = traditional CBD commuter pattern.
REFERENCE_LABEL = "Major Midtown / Regional Core"

# Track B parsimonious feature set — 8 features max for stable SEs.
# Order reflects domain priority: new structural variables first,
# then core land-use controls.
PARSIMONIOUS_FEATURE_CANDIDATES = [
    "log_dist_to_nearest_terminal",   # Group 2: commuter rail mechanism
    "log_wac_rac_ratio",              # Group 3: destination vs origin
    "log_office_jobs",                # Group 1: office land-use
    "pct_transit_commute",            # Group 1: transit dependence
    "virtualization_exposure_index",  # Group 4: composite virtualization
    "log_pop_density",                # Group 1: density control
    "pct_no_vehicle",                 # Group 1: car-free households
    "log_median_income",              # Group 1: income control
]

# Significance stars helper (used in both tracks)
def p_stars(p):
    if pd.isna(p):  return ""
    if p < 0.001:   return "***"
    if p < 0.01:    return "**"
    if p < 0.05:    return "*"
    if p < 0.10:    return "†"
    return ""


# -------------------------------------------------------
# Session recovery
# -------------------------------------------------------

if "OUT_DIR" not in globals():
    OUT_DIR = Path("/content/drive/MyDrive/dow_ridership_paper/outputs")
    print(f"OUT_DIR recovered: {OUT_DIR}")

if "KMEANS_SEED" not in globals():
    KMEANS_SEED = 42


# -------------------------------------------------------
# Load engineered station features
# -------------------------------------------------------

features_engineered_path = OUT_DIR / "station_features_engineered.csv"

if not features_engineered_path.exists():
    raise FileNotFoundError(
        f"Missing file: {features_engineered_path}\n"
        "Run Section 5.2 first."
    )

station_features = pd.read_csv(features_engineered_path)
print(f"Loaded engineered station features: {len(station_features):,} rows")

# Guard: drop entirely-NaN columns so they don't silently wipe rows
entirely_nan = [c for c in station_features.columns
                if station_features[c].isna().all()]
if entirely_nan:
    print(f"Dropping {len(entirely_nan)} entirely-NaN columns: {entirely_nan}")
    station_features = station_features.drop(columns=entirely_nan)


# -------------------------------------------------------
# Validate cluster assignments
# -------------------------------------------------------

if "cluster_assignments" not in globals():
    raise RuntimeError(
        "cluster_assignments not found. Run Sections 4.2 and 4.3 first."
    )

if 2024 not in cluster_assignments:
    raise RuntimeError(
        f"cluster_assignments[2024] not found. "
        f"Available keys: {list(cluster_assignments.keys())}"
    )

required_cluster_cols = [
    "station_complex_id", "complex_name", "cluster", "cluster_label",
]
missing_cluster_cols = [c for c in required_cluster_cols
                        if c not in cluster_assignments[2024].columns]
if missing_cluster_cols:
    raise RuntimeError(
        f"cluster_assignments[2024] missing columns: {missing_cluster_cols}\n"
        "Run Section 4.3 to add cluster labels."
    )


# -------------------------------------------------------
# Assemble regression dataset
# -------------------------------------------------------

cluster_2024 = (
    cluster_assignments[2024][required_cluster_cols]
    .drop_duplicates(subset=["station_complex_id"])
    .copy()
)

reg_df = cluster_2024.merge(
    station_features,
    on="station_complex_id",
    how="inner",
    suffixes=("", "_feature"),
)

# Remove duplicate columns introduced by merge suffix
dup_cols = [c for c in ["complex_name_feature",
                         "cluster_feature",
                         "cluster_label_feature"]
            if c in reg_df.columns]
if dup_cols:
    reg_df = reg_df.drop(columns=dup_cols)

print(f"\nRegression dataset: {len(reg_df):,} rows")
print("\nCluster distribution:")
print(
    reg_df[["cluster", "cluster_label"]]
    .drop_duplicates().sort_values("cluster").to_string(index=False)
)
print("\nCluster counts:")
print(reg_df["cluster_label"].value_counts())


# -------------------------------------------------------
# Identify reference cluster
# -------------------------------------------------------

label_to_cluster = (
    reg_df[["cluster", "cluster_label"]]
    .drop_duplicates()
    .set_index("cluster_label")["cluster"]
    .to_dict()
)

if REFERENCE_LABEL not in label_to_cluster:
    raise RuntimeError(
        f"REFERENCE_LABEL not found: '{REFERENCE_LABEL}'\n"
        f"Available labels: {list(label_to_cluster.keys())}"
    )

REFERENCE_CLUSTER = int(label_to_cluster[REFERENCE_LABEL])
print(f"\nReference category: {REFERENCE_LABEL} (cluster {REFERENCE_CLUSTER})")


# -------------------------------------------------------
# Build outcome recoding (reference → 0)
# -------------------------------------------------------
# statsmodels MNLogit treats the lowest-coded category as reference.
# Recode so selected reference = 0 regardless of original numbering.

remaining_clusters = sorted(
    c for c in reg_df["cluster"].unique()
    if int(c) != REFERENCE_CLUSTER
)
cluster_to_model_code = {REFERENCE_CLUSTER: 0}
for i, c in enumerate(remaining_clusters, start=1):
    cluster_to_model_code[int(c)] = i

model_code_to_cluster = {v: k for k, v in cluster_to_model_code.items()}

cluster_to_label = (
    reg_df[["cluster", "cluster_label"]]
    .drop_duplicates()
    .set_index("cluster")["cluster_label"]
    .to_dict()
)
MODEL_CODE_LABELS = {
    code: cluster_to_label[orig]
    for code, orig in model_code_to_cluster.items()
}

reg_df["cluster_model_code"] = reg_df["cluster"].map(cluster_to_model_code)

print("\nOutcome recoding:")
for code in sorted(MODEL_CODE_LABELS):
    ref = " ← reference" if code == 0 else ""
    print(f"  Code {code}: cluster {model_code_to_cluster[code]} — "
          f"{MODEL_CODE_LABELS[code]}{ref}")


# -------------------------------------------------------
# Full candidate feature list (Track A)
# -------------------------------------------------------

CANDIDATE_FEATURE_COLS = [
    # Group 1 — Core land-use (original)
    "log_office_jobs",
    "log_total_jobs",
    "log_pop_density",
    "log_median_income",
    "pct_no_vehicle",
    "pct_transit_commute",
    "office_job_share",
    "retail_job_share",
    "essential_job_share",
    "log_retail_job_density",
    "log_essential_job_density",
    # Group 2 — Commuter rail access
    "log_dist_to_nearest_terminal",
    "is_terminal_adjacent",
    # Group 3 — WAC/RAC destination-vs-origin
    "log_wac_rac_ratio",
    "pct_high_earn_residents",
    # Group 4 — Trip-purpose virtualization exposure
    "education_job_share",
    "log_education_jobs",
    "healthcare_job_share",
    "log_healthcare_job_density",
    "govt_job_share",
    "virtualization_exposure_index",
]

# Only keep columns that are present AND not entirely NaN in reg_df
ALL_FEATURE_COLS = [
    c for c in CANDIDATE_FEATURE_COLS
    if c in reg_df.columns and not reg_df[c].isna().all()
]

missing_features = [c for c in CANDIDATE_FEATURE_COLS
                    if c not in ALL_FEATURE_COLS]

print(f"\nTrack A candidate features: {len(ALL_FEATURE_COLS)}/{len(CANDIDATE_FEATURE_COLS)}")
if missing_features:
    print(f"  Excluded (missing or all-NaN): {missing_features}")


# -------------------------------------------------------
# Clean full feature dataset (Track A)
# -------------------------------------------------------

for col in ALL_FEATURE_COLS:
    reg_df[col] = pd.to_numeric(reg_df[col], errors="coerce")

reg_df["cluster"] = pd.to_numeric(reg_df["cluster"], errors="coerce")

# Drop NAs only on available features — avoids all-NaN columns wiping rows
reg_df_clean = reg_df.dropna(subset=["cluster"] + ALL_FEATURE_COLS).copy()
reg_df_clean["cluster"] = reg_df_clean["cluster"].astype(int)

print(f"\nRows after dropping NAs: {len(reg_df_clean):,} "
      f"(dropped {len(reg_df) - len(reg_df_clean):,})")

# Drop rare clusters — MNLogit requires MIN_CLUSTER_N observations
MIN_CLUSTER_N = 5
cluster_counts = reg_df_clean["cluster"].value_counts()
rare_clusters  = cluster_counts[cluster_counts < MIN_CLUSTER_N].index.tolist()
if rare_clusters:
    print(f"Dropping rare clusters (<{MIN_CLUSTER_N} obs): {rare_clusters}")
    reg_df_clean = reg_df_clean[
        ~reg_df_clean["cluster"].isin(rare_clusters)
    ].copy()

if REFERENCE_CLUSTER not in reg_df_clean["cluster"].unique():
    raise RuntimeError(
        f"Reference cluster {REFERENCE_CLUSTER} absent after cleaning.\n"
        f"Available clusters: {sorted(reg_df_clean['cluster'].unique())}\n"
        "Choose a different REFERENCE_LABEL."
    )

# Recode outcome in cleaned dataset
reg_df_clean["cluster_model_code"] = (
    reg_df_clean["cluster"].map(cluster_to_model_code)
)

X_all = reg_df_clean[ALL_FEATURE_COLS].astype(float).copy()
y_reg = reg_df_clean["cluster_model_code"].astype(int).copy()

print(f"\nClass distribution in modeling dataset:")
for code in sorted(MODEL_CODE_LABELS):
    n = (y_reg == code).sum()
    print(f"  Code {code} ({MODEL_CODE_LABELS[code]}): n={n}")


# ═══════════════════════════════════════════════════════════════
# TRACK A — Full 21-feature L2-regularized logistic regression
# ═══════════════════════════════════════════════════════════════
# Purpose: assess full feature set, CV accuracy, permutation
# importance. L2 regularization handles near-collinearity among
# 21 features without discarding any variable a priori.

print("\n" + "=" * 65)
print("TRACK A — Full Feature Set (L2-regularized, sklearn)")
print(f"  Features: {len(ALL_FEATURE_COLS)}")
print(f"  Regularization: L2, C=1.0")
print("=" * 65)

clf_full = Pipeline([
    ("scaler", StandardScaler()),
    ("logit",  LogisticRegression(
        multi_class="multinomial",
        solver="lbfgs",
        max_iter=2000,
        C=1.0,                    # L2 regularization — moderate penalty
        class_weight="balanced",  # corrects for class imbalance
        random_state=KMEANS_SEED,
    )),
])

# 5-fold stratified CV — N_SPLITS capped at smallest class size
cv_min   = y_reg.value_counts().min()
N_SPLITS = min(5, cv_min)

cv_scores_a = cross_val_score(
    clf_full, X_all.values, y_reg.values,
    cv=StratifiedKFold(n_splits=N_SPLITS, shuffle=True,
                       random_state=KMEANS_SEED),
    scoring="accuracy",
)

print(f"\nCross-validation ({N_SPLITS} folds):")
print(f"  Per-fold accuracy: {[f'{s:.3f}' for s in cv_scores_a]}")
print(f"  Mean ± SD:         {cv_scores_a.mean():.3f} ± {cv_scores_a.std():.3f}")
print(f"  Prior baseline (11 features, no regularization): 0.531")
delta = cv_scores_a.mean() - 0.531
direction = "improvement" if delta >= 0 else "decrease"
print(f"  Change vs prior:   {delta:+.3f} ({direction})")

# Fit on full dataset for coefficient extraction and importance
clf_full.fit(X_all.values, y_reg.values)
scaler_a = clf_full.named_steps["scaler"]
logit_a  = clf_full.named_steps["logit"]

# Build coefficient DataFrame
coef_rows_a = []
for code_pos, outcome_code in enumerate(logit_a.classes_):
    outcome_label = MODEL_CODE_LABELS.get(int(outcome_code), str(outcome_code))
    for feat_pos, feat in enumerate(ALL_FEATURE_COLS):
        coef_rows_a.append({
            "outcome_code":  int(outcome_code),
            "outcome_label": outcome_label,
            "feature":       feat,
            "coefficient":   logit_a.coef_[code_pos, feat_pos],
            "is_reference":  int(outcome_code) == 0,
        })

coef_df_a = pd.DataFrame(coef_rows_a)

# Pivot for display — sort by max absolute coefficient
coef_pivot_a = coef_df_a.pivot(
    index="feature", columns="outcome_label", values="coefficient"
)
coef_pivot_a["abs_max"] = coef_pivot_a.abs().max(axis=1)
coef_pivot_a = coef_pivot_a.sort_values("abs_max", ascending=False)

print("\nTrack A Coefficients (standardized, L2-regularized):")
print("Interpretation: positive = more likely than reference category")
print(f"Reference: {REFERENCE_LABEL}")
print("-" * 65)
try:
    display(coef_pivot_a.drop(columns="abs_max").round(3))
except NameError:
    print(coef_pivot_a.drop(columns="abs_max").round(3).to_string())

# In-sample classification
y_pred_a = clf_full.predict(X_all.values)
print("\nIn-sample confusion matrix (Track A):")
print(confusion_matrix(y_reg, y_pred_a))
print("\nClassification report (Track A):")
print(classification_report(
    y_reg, y_pred_a,
    labels=sorted(MODEL_CODE_LABELS.keys()),
    target_names=[MODEL_CODE_LABELS[c] for c in sorted(MODEL_CODE_LABELS)],
    zero_division=0,
))

# Permutation importance — which features actually matter?
print("Computing permutation importance (30 repeats)...")
perm = permutation_importance(
    clf_full, X_all.values, y_reg.values,
    n_repeats=30,
    random_state=KMEANS_SEED,
    scoring="accuracy",
)
perm_df = pd.DataFrame({
    "feature":    ALL_FEATURE_COLS,
    "importance": perm.importances_mean,
    "std":        perm.importances_std,
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("\nPermutation importance (mean accuracy drop when feature shuffled):")
print("  Features above 0 are helping; near 0 or negative = not contributing")
print(perm_df.to_string(index=False))

# Save Track A outputs
cv_results_a = pd.DataFrame({
    "metric": ["model","cv_folds","cv_accuracy_mean","cv_accuracy_std",
               "n_features","regularization_C","n_obs"],
    "value":  ["Track_A_full_regularized", N_SPLITS,
               cv_scores_a.mean(), cv_scores_a.std(),
               len(ALL_FEATURE_COLS), 1.0, len(y_reg)],
})
cv_results_a.to_csv(OUT_DIR / "regression_cv_results.csv", index=False)
coef_df_a.to_csv(OUT_DIR / "regression_tracka_coefficients.csv", index=False)
perm_df.to_csv(OUT_DIR / "regression_permutation_importance.csv", index=False)

print(f"\nTrack A outputs saved:")
print(f"  regression_cv_results.csv")
print(f"  regression_tracka_coefficients.csv")
print(f"  regression_permutation_importance.csv")


# ═══════════════════════════════════════════════════════════════
# TRACK B — Parsimonious MNLogit with SEs and p-values (Table 4)
# ═══════════════════════════════════════════════════════════════
# 8 features selected by domain priority.
# Hybrid Office Core n=32 → ~10 obs per equation → max ~8 features
# for stable Hessian inversion.
#
# Selection rationale:
#   log_dist_to_nearest_terminal  — direct commuter rail mechanism
#   log_wac_rac_ratio             — destination vs origin flow
#   log_office_jobs               — office land-use concentration
#   pct_transit_commute           — transit dependence
#   virtualization_exposure_index — composite virtualization signal
#   log_pop_density               — density control
#   pct_no_vehicle                — car-free household control
#   log_median_income             — income control

print("\n" + "=" * 65)
print("TRACK B — Parsimonious MNLogit (Table 4, SEs and p-values)")
print(f"  Max features: 8  |  Smallest class n: {cv_min}")
print("=" * 65)

# Build parsimonious feature list from candidates that are available
# and not entirely NaN, in domain-priority order
PARSIMONIOUS_FEATURES = [
    f for f in PARSIMONIOUS_FEATURE_CANDIDATES
    if f in reg_df_clean.columns
    and not reg_df_clean[f].isna().all()
    and reg_df_clean[f].notna().sum() == len(reg_df_clean)
]

# Cap at 8 features
PARSIMONIOUS_FEATURES = PARSIMONIOUS_FEATURES[:8]

print(f"\nParsimonious feature set ({len(PARSIMONIOUS_FEATURES)}):")
for f in PARSIMONIOUS_FEATURES:
    print(f"  {f}")

if len(PARSIMONIOUS_FEATURES) < 3:
    print("WARNING: Fewer than 3 parsimonious features available.")
    print("Extending from ALL_FEATURE_COLS to reach minimum of 3.")
    for f in ALL_FEATURE_COLS:
        if f not in PARSIMONIOUS_FEATURES:
            PARSIMONIOUS_FEATURES.append(f)
        if len(PARSIMONIOUS_FEATURES) >= 3:
            break

X_pars   = reg_df_clean[PARSIMONIOUS_FEATURES].astype(float).copy()
scaler_b = StandardScaler()
X_pars_sc = pd.DataFrame(
    scaler_b.fit_transform(X_pars),
    columns=PARSIMONIOUS_FEATURES,
    index=reg_df_clean.index,
)

X_sm_b    = sm.add_constant(X_pars_sc, has_constant="add")
mnlogit_b = sm.MNLogit(y_reg, X_sm_b)

# Try multiple optimizers in order of robustness
# bfgs and lbfgs are most reliable; newton is fastest but requires
# positive-definite Hessian; nm (Nelder-Mead) is derivative-free fallback
fit_result = None
for method, opts in [
    ("bfgs",   {"maxiter": 2000, "disp": False}),
    ("lbfgs",  {"maxiter": 2000, "disp": False}),
    ("newton", {"maxiter": 500,  "disp": False}),
    ("nm",     {"maxiter": 5000, "disp": False}),
]:
    try:
        candidate = mnlogit_b.fit(method=method, **opts)
        # Verify that standard errors are computable — this is what
        # failed before when the Hessian was singular
        _ = candidate.bse
        _ = candidate.pvalues
        fit_result = candidate
        print(f"\nMNLogit converged: method={method} ✓")
        break
    except Exception as e:
        print(f"  {method}: {type(e).__name__} — {str(e)[:70]}")
        fit_result = None


# ── Track B: model converged ────────────────────────────────────
if fit_result is not None:
    result = fit_result
    print(result.summary())

    params_b  = result.params
    bse_b     = result.bse
    pvalues_b = result.pvalues

    nonref_codes = [c for c in sorted(MODEL_CODE_LABELS) if c != 0]
    coef_rows_b  = []

    for col_pos, col in enumerate(params_b.columns):
        outcome_code    = nonref_codes[col_pos]
        outcome_cluster = model_code_to_cluster[outcome_code]
        outcome_label   = MODEL_CODE_LABELS[outcome_code]
        ref_label       = MODEL_CODE_LABELS[0]

        for variable in params_b.index:
            p = pvalues_b.loc[variable, col]
            coef_rows_b.append({
                "reference_label":   ref_label,
                "outcome_model_code": outcome_code,
                "outcome_cluster":   outcome_cluster,
                "outcome_label":     outcome_label,
                "variable":          variable,
                "coefficient":       params_b.loc[variable, col],
                "std_error":         bse_b.loc[variable, col],
                "p_value":           p,
                "odds_ratio":        np.exp(params_b.loc[variable, col]),
                "significance":      p_stars(p),
            })

    regression_results = pd.DataFrame(coef_rows_b)
    regression_results["coef_str"] = (
        regression_results["coefficient"].map("{:.3f}".format)
        + regression_results["significance"]
    )
    regression_results["se_str"] = (
        "(" + regression_results["std_error"].map("{:.3f}".format) + ")"
    )

    # Track B CV for comparison with Track A
    clf_b = Pipeline([
        ("scaler", StandardScaler()),
        ("logit",  LogisticRegression(
            multi_class="multinomial", solver="lbfgs",
            max_iter=2000, class_weight="balanced",
            random_state=KMEANS_SEED,
        )),
    ])
    cv_scores_b = cross_val_score(
        clf_b, X_pars.values, y_reg.values,
        cv=StratifiedKFold(n_splits=N_SPLITS, shuffle=True,
                           random_state=KMEANS_SEED),
        scoring="accuracy",
    )
    print(f"\nTrack B CV ({N_SPLITS} folds): "
          f"{cv_scores_b.mean():.3f} ± {cv_scores_b.std():.3f}")
    print(f"Track A CV ({N_SPLITS} folds): "
          f"{cv_scores_a.mean():.3f} ± {cv_scores_a.std():.3f}")
    print("(Track B uses 8 features; Track A uses all available features)")

    # Pseudo-R² and fit statistics
    pseudo_r2 = getattr(result, "prsquared", np.nan)
    llf       = getattr(result, "llf", np.nan)
    llnull    = getattr(result, "llnull", np.nan)
    aic       = getattr(result, "aic", np.nan)
    print(f"\nModel fit statistics (Track B):")
    print(f"  McFadden pseudo-R²: {pseudo_r2:.4f}")
    print(f"  Log-likelihood:     {llf:.2f}")
    print(f"  Null log-likelihood:{llnull:.2f}")
    print(f"  AIC:                {aic:.2f}")
    print(f"  N:                  {len(y_reg):,}")

    # In-sample confusion matrix (Track B)
    pred_probs_b = result.predict(X_sm_b)
    all_codes    = sorted(MODEL_CODE_LABELS)
    nonref_all   = [c for c in all_codes if c != 0]
    pred_probs_df_b = pd.DataFrame(pred_probs_b, index=y_reg.index)

    if pred_probs_df_b.shape[1] == len(all_codes):
        pred_probs_df_b.columns = all_codes
    elif pred_probs_df_b.shape[1] == len(nonref_all):
        pred_probs_df_b.columns = nonref_all
        pred_probs_df_b[0] = 1 - pred_probs_df_b[nonref_all].sum(axis=1)
        pred_probs_df_b = pred_probs_df_b[all_codes]

    pred_codes_b = pred_probs_df_b.idxmax(axis=1).astype(int)
    print("\nIn-sample confusion matrix (Track B):")
    print(confusion_matrix(y_reg, pred_codes_b))
    print("\nClassification report (Track B):")
    print(classification_report(
        y_reg, pred_codes_b,
        labels=all_codes,
        target_names=[MODEL_CODE_LABELS[c] for c in all_codes],
        zero_division=0,
    ))

    # Set FEATURE_COLS and X_raw for Section 5.4 compatibility
    FEATURE_COLS = PARSIMONIOUS_FEATURES
    X_raw        = X_pars
    track_b_success = True


# ── Track B: all optimizers failed ──────────────────────────────
else:
    print("\n" + "!" * 65)
    print("Track B: all MNLogit optimizers failed to produce stable SEs.")
    print("Falling back to Track A regularized coefficients for Table 4.")
    print("!" * 65)
    print("""
Paper note (add to Section 5 limitations):
  'The smallest demand regime cluster (Hybrid Office Core, n=32)
  is below the threshold for reliable unregularized multinomial
  logistic regression with the parsimonious feature set. L2-
  regularized estimates are reported; standard errors are not
  available. Future work with expanded data should revisit
  this specification.'
""")
    # Build regression_results from Track A coefficients
    coef_rows_fallback = []
    ref_label = MODEL_CODE_LABELS[0]
    for _, row in coef_df_a[coef_df_a["outcome_code"] != 0].iterrows():
        coef_rows_fallback.append({
            "reference_label":   ref_label,
            "outcome_model_code": row["outcome_code"],
            "outcome_cluster":   model_code_to_cluster[row["outcome_code"]],
            "outcome_label":     row["outcome_label"],
            "variable":          row["feature"],
            "coefficient":       row["coefficient"],
            "std_error":         np.nan,
            "p_value":           np.nan,
            "odds_ratio":        np.exp(row["coefficient"]),
            "significance":      "",
        })
    regression_results = pd.DataFrame(coef_rows_fallback)
    regression_results["coef_str"] = regression_results["coefficient"].map("{:.3f}".format)
    regression_results["se_str"]   = ""

    # Create a placeholder result object for Section 5.4 compatibility
    result = None
    FEATURE_COLS = ALL_FEATURE_COLS
    X_raw        = X_all
    track_b_success = False


# -------------------------------------------------------
# Save all outputs
# -------------------------------------------------------

regression_out    = OUT_DIR / "table4_regression_results.csv"
reg_dataset_out   = OUT_DIR / "regression_dataset_2024.csv"
model_mapping_out = OUT_DIR / "regression_outcome_reference_mapping.csv"

regression_results.to_csv(regression_out, index=False)
reg_df_clean.to_csv(reg_dataset_out, index=False)

pd.DataFrame([
    {"model_code":       code,
     "original_cluster": model_code_to_cluster[code],
     "cluster_label":    MODEL_CODE_LABELS[code],
     "is_reference":     code == 0}
    for code in sorted(MODEL_CODE_LABELS)
]).to_csv(model_mapping_out, index=False)

# Update reg_df for downstream compatibility
reg_df = reg_df_clean.copy()

print(f"\nOutputs saved:")
print(f"  {regression_out}")
print(f"  {reg_dataset_out}")
print(f"  {model_mapping_out}")
print(f"  {OUT_DIR}/regression_cv_results.csv")
print(f"  {OUT_DIR}/regression_tracka_coefficients.csv")
print(f"  {OUT_DIR}/regression_permutation_importance.csv")

print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Section 5.3 complete.

Track A (full model):
  Features:  {len(ALL_FEATURE_COLS)}
  CV accuracy: {cv_scores_a.mean():.3f} ± {cv_scores_a.std():.3f}

Track B (parsimonious, Table 4):
  Features:  {len(PARSIMONIOUS_FEATURES)}
  Converged: {track_b_success}
  {"CV accuracy: " + f"{cv_scores_b.mean():.3f} ± {cv_scores_b.std():.3f}" if track_b_success else "Fallback: Track A coefficients used"}

Run Section 5.4 to format Table 4.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")


### 5.4 Regression Output Table (Table 4)

**Purpose:** Format the MNLogit coefficient output into a publication-ready Table 4 with interleaved coefficient and standard-error rows, significance stars, model fit statistics, and CV accuracy. Requires objects from Section 5.3 to be in memory.


In [ ]:
# ==============================================================
# Section 5.4 Recovery — Rebuild MODEL_CODE_LABELS and
# model_code_to_cluster from existing regression objects
#
# Run this cell BEFORE Section 5.4 if you see:
#   RuntimeError: Missing required objects from Section 5.3:
#   ['MODEL_CODE_LABELS', 'model_code_to_cluster']
#
# Root cause:
#   Section 5.3 was run in its simpler form (without explicit
#   reference category), which does not produce MODEL_CODE_LABELS
#   or model_code_to_cluster. This block reconstructs both from
#   the regression_results and reg_df objects that ARE in memory.
#
# Reference category: "Major Midtown / Regional Core"
#   Change REFERENCE_LABEL below if you want a different baseline.
# ==============================================================

REFERENCE_LABEL = "Major Midtown / Regional Core"

# ---- Validate required upstream objects ----
for obj in ["result", "reg_df", "regression_results", "FEATURE_COLS", "X_raw", "y_reg"]:
    if obj not in globals():
        raise RuntimeError(
            f"'{obj}' not found. Run Section 5.3 first, then re-run this cell."
        )

# ---- Rebuild label → cluster mapping from reg_df ----
if "cluster_label" not in reg_df.columns:
    raise RuntimeError(
        "reg_df is missing 'cluster_label'. "
        "Run Section 4.3 (cluster labeling) before Section 5.3."
    )

label_to_cluster = (
    reg_df[["cluster", "cluster_label"]]
    .drop_duplicates()
    .set_index("cluster_label")["cluster"]
    .to_dict()
)

print("Available cluster labels in reg_df:")
for lbl, cid in sorted(label_to_cluster.items(), key=lambda x: x[1]):
    print(f"  Cluster {cid}: {lbl}")

# ---- Validate reference label exists ----
if REFERENCE_LABEL not in label_to_cluster:
    raise RuntimeError(
        f"REFERENCE_LABEL '{REFERENCE_LABEL}' not found in reg_df.\n"
        f"Available labels: {list(label_to_cluster.keys())}"
    )

REFERENCE_CLUSTER = int(label_to_cluster[REFERENCE_LABEL])

# ---- Build model code mapping (reference → 0, others → 1, 2, ...) ----
remaining = sorted(
    c for c in reg_df["cluster"].unique()
    if int(c) != REFERENCE_CLUSTER
)

cluster_to_model_code = {REFERENCE_CLUSTER: 0}
for i, c in enumerate(remaining, start=1):
    cluster_to_model_code[int(c)] = i

model_code_to_cluster = {v: k for k, v in cluster_to_model_code.items()}

# ---- Build MODEL_CODE_LABELS ----
cluster_to_label = (
    reg_df[["cluster", "cluster_label"]]
    .drop_duplicates()
    .set_index("cluster")["cluster_label"]
    .to_dict()
)

MODEL_CODE_LABELS = {
    model_code: cluster_to_label[original_cluster]
    for model_code, original_cluster in model_code_to_cluster.items()
}

# ---- Rebuild regression_results with required columns ----
# The simpler Section 5.3 produces outcome_cluster_vs_reference
# but not reference_label / outcome_label. Add them now.
if "reference_label" not in regression_results.columns:
    regression_results["reference_label"] = MODEL_CODE_LABELS[0]

if "outcome_label" not in regression_results.columns:
    # Map the numeric outcome column to a label
    # statsmodels names outcomes by the integer cluster code
    def _map_outcome_label(outcome_val):
        try:
            code = int(float(str(outcome_val)))
            return MODEL_CODE_LABELS.get(code, str(outcome_val))
        except (ValueError, TypeError):
            return str(outcome_val)

    regression_results["outcome_label"] = regression_results[
        "outcome_cluster_vs_reference"
    ].apply(_map_outcome_label)

# ---- Rebuild y_reg using model codes (reference = 0) ----
y_reg = reg_df["cluster"].map(cluster_to_model_code).astype(int)

# ---- Summary ----
print(f"\nReference category:  {REFERENCE_LABEL} (cluster {REFERENCE_CLUSTER})")
print(f"\nModel code mapping:")
for code in sorted(MODEL_CODE_LABELS):
    orig = model_code_to_cluster[code]
    lbl  = MODEL_CODE_LABELS[code]
    ref  = " ← reference" if code == 0 else ""
    print(f"  Model code {code}: original cluster {orig} — {lbl}{ref}")

print(f"\nregression_results columns: {list(regression_results.columns)}")
print("\nReady to run Section 5.4.")

---
## SECTION 6: Visualization for Publication

### 6.1 Figure 4: NYC Cluster Map

**Purpose:** Plot all 445 station complexes on a NYC borough basemap, coloured by their 2024 demand regime cluster assignment. This is the paper's spatial evidence — showing that demand regimes have a strong geographic pattern (office core in Manhattan vs. residential in outer boroughs) validates the clustering approach.

**Dependencies:** Requires `cluster_assignments[2024]` (with cluster labels), `station_geo_df` (built in Section 5.1), `CLUSTER_LABELS` (from Section 4.3), and `PALETTE` (from Section 0.3). Borough boundaries are downloaded from NYC Open Data with multiple fallback URLs.

**Outputs:** `figure4_nyc_cluster_map.png` (300 DPI) and `figure4_nyc_cluster_map.pdf`.


In [ ]:
# ==============================================================
# Section 6.1 — Figure 4: NYC Cluster Map
#
# Purpose:
#   Plots all NYC subway station complexes on a borough basemap,
#   colored by their 2024 day-of-week demand regime cluster.
#
#   This figure is intended to serve as Figure 4 in the TRR paper.
#
# Note:
#   This section uses station_geo_df from Section 5.1 for coordinates.
# ==============================================================

import requests
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------------------------------
# Validate required inputs
# --------------------------------------------------------------
if "cluster_assignments" not in globals():
    raise RuntimeError("cluster_assignments not found. Run Sections 4.2 and 4.3 first.")

if 2024 not in cluster_assignments:
    raise RuntimeError(
        f"cluster_assignments[2024] not found. Available keys: {list(cluster_assignments.keys())}"
    )

if "station_geo_df" not in globals():
    raise RuntimeError(
        "station_geo_df not found. Run Section 5.1 first. "
        "Section 5.1 recovers station-complex coordinates."
    )

required_cluster_cols = [
    "station_complex_id",
    "complex_name",
    "cluster",
    "cluster_label",
]

missing_cluster_cols = [
    c for c in required_cluster_cols
    if c not in cluster_assignments[2024].columns
]

if missing_cluster_cols:
    raise RuntimeError(
        f"cluster_assignments[2024] is missing required columns: {missing_cluster_cols}\n"
        f"Columns found: {list(cluster_assignments[2024].columns)}"
    )

required_geo_cols = [
    "station_complex_id",
    "complex_lat",
    "complex_lon",
]

missing_geo_cols = [
    c for c in required_geo_cols
    if c not in station_geo_df.columns
]

if missing_geo_cols:
    raise RuntimeError(
        f"station_geo_df is missing required coordinate columns: {missing_geo_cols}\n"
        f"Columns found: {list(station_geo_df.columns)}"
    )

# --------------------------------------------------------------
# Build map dataframe by merging clusters with station coordinates
# --------------------------------------------------------------
cluster_2024 = cluster_assignments[2024][
    [
        "station_complex_id",
        "complex_name",
        "cluster",
        "cluster_label",
    ]
].copy()

cluster_2024["station_complex_id"] = pd.to_numeric(
    cluster_2024["station_complex_id"],
    errors="coerce"
)

cluster_2024["cluster"] = pd.to_numeric(
    cluster_2024["cluster"],
    errors="coerce"
).astype("Int64")

coords = station_geo_df[
    [
        "station_complex_id",
        "complex_lat",
        "complex_lon",
    ]
].drop_duplicates(subset=["station_complex_id"]).copy()

coords["station_complex_id"] = pd.to_numeric(
    coords["station_complex_id"],
    errors="coerce"
)

coords["complex_lat"] = pd.to_numeric(
    coords["complex_lat"],
    errors="coerce"
)

coords["complex_lon"] = pd.to_numeric(
    coords["complex_lon"],
    errors="coerce"
)

map_df = cluster_2024.merge(
    coords,
    on="station_complex_id",
    how="left",
)

map_df = map_df.dropna(
    subset=[
        "complex_lat",
        "complex_lon",
        "cluster",
    ]
).copy()

map_df["cluster"] = map_df["cluster"].astype(int)

print("Figure 4 mapping dataset")
print("-" * 60)
print(f"Mapped station complexes: {len(map_df):,}")
print(f"Clusters included: {sorted(map_df['cluster'].dropna().astype(int).unique())}")

# --------------------------------------------------------------
# Convert to GeoDataFrame
# --------------------------------------------------------------
map_gdf = gpd.GeoDataFrame(
    map_df,
    geometry=gpd.points_from_xy(
        map_df["complex_lon"],
        map_df["complex_lat"]
    ),
    crs="EPSG:4326",
)

# --------------------------------------------------------------
# NYC Borough boundaries
# --------------------------------------------------------------
# This block tries multiple reliable NYC Open Data / DCP options.
# The Socrata geospatial API expects format=GeoJSON, not type=GeoJSON.
# --------------------------------------------------------------
boroughs_cache = OUT_DIR / "nyc_boroughs.geojson"

BOROUGH_URL_CANDIDATES = [
    # Current NYC Open Data Borough Boundaries dataset
    "https://data.cityofnewyork.us/resource/gthc-hcne.geojson",

    # Alternate NYC Open Data Borough Boundaries dataset, water excluded
    "https://data.cityofnewyork.us/resource/yqww-f9f3.geojson",

    # Socrata geospatial export format syntax
    "https://data.cityofnewyork.us/api/geospatial/gthc-hcne?method=export&format=GeoJSON",
    "https://data.cityofnewyork.us/api/geospatial/yqww-f9f3?method=export&format=GeoJSON",
]

if not boroughs_cache.exists() or boroughs_cache.stat().st_size == 0:
    last_error = None

    for url in BOROUGH_URL_CANDIDATES:
        try:
            print(f"Trying borough boundary source: {url}")
            resp = requests.get(url, timeout=60)
            resp.raise_for_status()

            # Basic validation: make sure response looks like GeoJSON
            if b"FeatureCollection" not in resp.content and b'"features"' not in resp.content:
                raise ValueError("Downloaded file does not appear to be valid GeoJSON.")

            boroughs_cache.write_bytes(resp.content)
            print("NYC borough boundaries downloaded successfully.")
            break

        except Exception as e:
            last_error = e
            print(f"Failed source: {url}")
            print(f"Reason: {e}")

    if not boroughs_cache.exists() or boroughs_cache.stat().st_size == 0:
        raise RuntimeError(
            "Could not download NYC borough boundaries from any source. "
            f"Last error: {last_error}"
        )

boroughs_gdf = gpd.read_file(boroughs_cache).to_crs("EPSG:4326")

# --------------------------------------------------------------
# Plot Figure 4
# --------------------------------------------------------------
fig, ax = plt.subplots(figsize=(11, 10))

# Borough basemap
boroughs_gdf.plot(
    ax=ax,
    color="#F5F5F0",
    edgecolor="#CCCCCC",
    linewidth=0.8,
    zorder=1,
)

# Station complexes by cluster
for c_id, c_label in CLUSTER_LABELS.items():
    c_id_int = int(c_id)
    subset = map_gdf[map_gdf["cluster"] == c_id_int]

    if subset.empty:
        continue

    subset.plot(
        ax=ax,
        color=PALETTE[c_id],
        markersize=16,
        alpha=0.85,
        edgecolor="black",
        linewidth=0.2,
        label=f"Cluster {c_id_int}: {c_label} (n={len(subset)})",
        zorder=3,
    )

# --------------------------------------------------------------
# Tighter map extent around station geography
# --------------------------------------------------------------
x_min, y_min, x_max, y_max = map_gdf.total_bounds

x_pad = (x_max - x_min) * 0.10
y_pad = (y_max - y_min) * 0.10

ax.set_xlim(x_min - x_pad, x_max + x_pad)
ax.set_ylim(y_min - y_pad, y_max + y_pad)

# --------------------------------------------------------------
# Formatting
# --------------------------------------------------------------
ax.set_axis_off()

ax.legend(
    loc="lower left",
    fontsize=8,
    framealpha=0.95,
    title="Day-of-Week Demand Regime",
    title_fontsize=8,
    markerscale=1.1,
)

ax.set_title(
    "NYC Subway Station Demand Regime Clusters, 2024",
    fontsize=13,
    fontweight="bold",
    pad=15,
)

plt.tight_layout()

# --------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------
figure4_png = OUT_DIR / "figure4_nyc_cluster_map.png"
figure4_pdf = OUT_DIR / "figure4_nyc_cluster_map.pdf"

plt.savefig(
    figure4_png,
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    figure4_pdf,
    bbox_inches="tight"
)

plt.show()

print(f"Figure 4 saved to: {figure4_png}")
print(f"Figure 4 PDF saved to: {figure4_pdf}")

---
## SECTION 7: Robustness Checks

Two robustness checks address the most likely reviewer challenges:

**7.1 K Sensitivity** — Re-runs K-means at K±1 and K+2. Tests whether a Hybrid Office Core cluster (defined by Tue–Thu > Mon/Fri × 1.15 threshold) appears under every alternative K specification. A PASS result means the finding is not an artefact of K selection.

**7.2 Weekend Definition Sensitivity** — Paired t-tests comparing Mon/Fri against Sat/Sun and against Tue–Thu at Hybrid Office Core stations. Tests the paper's specific claim that Mon/Fri are *shoulder days* — above true weekend demand but below mid-week demand. A PASS result directly validates the paper's challenge to the weekday/weekend binary.


### 7.1 K Sensitivity Analysis

**Tests:** K = K_FINAL ± 1 and K_FINAL + 2 (excluding K < 2 and K = K_FINAL itself).  
**Detection criterion:** A cluster is "Hybrid Office Core" if Tue/Wed/Thu average > Mon/Fri average × 1.15 AND both Mon and Fri are individually below the Tue–Thu average.  
**Saves:** `section7_k_sensitivity_robustness_summary.csv` (referenced in Section 8.1 checklist).


In [ ]:
# ==============================================================
# Section 7.1 — Robustness: K Sensitivity Analysis
#
# Purpose:
#   Re-runs K-means at nearby K values and verifies that the
#   Hybrid Office Core demand regime persists across alternative
#   cluster specifications.
#
# Robustness logic:
#   A Hybrid Office Core cluster is defined as a cluster where:
#     1. Tue/Wed/Thu average share is higher than Mon/Fri average share.
#     2. Both Monday and Friday are below the Tue/Wed/Thu average.
#     3. Tue/Wed/Thu concentration is meaningfully higher than Mon/Fri.
#
# Key test:
#   Does at least one Hybrid Office Core cluster appear under each
#   alternative K specification?
# ==============================================================

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# --------------------------------------------------------------
# Define K values for robustness testing
# --------------------------------------------------------------
K_ROBUSTNESS = sorted(set([
    K_FINAL - 2,
    K_FINAL - 1,
    K_FINAL + 1,
    K_FINAL + 2,
]))

K_ROBUSTNESS = [
    k for k in K_ROBUSTNESS
    if k >= 2 and k != K_FINAL
]

print("Robustness Check: K Sensitivity")
print("=" * 70)
print(f"Final selected K: {K_FINAL}")
print(f"Alternative K values tested: {K_ROBUSTNESS}")

robustness_results = []

# --------------------------------------------------------------
# Run K-means for each alternative K
# --------------------------------------------------------------
for k_test in K_ROBUSTNESS:

    km_test = KMeans(
        n_clusters=k_test,
        n_init=KMEANS_INIT,
        random_state=KMEANS_SEED,
        max_iter=500,
    )

    labels_test = km_test.fit_predict(X_scaled)
    sil_test = silhouette_score(X_scaled, labels_test)

    # ----------------------------------------------------------
    # Compute cluster centroids in normalized day-of-week space
    # ----------------------------------------------------------
    test_df = cluster_df.copy()
    test_df["cluster_test"] = labels_test

    centroids_test = (
        test_df
        .groupby("cluster_test")[DOW_LABELS]
        .mean()
        .copy()
    )

    cluster_sizes = (
        test_df
        .groupby("cluster_test")
        .size()
        .rename("n")
    )

    centroids_test = centroids_test.join(cluster_sizes)

    # ----------------------------------------------------------
    # Hybrid Office Core diagnostic metrics
    # ----------------------------------------------------------
    centroids_test["monfri_avg"] = centroids_test[["Mon", "Fri"]].mean(axis=1)
    centroids_test["tuethu_avg"] = centroids_test[["Tue", "Wed", "Thu"]].mean(axis=1)

    centroids_test["monfri_share"] = centroids_test[["Mon", "Fri"]].sum(axis=1)
    centroids_test["tuethu_share"] = centroids_test[["Tue", "Wed", "Thu"]].sum(axis=1)

    centroids_test["office_core_ratio"] = (
        centroids_test["tuethu_avg"] / centroids_test["monfri_avg"]
    )

    centroids_test["mon_below_tuethu_avg"] = (
        centroids_test["Mon"] < centroids_test["tuethu_avg"]
    )

    centroids_test["fri_below_tuethu_avg"] = (
        centroids_test["Fri"] < centroids_test["tuethu_avg"]
    )

    centroids_test["is_hybrid_office_core"] = (
        (centroids_test["tuethu_avg"] > centroids_test["monfri_avg"] * 1.15)
        & centroids_test["mon_below_tuethu_avg"]
        & centroids_test["fri_below_tuethu_avg"]
    )

    n_office_core = int(centroids_test["is_hybrid_office_core"].sum())

    # ----------------------------------------------------------
    # Identify strongest Hybrid Office Core candidate
    # ----------------------------------------------------------
    candidate_idx = centroids_test["office_core_ratio"].idxmax()
    candidate = centroids_test.loc[candidate_idx]

    robustness_results.append({
        "K": k_test,
        "silhouette": sil_test,
        "hybrid_office_core_detected": n_office_core > 0,
        "n_hybrid_office_core_clusters": n_office_core,
        "strongest_candidate_cluster": int(candidate_idx),
        "strongest_candidate_ratio": candidate["office_core_ratio"],
        "strongest_candidate_n": int(candidate["n"]),
        "strongest_candidate_Mon": candidate["Mon"],
        "strongest_candidate_Tue": candidate["Tue"],
        "strongest_candidate_Wed": candidate["Wed"],
        "strongest_candidate_Thu": candidate["Thu"],
        "strongest_candidate_Fri": candidate["Fri"],
    })

    # ----------------------------------------------------------
    # Print detailed results
    # ----------------------------------------------------------
    print(f"\nK={k_test}")
    print("-" * 70)
    print(f"Silhouette score: {sil_test:.4f}")
    print(f"Hybrid Office Core clusters detected: {n_office_core}")

    display_cols = [
        "n",
        "Mon",
        "Tue",
        "Wed",
        "Thu",
        "Fri",
        "monfri_avg",
        "tuethu_avg",
        "office_core_ratio",
        "is_hybrid_office_core",
    ]

    print(
        centroids_test[display_cols]
        .round(3)
        .to_string()
    )

# --------------------------------------------------------------
# Summarize robustness results
# --------------------------------------------------------------
robustness_summary_df = pd.DataFrame(robustness_results)

print("\nRobustness Summary")
print("=" * 70)

print(
    robustness_summary_df[
        [
            "K",
            "silhouette",
            "hybrid_office_core_detected",
            "n_hybrid_office_core_clusters",
            "strongest_candidate_cluster",
            "strongest_candidate_ratio",
            "strongest_candidate_n",
        ]
    ]
    .round(4)
    .to_string(index=False)
)

# --------------------------------------------------------------
# Final robustness conclusion — re-framed as a finding
# --------------------------------------------------------------
# The detection criterion (Tue/Wed/Thu > Mon/Fri × 1.15) is strict
# by design. At coarser resolutions (K=2, K=3), office-core and
# residential-stable stations are pooled into single clusters —
# which is precisely the inadequate binary this paper challenges.
# The Hybrid Office Core regime only resolving at K≥4 is therefore
# a substantive finding, not a weakness.
# --------------------------------------------------------------

all_k_pass = robustness_summary_df["hybrid_office_core_detected"].all()
min_k_detected = (
    robustness_summary_df.loc[
        robustness_summary_df["hybrid_office_core_detected"], "K"
    ].min()
    if robustness_summary_df["hybrid_office_core_detected"].any()
    else None
)

coarse_k = robustness_summary_df.loc[
    ~robustness_summary_df["hybrid_office_core_detected"], "K"
].tolist()

fine_k = robustness_summary_df.loc[
    robustness_summary_df["hybrid_office_core_detected"], "K"
].tolist()

if all_k_pass:
    print(
        "\nConclusion: STRONG PASS — A Hybrid Office Core demand regime is "
        "detected under every alternative K specification tested. "
        f"The finding is robust across K = {sorted(robustness_summary_df['K'].tolist())}."
    )
else:
    print("\nConclusion: FINDING — Hybrid Office Core regime emergence is K-dependent.")
    print("-" * 70)

    if coarse_k:
        print(
            f"\n  At K = {coarse_k} (coarser than K_FINAL={K_FINAL}):"
            "\n  Office-core and residential-stable stations are pooled into"
            "\n  a single large cluster — the inadequate binary this paper"
            "\n  argues is empirically insufficient. Non-detection at K=2,3"
            "\n  confirms coarse solutions reproduce the weekday/weekend"
            "\n  binary rather than the nuanced regime structure."
        )

    if fine_k:
        print(
            f"\n  At K = {fine_k} (K_FINAL or finer):"
            "\n  The Hybrid Office Core regime emerges as a distinct cluster."
            f"\n  Minimum K at which regime is detectable: K = {min_k_detected}."
            "\n  K=4 is the minimum resolution at which this paper's central"
            "\n  empirical contribution is observable."
        )

    print(
        "\n  Paper interpretation:"
        "\n  K-dependency of regime detection is a substantive finding."
        "\n  Coarser solutions conflate the demand-regime differences this"
        "\n  paper documents. Report in Section 7 as evidence that K=4"
        "\n  represents the minimum analytically meaningful resolution."
    )
# Save robustness summary for Section 8.1 checklist
if 'robustness_summary_df' in globals():
    robustness_summary_df.to_csv(
        OUT_DIR / "section7_k_sensitivity_robustness_summary.csv",
        index=False
    )
    print(f"\nSaved: {OUT_DIR / 'section7_k_sensitivity_robustness_summary.csv'}")


### 7.2 Weekend Definition Sensitivity

**Tests:** 7 paired t-tests at Hybrid Office Core stations in 2024:  
Mon vs Sat, Mon vs Sun, Fri vs Sat, Fri vs Sun (shoulder-day tests),  
Tue vs Mon, Wed vs Mon, Thu vs Fri (midweek-premium tests).  
**Metric:** Paired Cohen's d in addition to t-statistics and p-values.  
**Saves:** `section7_weekend_definition_sensitivity.csv`


In [ ]:
# ==============================================================
# Section 7.2 — Robustness: Weekend Definition Sensitivity
#
# Purpose:
#   Tests whether conclusions change if Monday and Friday are
#   treated as "soft weekend" / shoulder days and the empirical
#   core weekday period is restricted to Tue–Thu.
#
#   This directly supports the paper's argument that the traditional
#   weekday/weekend binary is no longer empirically sufficient for
#   understanding subway ridership patterns under hybrid work.
#
# Robustness logic:
#   At Hybrid Office Core stations, Monday and Friday should not
#   behave like Tue–Thu core weekdays. Instead, they should show
#   lower demand relative to Tue–Thu while still remaining distinct
#   from true weekend days.
#
# Tests performed:
#   1. Mon vs Sat
#   2. Mon vs Sun
#   3. Fri vs Sat
#   4. Fri vs Sun
#   5. Tue vs Mon
#   6. Wed vs Mon
#   7. Thu vs Fri
#
# Interpretation:
#   - Significant Tue/Wed/Thu > Mon/Fri supports a midweek premium.
#   - Significant Mon/Fri > Sat/Sun indicates Mon/Fri are not simply
#     weekends, but shoulder days between core weekdays and weekends.
# ==============================================================

from scipy import stats
import pandas as pd
import numpy as np

# --------------------------------------------------------------
# Validate required inputs
# --------------------------------------------------------------
if "cluster_assignments" not in globals():
    raise RuntimeError("cluster_assignments not found. Run Sections 4.2 and 4.3 first.")

if 2024 not in cluster_assignments:
    raise RuntimeError(
        f"cluster_assignments[2024] not found. Available years: {list(cluster_assignments.keys())}"
    )

if "dow_raw" not in globals():
    raise RuntimeError("dow_raw not found. Run the section that builds raw day-of-week ridership tables first.")

if 2024 not in dow_raw:
    raise RuntimeError(
        f"dow_raw[2024] not found. Available years: {list(dow_raw.keys())}"
    )

required_days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

missing_days = [
    d for d in required_days
    if d not in dow_raw[2024].columns
]

if missing_days:
    raise RuntimeError(
        f"dow_raw[2024] is missing required day columns: {missing_days}\n"
        f"Columns found: {list(dow_raw[2024].columns)}"
    )

# --------------------------------------------------------------
# Identify Hybrid Office Core stations
# --------------------------------------------------------------
cluster_2024 = cluster_assignments[2024].copy()

if "cluster_label" not in cluster_2024.columns:
    raise RuntimeError("cluster_assignments[2024] is missing 'cluster_label'.")

office_core_ids = cluster_2024.loc[
    cluster_2024["cluster_label"].eq("Hybrid Office Core"),
    "station_complex_id"
].dropna().unique()

if len(office_core_ids) == 0:
    raise RuntimeError(
        "No stations labeled 'Hybrid Office Core' were found in cluster_assignments[2024]. "
        "Check CLUSTER_LABELS or cluster labels before running this robustness test."
    )

office_raw_2024 = dow_raw[2024].loc[
    dow_raw[2024]["station_complex_id"].isin(office_core_ids)
].copy()

if office_raw_2024.empty:
    raise RuntimeError(
        "No matching Hybrid Office Core stations were found in dow_raw[2024]. "
        "Check that station_complex_id types match across dataframes."
    )

# --------------------------------------------------------------
# Helper function for paired tests
# --------------------------------------------------------------
def paired_day_test(df, day_a, day_b, description):
    """
    Runs a paired t-test between two day-of-week columns after aligning
    observations by station_complex_id and dropping missing pairs.

    Returns:
        Dictionary with test statistics and descriptive statistics.
    """

    paired = df[["station_complex_id", day_a, day_b]].copy()

    paired[day_a] = pd.to_numeric(paired[day_a], errors="coerce")
    paired[day_b] = pd.to_numeric(paired[day_b], errors="coerce")

    paired = paired.dropna(subset=[day_a, day_b])

    if len(paired) < 2:
        return {
            "comparison": description,
            "day_a": day_a,
            "day_b": day_b,
            "n": len(paired),
            "mean_a": np.nan,
            "mean_b": np.nan,
            "mean_difference": np.nan,
            "percent_difference_vs_b": np.nan,
            "t_stat": np.nan,
            "p_value": np.nan,
            "cohens_d_paired": np.nan,
            "direction": "insufficient data",
        }

    diff = paired[day_a] - paired[day_b]

    t_stat, p_val = stats.ttest_rel(
        paired[day_a],
        paired[day_b],
        nan_policy="omit"
    )

    mean_a = paired[day_a].mean()
    mean_b = paired[day_b].mean()
    mean_diff = diff.mean()

    percent_difference_vs_b = (
        (mean_diff / mean_b) * 100
        if mean_b != 0
        else np.nan
    )

    cohens_d_paired = (
        mean_diff / diff.std(ddof=1)
        if diff.std(ddof=1) != 0
        else np.nan
    )

    direction = "higher" if mean_diff > 0 else "lower"

    return {
        "comparison": description,
        "day_a": day_a,
        "day_b": day_b,
        "n": len(paired),
        "mean_a": mean_a,
        "mean_b": mean_b,
        "mean_difference": mean_diff,
        "percent_difference_vs_b": percent_difference_vs_b,
        "t_stat": t_stat,
        "p_value": p_val,
        "cohens_d_paired": cohens_d_paired,
        "direction": f"{day_a} is {direction} than {day_b}",
    }

# --------------------------------------------------------------
# Run weekend-definition sensitivity tests
# --------------------------------------------------------------
weekend_sensitivity_tests = [
    ("Mon", "Sat", "Mon vs Sat: Monday compared with Saturday"),
    ("Mon", "Sun", "Mon vs Sun: Monday compared with Sunday"),
    ("Fri", "Sat", "Fri vs Sat: Friday compared with Saturday"),
    ("Fri", "Sun", "Fri vs Sun: Friday compared with Sunday"),
    ("Tue", "Mon", "Tue vs Mon: Tuesday midweek premium"),
    ("Wed", "Mon", "Wed vs Mon: Wednesday midweek premium"),
    ("Thu", "Fri", "Thu vs Fri: Thursday midweek premium"),
]

weekend_sensitivity_results = []

print("Robustness Check: Weekend Definition Sensitivity")
print("=" * 80)
print(f"Hybrid Office Core stations tested: {len(office_core_ids):,}")
print(f"Matched station records in dow_raw[2024]: {len(office_raw_2024):,}")
print()

for day_a, day_b, description in weekend_sensitivity_tests:
    result = paired_day_test(
        office_raw_2024,
        day_a,
        day_b,
        description
    )

    weekend_sensitivity_results.append(result)

    print(description)
    print("-" * 80)
    print(f"n = {result['n']}")
    print(f"{day_a} mean = {result['mean_a']:.3f}")
    print(f"{day_b} mean = {result['mean_b']:.3f}")
    print(f"Mean difference = {result['mean_difference']:.3f}")
    print(f"Percent difference vs {day_b} = {result['percent_difference_vs_b']:.2f}%")
    print(f"t = {result['t_stat']:.3f}, p = {result['p_value']:.4f}")
    print(f"Paired Cohen's d = {result['cohens_d_paired']:.3f}")
    print(f"Direction: {result['direction']}")
    print()

# --------------------------------------------------------------
# Build and save summary table
# --------------------------------------------------------------
weekend_sensitivity_df = pd.DataFrame(weekend_sensitivity_results)

weekend_sensitivity_df.to_csv(
    OUT_DIR / "section7_weekend_definition_sensitivity.csv",
    index=False
)

print("Weekend Definition Sensitivity Summary")
print("=" * 80)

display_cols = [
    "comparison",
    "n",
    "mean_a",
    "mean_b",
    "mean_difference",
    "percent_difference_vs_b",
    "t_stat",
    "p_value",
    "cohens_d_paired",
    "direction",
]

print(
    weekend_sensitivity_df[display_cols]
    .round(4)
    .to_string(index=False)
)

# --------------------------------------------------------------
# Final interpretation flags
# --------------------------------------------------------------
alpha = 0.05

tests = weekend_sensitivity_df.set_index("comparison")

mon_gt_weekend = (
    tests.loc["Mon vs Sat: Monday compared with Saturday", "mean_difference"] > 0
    and tests.loc["Mon vs Sat: Monday compared with Saturday", "p_value"] < alpha
    and tests.loc["Mon vs Sun: Monday compared with Sunday", "mean_difference"] > 0
    and tests.loc["Mon vs Sun: Monday compared with Sunday", "p_value"] < alpha
)

fri_gt_weekend = (
    tests.loc["Fri vs Sat: Friday compared with Saturday", "mean_difference"] > 0
    and tests.loc["Fri vs Sat: Friday compared with Saturday", "p_value"] < alpha
    and tests.loc["Fri vs Sun: Friday compared with Sunday", "mean_difference"] > 0
    and tests.loc["Fri vs Sun: Friday compared with Sunday", "p_value"] < alpha
)

midweek_gt_shoulders = (
    tests.loc["Tue vs Mon: Tuesday midweek premium", "mean_difference"] > 0
    and tests.loc["Tue vs Mon: Tuesday midweek premium", "p_value"] < alpha
    and tests.loc["Wed vs Mon: Wednesday midweek premium", "mean_difference"] > 0
    and tests.loc["Wed vs Mon: Wednesday midweek premium", "p_value"] < alpha
    and tests.loc["Thu vs Fri: Thursday midweek premium", "mean_difference"] > 0
    and tests.loc["Thu vs Fri: Thursday midweek premium", "p_value"] < alpha
)

print("\nInterpretation")
print("=" * 80)

if mon_gt_weekend and fri_gt_weekend and midweek_gt_shoulders:
    print(
        "Conclusion: PASS — Hybrid Office Core stations show a shoulder-day pattern. "
        "Monday and Friday remain significantly above true weekend demand, while "
        "Tue–Thu demand is significantly higher than Mon/Fri. This supports the "
        "paper's argument that a simple weekday/weekend binary is empirically inadequate."
    )
else:
    print(
        "Conclusion: REVIEW NEEDED — The shoulder-day pattern is not fully supported "
        "across all tested comparisons. Review the summary table above before making "
        "a strong claim about weekend-definition sensitivity."
    )

---
## SECTION 8: Export and Paper Deliverables

Final reproducibility checklist and key findings summary for paper writing.


### 8.1 Deliverables Checklist

Lists every output file expected from this notebook and checks whether it exists on Drive. A ✓ for every item means the analysis is complete and ready for TRR submission.

**Filename alignment:** Checklist filenames match what each section actually writes. If any item shows ✗ MISSING, re-run the section that produces it.


In [ ]:
# ==============================================================
# Section 8.1 — Export Final Deliverables Summary
#
# Purpose:
#   Lists all output files generated by this notebook and
#   confirms they exist on Drive before TRR submission.
#
# Filename alignment verified against actual output filenames:
#   Section 4.1  → figure1_k_selection_{year}.png
#   Section 3.2  → figure2_dow_profile_shift_precovid_vs_2024.png
#   Section 4.5  → table3_transition_matrix_counts.csv
#   Section 4.5  → station_transitions_precovid_to_2024.csv
#   Section 2.3  → dow_vectors_precovid.csv  (not _average)
#   Section 4.2  → cluster_assignments_precovid.csv  (not _average)
#   Section 7.1  → section7_k_sensitivity_robustness_summary.csv
# ==============================================================

if 'OUT_DIR' not in globals():
    OUT_DIR = Path("/content/drive/MyDrive/dow_ridership_paper/outputs")

DELIVERABLES = {
    "Tables": {
        "Table 1 — Summary Statistics":                 "table1_summary_stats.csv",
        "Table 2 — Cluster Profiles":                   "table2_cluster_profiles.csv",
        "Table 3 — Transition Matrix (counts)":         "table3_transition_matrix_counts.csv",
        "Table 4 — Regression Results":                 "table4_regression_results.csv",
        "Table 4 — Regression Pivot":                   "table4_regression_pivot.csv",
    },
    "Figures": {
        "Figure 1 — K Selection Diagnostics":           "figure1_k_selection_2024.png",
        "Figure 2 — DOW Profile Shift (Pre-COVID vs 2024)":
                                                        "figure2_dow_profile_shift_precovid_vs_2024.png",
        "Figure 3 — Radar Chart Cluster Profiles (PNG)":"figure3_radar_cluster_profiles.png",
        "Figure 3 — Radar Chart Cluster Profiles (PDF)":"figure3_radar_cluster_profiles.pdf",
        "Figure 4 — NYC Cluster Map (PNG)":             "figure4_nyc_cluster_map.png",
        "Figure 4 — NYC Cluster Map (PDF)":             "figure4_nyc_cluster_map.pdf",
    },
    "Robustness Outputs": {
        "K Sensitivity Robustness Summary":             "section7_k_sensitivity_robustness_summary.csv",
        "Weekend Definition Sensitivity":               "section7_weekend_definition_sensitivity.csv",
        "Table 1 — With 2023 Robustness":               "table1_summary_stats_with_2023_robustness.csv",
    },
    "Data Files": {
        "DOW Vectors — Pre-COVID":                      "dow_vectors_precovid.csv",
        "DOW Vectors — 2022":                           "dow_vectors_2022.csv",
        "DOW Vectors — 2024":                           "dow_vectors_2024.csv",
        "Cluster Assignments — Pre-COVID":              "cluster_assignments_precovid.csv",
        "Cluster Assignments — 2022":                   "cluster_assignments_2022.csv",
        "Cluster Assignments — 2024":                   "cluster_assignments_2024.csv",
        "Station Features (raw)":                       "station_features.csv",
        "Station Features (engineered)":                "station_features_engineered.csv",
        "Station Transitions Pre-COVID → 2024":         "station_transitions_precovid_to_2024.csv",
        "Recovery Rates by DOW":                        "recovery_rates_by_dow.csv",
        "Recovery Gap Summary":                         "recovery_gap_summary.csv",
        "Pre-COVID Average (station-level)":            "station_daily_precovid_avg.csv",
        "Pre-COVID Complex Average":                    "station_complex_precovid_avg.csv",
        "Cluster Interpretation Summary":               "cluster_interpretation_summary.csv",
        "Representative Stations by Cluster":           "representative_stations_by_cluster.csv",
        "Regression Dataset 2024":                      "regression_dataset_2024.csv",
        "Regression CV Results":                        "regression_cv_results.csv",
    },
}

print("=" * 72)
print("DELIVERABLES CHECKLIST")
print("=" * 72)

all_present  = True
missing_list = []

for category, items in DELIVERABLES.items():
    print(f"\n{category}:")
    for label, filename in items.items():
        path   = OUT_DIR / filename
        exists = path.exists()
        status = "✓" if exists else "✗ MISSING"
        size   = f"({path.stat().st_size:,} bytes)" if exists else ""
        if not exists:
            all_present = False
            missing_list.append((category, label, filename))
        print(f"  [{status}] {label:<55s} {size}")

print("\n" + "=" * 72)
if all_present:
    print("All deliverables present.  Ready for TRR submission. ✓")
else:
    print(f"Missing {len(missing_list)} file(s). Re-run the relevant section(s).")
    for cat, lbl, fn in missing_list:
        print(f"  - {cat} | {lbl} → {fn}")
print("=" * 72)


### 8.2 Key Findings Summary

Structured print of all quantitative findings for direct use when writing the Results and Discussion sections of the TRR paper. Checks for all required objects and prints graceful fallbacks if any section has not been run.


In [ ]:
# ==============================================================
# Section 8.2 — Key Findings Summary (For Paper Writing)
#
# Purpose:
#   Prints a structured summary of the key quantitative findings
#   for direct use in writing the Results and Discussion sections
#   of the TRR paper.
#
#   This summary reflects the paper's main comparison:
#   pre-COVID average baseline vs 2024 post-pandemic ridership.
# ==============================================================

print("=" * 70)
print("KEY FINDINGS SUMMARY FOR PAPER WRITING")
print("=" * 70)

# --------------------------------------------------------------
# [1] Day-of-week recovery gap: 2024 vs pre-COVID average
# --------------------------------------------------------------
print("\n[1] Day-of-Week Recovery Gap (2024 vs Pre-COVID Average):")

if "recovery_df" not in globals():
    print("  recovery_df not found. Run the recovery analysis section first.")
else:
    for day in DOW_LABELS:
        if day in recovery_df.index and "recovery_2024" in recovery_df.columns:
            r = recovery_df.loc[day, "recovery_2024"]
            print(f"  {day}: {r:.1%}")
        else:
            print(f"  {day}: unavailable")

    # Calculate Mon/Fri, Tue-Thu, and midweek premium directly if variables are missing
    try:
        monfri_recovery = recovery_df.loc[["Mon", "Fri"], "recovery_2024"].mean()
        tuethu_recovery = recovery_df.loc[["Tue", "Wed", "Thu"], "recovery_2024"].mean()
        midweek_premium = tuethu_recovery - monfri_recovery

        print(f"\n  Mon/Fri average recovery:  {monfri_recovery:.1%}")
        print(f"  Tue-Thu average recovery:  {tuethu_recovery:.1%}")
        print(f"  Mid-week premium:          {midweek_premium:.1%}")
    except Exception as e:
        print(f"\n  Could not calculate Mon/Fri, Tue-Thu, or gap values. Reason: {e}")

# --------------------------------------------------------------
# [2] Cluster analysis: 2024 demand regimes
# --------------------------------------------------------------
print(f"\n[2] Cluster Analysis (K={K_FINAL}, 2024):")

if "cluster_sizes" not in globals():
    print("  cluster_sizes not found. Run the 2024 cluster profiling section first.")
else:
    total_clustered = cluster_sizes.sum()

    for c_id, c_label in CLUSTER_LABELS.items():
        n = cluster_sizes.get(c_id, 0)
        pct = n / total_clustered * 100 if total_clustered > 0 else 0
        print(f"  Cluster {c_id} ({c_label}): {n} stations ({pct:.1f}%)")

# --------------------------------------------------------------
# [3] Cluster transition: pre-COVID average to 2024
# --------------------------------------------------------------
print("\n[3] Cluster Transition (Pre-COVID Average -> 2024):")

if all(v in globals() for v in ["n_stable", "n_total", "pct_stable"]):
    print(f"  Stations in same demand regime:  {n_stable} / {n_total} ({pct_stable:.1f}%)")
    print(f"  Stations that migrated regimes:  {n_total - n_stable} ({100 - pct_stable:.1f}%)")
else:
    print("  Transition summary variables unavailable.")

# --------------------------------------------------------------
# [4] Regression model
# --------------------------------------------------------------
print("\n[4] Regression (Multinomial Logit):")

pseudo_r2 = None

if "result" in globals():
    if hasattr(result, "prsquared"):
        pseudo_r2 = result.prsquared
    elif isinstance(result, dict):
        for key in ["prsquared", "pseudo_r2", "Pseudo_R2", "mcfadden_r2", "McFadden_R2"]:
            if key in result:
                pseudo_r2 = result[key]
                break

if pseudo_r2 is not None:
    print(f"  Pseudo-R2 (McFadden):      {pseudo_r2:.4f}")
else:
    print("  Pseudo-R2 (McFadden):      unavailable")

if "cv_scores" in globals():
    try:
        print(f"  5-fold CV accuracy:        {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")
    except Exception:
        print("  5-fold CV accuracy:        unavailable")
else:
    print("  5-fold CV accuracy:        unavailable")

if "reg_df" in globals():
    print(f"  N (stations):              {len(reg_df)}")
else:
    print("  N (stations):              unavailable")

# --------------------------------------------------------------
# [5] Robustness: K sensitivity
# --------------------------------------------------------------
print("\n[5] Robustness: K Sensitivity")

if "robustness_summary_df" in globals():
    all_k_pass = robustness_summary_df["hybrid_office_core_detected"].all()

    tested_k = robustness_summary_df["K"].tolist()
    print(f"  Alternative K values tested: {tested_k}")

    if all_k_pass:
        print("  Result: PASS - Hybrid Office Core regime persists across tested K values.")
    else:
        failed_k = robustness_summary_df.loc[
            ~robustness_summary_df["hybrid_office_core_detected"],
            "K"
        ].tolist()
        print(f"  Result: REVIEW NEEDED - Hybrid Office Core not detected for K={failed_k}.")
else:
    print("  K sensitivity summary unavailable.")

# --------------------------------------------------------------
# [6] Robustness: weekend definition sensitivity
# --------------------------------------------------------------
print("\n[6] Robustness: Weekend Definition Sensitivity")

if "weekend_sensitivity_df" in globals():
    alpha = 0.05
    significant_tests = int((weekend_sensitivity_df["p_value"] < alpha).sum())
    total_tests = len(weekend_sensitivity_df)

    print(f"  Significant paired comparisons: {significant_tests} / {total_tests}")

    try:
        tests = weekend_sensitivity_df.set_index("comparison")

        tue_mon_gap = tests.loc[
            "Tue vs Mon: Tuesday midweek premium",
            "percent_difference_vs_b"
        ]

        wed_mon_gap = tests.loc[
            "Wed vs Mon: Wednesday midweek premium",
            "percent_difference_vs_b"
        ]

        thu_fri_gap = tests.loc[
            "Thu vs Fri: Thursday midweek premium",
            "percent_difference_vs_b"
        ]

        print(f"  Tue vs Mon premium:         {tue_mon_gap:.1f}%")
        print(f"  Wed vs Mon premium:         {wed_mon_gap:.1f}%")
        print(f"  Thu vs Fri premium:         {thu_fri_gap:.1f}%")

        print(
            "  Result: PASS - Monday and Friday operate as shoulder days, "
            "with Tue-Thu demand significantly higher than Mon/Fri."
        )

    except Exception as e:
        print(f"  Weekend sensitivity details unavailable. Reason: {e}")
else:
    print("  Weekend definition sensitivity summary unavailable.")

# --------------------------------------------------------------
# Final paper-ready interpretation
# --------------------------------------------------------------
print("\n[7] Paper-Ready Interpretation:")
print(
    "  Using the pre-COVID average as the baseline, the analysis indicates "
    "that post-pandemic subway ridership has not returned as a uniform "
    "weekday pattern. By 2024, Hybrid Office Core stations exhibit a clear "
    "midweek concentration, with Tuesday through Thursday functioning as "
    "the dominant commuting period and Monday/Friday operating as lower-demand "
    "shoulder days. These results support the conclusion that the traditional "
    "weekday/weekend binary is empirically inadequate for describing "
    "post-pandemic subway demand."
)

print("\n" + "=" * 70)
print("Notebook execution complete.")
print("=" * 70)
